# TriFrame: Computer Vision with Deep Learning, end to end (v3)

**What is new in v3:** (1) 120 stocks instead of 60, and the sample contains the pilot's 60, so the two runs compare like for like;
(2) a leaderboard of every pretrained model on six metrics per target, with charts; (3) a 5-page plain-language PDF report;
(4) fine-tuning of all seven pretrained networks, scored against their frozen versions; (5) a clear table of the models the
dashboard uses; (6)-(9) the dashboard (in the GitHub repo) now analyses up to four companies at once, with charts, a risk ranking,
detailed plain-language explanations and a Monte Carlo simulation calibrated here in Stage 7M.

| Stage | Your step | What happens |
|---|---|---|
| 1 | Load the data | 979 US stocks, 25.8 years of daily/hourly bars (2001-01-02 to 2026-10-09), as chart sets: weekly 104, daily 126 and hourly 140 bars, plus 33 numeric features and the labels. This run: 120 stocks (the pilot's 60 + 60 more). Rows from 2023-01-03 on stay sealed. |
| 2 | Split | Train 2001-2016, validation 2017-2019, test 2020-2022, each with a 20-day purge at the boundary. Holdout 2023 onward. |
| 3 | Baselines | Numbers only: constant, HAR volatility model, linear and boosted-tree heads on the 33 features. |
| 4 | Pretrained models | Frozen embeddings of every chart from 7 ImageNet backbones (ResNet-18, EfficientNet-B0, ViT-B/16, ConvNeXt-Tiny, Swin-T, DINOv2-S, MobileNetV3) plus 3 random-weight controls; size and speed of each. Resumable. |
| 5 | Combine timeframes and tune | PCA per view, then image-only, numbers + images (all three views) and single-view heads; ridge/logistic paths and a tree grid tuned on validation. Every configuration is logged. |
| 5F | **Fine-tuning (new)** | Each pretrained backbone is trained end to end on the training years (early stopping on validation); its embeddings compete with the frozen ones. Frozen vs fine-tuned report with confidence intervals and training curves. |
| 6 | Pick the best model | Pre-registered rule; H2 test; calibration; the NO TRADE thresholds; everything frozen to a config file. |
| 6L | **Leaderboard (new)** | Model cards (what each network is, pretraining data, size, speed) and every model on six metrics per target, with bars, a heatmap and accuracy-vs-size charts. |
| 7 | Test | One look at 2020-2022: metrics with month-block confidence intervals, H1 and H3, what NO TRADE avoids, survivorship sensitivity, the test leaderboard, and 120 vs 60 stocks on the same rows. |
| 7M | **Monte Carlo (new)** | The dashboard's price simulation: tail shape and intraday range chosen on validation, coverage checked on test. |
| 8 | Model bundle | Saved with the simulation settings and risk percentiles, checked against the notebook's predictions, the training inputs and the dashboard's own embedding code, pushed to a private model repo. |
| 9 | Dashboard | The HTML block for krishnaharsha.com, then **which models the dashboard uses** and an example of its explanation. |
| 10 | Holdout (owner, once) | Off by default. Opens 2023 onward through the project's holdout guard with your key. |
| 11 | Report | REPORT.md and the **5-page PDF report** from computed values only (saved to Drive, Hugging Face and your Downloads). |

**Before you run**
1. Runtime > Change runtime type > **A100 GPU** and **High-RAM** (T4 works, about 3x slower).
2. Colab secrets (key icon): **HF_TOKEN** = a Hugging Face token with *write* access. Optional **HF_READ_TOKEN** = a *read-only* token for the dashboard.
3. Runtime > Run all. The 120-stock run takes about 2-3 hours on an A100 (fine-tuning is about half of it) and resumes after a disconnect.

**Rules this notebook enforces:** data only through the sealed `ImageStore`; preprocessing fitted on training rows only; hyper-parameters (and fine-tuning's early stopping, and the simulation's settings) chosen on validation only; the test years are read once (a second run needs a written reason); the holdout opens only with your key; trained weights stay in private repositories. Expect chart images to add little or nothing to the numbers: the renderer scales each chart to fit its panel, so pictures cannot show the volatility level. The notebook measures this rather than assuming it.

In [ ]:
# ==================================== CONFIG: edit here, then Runtime > Run all ====================================
import numpy as np

QUICK = True               # True = a fixed sample of QUICK_TICKERS stocks. False = all 979 stocks (several hours).
QUICK_TICKERS = 120        # v3: 120 stocks (the pilot used 60). About 2-3 hours on an A100, including fine-tuning.
PILOT_TICKERS = 60         # the 120-stock sample CONTAINS the pilot's 60 stocks (+ 60 new ones), so the two runs compare like for like
SEED = 11
HF_USER = "KrishnaHarshaTangirala"
DATA_REPO = f"{HF_USER}/triframe-us-equities-chart-ohlcv"   # existing private dataset: charts, features, labels, project code
RUNS_REPO = f"{HF_USER}/triframe-cv-runs"                   # private dataset: embeddings, results, figures, ledger, logs, report
MODEL_REPO = f"{HF_USER}/triframe-cv-model" + (f"-s{QUICK_TICKERS}" if QUICK else "-full")   # private model repo: the frozen bundle
                                                           # (each run gets its own repo, so the live dashboard keeps its model until you switch it)
PILOT_MODEL_REPO = f"{HF_USER}/triframe-cv-model"          # the 60-stock pilot's bundle, read only for the 60 vs 120 comparison
DEPLOY_HF_SPACE = False    # Gradio Spaces need a paid Hugging Face plan; the dashboard is deployed on Streamlit Cloud instead (see docs/DEPLOY.md)
SPACE_REPO = f"{HF_USER}/triframe-cv-dashboard"             # only used if DEPLOY_HF_SPACE = True
SPACE_PRIVATE = False      # a website iframe needs a public Space; the model files stay in the private MODEL_REPO
DASHBOARD_URL = ""         # the live dashboard address, e.g. "https://triframe-cv.streamlit.app" (fills the website snippet)
UPLOAD = True              # push every stage's outputs to Hugging Face
UPLOAD_EMBEDDINGS = True   # full run = about 20 GB of embeddings; if your Hugging Face storage is too small, set False
                           # (they are then kept on Google Drive instead, which is what lets a run resume after a disconnect)
GRADIO_VERSION = "6.20.0"  # version the optional Gradio app was tested with

SPLITS = {"train": ("2001-01-01", "2017-01-01"),            # fit the models
          "val":   ("2017-01-01", "2020-01-01"),            # tune, select, calibrate, set the NO TRADE rule and the simulation
          "test":  ("2020-01-01", "2023-01-03")}            # one honest check of the frozen choice (includes COVID-2020)
HOLDOUT_START = "2023-01-03"                                # sealed by holdout_guard; opened once, by the owner, in Stage 10

BACKBONE_NAMES = None      # None = the 7 pretrained backbones + 3 random-weight controls (tfcv_train.BACKBONES)
PCA_DIM = 128              # per backbone and view, fitted on training rows only
ALPHAS = [float(a) for a in np.logspace(-2, 6, 17)]         # ridge grid (volatility)
CS = [float(c) for c in np.logspace(-4, 1, 6)]              # logistic grid (direction, drawdown)
N_BOOT = 1000              # month-block bootstrap resamples for every confidence interval
CHUNK_TICKERS = 40         # embedding work unit: one resumable file per chunk of tickers

# ---- Stage 5F: fine-tuning (v3). Each pretrained backbone is trained end to end on the training years (early stopping on
#      validation); its fine-tuned embeddings then compete with the frozen ones under the same selection rule.
FINETUNE = True
FT_BACKBONES = "all"       # "all" = the 7 pretrained backbones, or a list such as ["resnet18", "convnext_tiny"]
FT_EPOCHS = 4              # maximum epochs; stops early when the validation loss stops improving
FT_MAX_TRAIN_ROWS = 60000  # cap on training chart sets (a fixed random subset if there are more; the 120-stock run has ~37k)
FT_MAX_VAL_ROWS = 20000    # cap on validation chart sets used for early stopping
FT_LR_BACKBONE, FT_LR_HEAD = 3e-5, 1e-3

# ---- Stage 7M: Monte Carlo price simulation (v3), calibrated on validation and checked once on the test years
MC_DRAWS = 20000           # simulation draws used for calibration
MC_TDF_GRID = [3, 4, 5, 7, 10, 30]          # tail shapes of the daily shocks (Student-t degrees of freedom; 30 = nearly normal)
MC_LOWK_GRID = [0.25, 0.5, 0.75, 1.0]       # intraday-low range factors (how far below the closes the daily lows reach)

REPORT_PDF = True          # Stage 11 writes a 5-page plain-language PDF report (saved to Drive, Hugging Face and your Downloads)

In [ ]:
# ==================================== SETUP: packages, Drive, Hugging Face, data, integrity ====================================
import os, sys, json, time, math, glob, shutil, hashlib, random, subprocess, warnings, gc, datetime as dt
import numpy as np, pandas as pd
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

IN_COLAB = "google.colab" in sys.modules
LOCAL_TEST = os.environ.get("TFCV_LOCAL_TEST") == "1"
RUN_TAG = f"s{QUICK_TICKERS}" if QUICK else "full"      # "s120" for the 120-stock run (the 60-stock pilot was "quick")
DST = os.environ.get("TFCV_DATA_ROOT", "/content/triframe").rstrip("/")
WORK = os.environ.get("TFCV_WORK", "/content/tfcv")
for sub in ("", "results", "figures", "frozen", "bundle", "space", "web", "ft", f"emb_{RUN_TAG}", f"emb_ft_{RUN_TAG}", f"pca_{RUN_TAG}"):
    os.makedirs(os.path.join(WORK, sub), exist_ok=True)

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm", "xgboost", "lightgbm", "yfinance",
                    "huggingface_hub>=0.25", "pyarrow", "pandas_market_calendars", "reportlab", "safetensors"], check=True)
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    DRIVE_DIR = "/content/drive/MyDrive/triframe_cv"

    def _secret(name):
        try:
            return userdata.get(name)
        except Exception:
            return None
    HF_TOKEN, HF_READ_TOKEN = _secret("HF_TOKEN"), _secret("HF_READ_TOKEN")
else:
    DRIVE_DIR = os.path.join(WORK, "drive_mirror")
    HF_TOKEN, HF_READ_TOKEN = os.environ.get("HF_TOKEN"), None
os.makedirs(DRIVE_DIR, exist_ok=True)
if LOCAL_TEST:
    UPLOAD = False

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if DEVICE == "cuda" else 0.0
BATCH = 384 if GPU_GB > 30 else 160
print(f"Device: {DEVICE} ({torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU'}, {GPU_GB:.0f} GB) | run: {RUN_TAG} | upload: {UPLOAD}")
if DEVICE != "cuda" and not LOCAL_TEST:
    print("WARNING: no GPU. Runtime > Change runtime type > A100 (or T4). The embedding stage needs a GPU.")

from huggingface_hub import HfApi, snapshot_download, hf_hub_download
api = HfApi(token=HF_TOKEN) if HF_TOKEN else None
if UPLOAD:
    assert HF_TOKEN, "Add a Hugging Face token with WRITE access as the Colab secret HF_TOKEN (key icon on the left)."
    print("Hugging Face user:", api.whoami()["name"])
    for rid, rtype in ((RUNS_REPO, "dataset"), (MODEL_REPO, "model")):
        api.create_repo(rid, repo_type=rtype, private=True, exist_ok=True)

FAILED_UPLOADS = []


def push(local, path_in_repo, repo_id=None, repo_type="dataset", msg=None):
    """Upload a file or folder to Hugging Face. Failures are recorded and retried in Stage 11, never silently dropped."""
    if not UPLOAD:
        return
    repo_id = repo_id or RUNS_REPO
    try:
        if os.path.isdir(local):
            api.upload_folder(folder_path=local, path_in_repo=path_in_repo, repo_id=repo_id, repo_type=repo_type,
                              commit_message=msg or f"update {path_in_repo}")
        else:
            api.upload_file(path_or_fileobj=local, path_in_repo=path_in_repo, repo_id=repo_id, repo_type=repo_type,
                            commit_message=msg or f"update {path_in_repo}")
    except Exception as e:
        FAILED_UPLOADS.append((local, path_in_repo, repo_id, repo_type))
        print(f"   upload failed for {path_in_repo}: {e} (will be retried in Stage 11)")


# ---- data: download once (about 13 GB), then verify sizes and sample checksums against the manifests
if IN_COLAB:
    print("Downloading / refreshing the TriFrame dataset from Hugging Face (first run takes 10-20 minutes)...")
    snapshot_download(DATA_REPO, repo_type="dataset", local_dir=DST, token=HF_TOKEN, max_workers=8)


def _sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for ch in iter(lambda: f.read(1 << 22), b""):
            h.update(ch)
    return h.hexdigest()


bad, n_files = [], 0
for man in ("cache/MANIFEST_cache.json", "cache_2001_2009/MANIFEST_cache_2001_2009.json",
            "cache_patch_r11/MANIFEST_cache_patch_r11.json", "features_visible/MANIFEST_features_visible.json"):
    m = json.load(open(f"{DST}/{man}"))["manifest"]
    for k, v in m.items():
        n_files += 1
        if not os.path.exists(f"{DST}/{k}") or os.path.getsize(f"{DST}/{k}") != v["bytes"]:
            bad.append(k)
    for k in random.Random(7).sample(sorted(m), min(8, len(m))):
        if k not in bad and _sha(f"{DST}/{k}") != m[k]["sha256"]:
            bad.append(k)
assert not bad, f"integrity check failed for {len(bad)} files, e.g. {bad[:3]}"
print(f"Integrity: {n_files:,} files match their manifests (sizes + sampled SHA-256).")

# ---- governance: the holdout access log must survive Colab resets (Drive + Hugging Face copy)
GOV = os.path.join(DRIVE_DIR, "governance")
os.makedirs(GOV, exist_ok=True)
os.environ["TRIFRAME_GOV_DIR"] = GOV
if UPLOAD and not os.path.exists(os.path.join(GOV, "holdout_access_log.csv")):
    try:
        hf_hub_download(RUNS_REPO, "governance/holdout_access_log.csv", repo_type="dataset", local_dir=DRIVE_DIR, token=HF_TOKEN)
        print("Restored the holdout access log from Hugging Face.")
    except Exception:
        pass
for p in (os.path.join(DST, "scripts"), WORK):
    if p not in sys.path:
        sys.path.insert(0, p)

## Code modules
The next cells write the modules: shared inference code (including the Monte Carlo engine), training code (including fine-tuning), the screenshot reader, the dashboard logic, the PDF report and the Gradio app.

In [ ]:
# writes tfcv_core.py (used by this notebook and, later, by the dashboard)
open(os.path.join(WORK, 'tfcv_core.py'), 'w', encoding='utf-8').write(r'''"""TriFrame CV - shared inference code, used by the Colab notebook AND the Hugging Face Space.

Everything the dashboard computes goes through the same functions the notebook used to build training inputs:
  * numeric inputs  -> feat_visible.from_series (the 33 deployable features, last 126 daily bars)
  * chart images    -> tf_render.StockWindows (weekly 104 / daily 126 / hourly 140 bars, binary, scale-free)
  * predictions     -> Bundle.predict (same scalers, PCA, heads, calibrators and NO TRADE rule as the frozen config)
"""
import os, io, json, math
import numpy as np
import pandas as pd
import feat_visible as FV
import tf_render as RN

DEPLOY = ["x_" + f for f in FV.DEPLOY]                 # 33 inputs (x_mean_dvol_log is excluded by project rule)
VIEWS = ("W", "D", "H")
TRADING_DAYS, HORIZON_V = 252, 20


def ann_vol(logrv20):
    """logrv20 = ln sqrt(sum of squared daily returns over the next 20 sessions) -> annualised volatility (fraction)."""
    return np.exp(np.asarray(logrv20, float)) * math.sqrt(TRADING_DAYS / HORIZON_V)


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -40, 40)))


# ------------------------------------------------------------------------------------------------ live market data
def _clean(df):
    df = df.dropna(subset=["open", "high", "low", "close"])
    df = df[(df[["open", "high", "low", "close"]] > 0).all(axis=1)]
    df["volume"] = df["volume"].fillna(0.0).clip(lower=0)
    return df


def fetch_yahoo(ticker, daily_period="3y", hourly_period="60d"):
    """Split-adjusted (not dividend-adjusted) daily and regular-hours hourly bars from Yahoo Finance, in the same layout as
    the training data: daily(date, open, high, low, close, volume) and hourly(timestamp UTC, open, high, low, close, volume)."""
    import yfinance as yf
    d = yf.download(ticker, period=daily_period, interval="1d", auto_adjust=False, actions=False, progress=False,
                    multi_level_index=False, threads=False)
    if d is None or len(d) == 0:
        raise ValueError(f"no daily data for {ticker}")
    di = pd.DatetimeIndex(d.index)
    di = di.tz_localize(None) if di.tz is not None else di
    daily = _clean(pd.DataFrame({"date": di.normalize(), "open": d["Open"].values, "high": d["High"].values,
                                 "low": d["Low"].values, "close": d["Close"].values, "volume": d["Volume"].values}))
    h = yf.download(ticker, period=hourly_period, interval="60m", auto_adjust=False, actions=False, prepost=False,
                    progress=False, multi_level_index=False, threads=False)
    if h is None or len(h) == 0:
        hourly = pd.DataFrame(columns=["timestamp", "open", "high", "low", "close", "volume"])
    else:
        hi = pd.DatetimeIndex(h.index)
        hi = hi.tz_localize("America/New_York") if hi.tz is None else hi
        hourly = _clean(pd.DataFrame({"timestamp": hi.tz_convert("UTC"), "open": h["Open"].values, "high": h["High"].values,
                                      "low": h["Low"].values, "close": h["Close"].values, "volume": h["Volume"].values}))
    return daily.reset_index(drop=True), rth_only(hourly)


def rth_only(hourly, early_close_days=None):
    """Keep regular-session hourly bars stamped at :30 New York time (09:30 ... 15:30), as in the training images.
    early_close_days: optional set of dates (13:00 closes) on which only the first 4 slots are kept (training rule r1.1;
    Yahoo has no bars after an early close, so the dashboard does not need it)."""
    if len(hourly) == 0:
        return hourly.reset_index(drop=True)
    et = pd.DatetimeIndex(hourly["timestamp"]).tz_convert("America/New_York")
    keep = np.asarray((et.minute == 30) & (et.hour >= 9) & (et.hour <= 15))
    if early_close_days:
        day = et.tz_localize(None).normalize()
        keep &= ~(np.asarray(day.isin(list(early_close_days))) & np.asarray(et.hour - 9 >= 4))
    return hourly[keep].sort_values("timestamp").reset_index(drop=True)


def drop_incomplete_session(daily, now=None):
    """Drop today's daily bar while the US session is still open (before 16:05 New York time)."""
    now = now or pd.Timestamp.now(tz="America/New_York")
    if len(daily) and daily["date"].iloc[-1] == now.tz_localize(None).normalize() and (now.hour, now.minute) < (16, 5):
        return daily.iloc[:-1].reset_index(drop=True)
    return daily


# ------------------------------------------------------------------------------------------------ as-of date from a screenshot legend
def _close(a, b, tol):
    return abs(a - b) <= tol * max(abs(b), 1e-9)


def match_daily(daily, ohlc, tol=0.003, lookback=400):
    """Index of the latest daily bar whose O/H/L/C all match the screenshot legend, else None."""
    n = len(daily)
    for k in range(n - 1, max(-1, n - 1 - lookback), -1):
        r = daily.iloc[k]
        if all(_close(r[c], ohlc[c], tol) for c in ("open", "high", "low", "close")):
            return k
    return None


def match_weekly(daily, ohlc, tol=0.003, lookback=60):
    """Index k of the daily bar at which the week-to-date bar equals the weekly legend (handles a week still in progress)."""
    S = RN.StockWindows(daily[["date", "open", "high", "low", "close", "volume"]], _empty_hourly())
    n = len(daily)
    for k in range(n - 1, max(-1, n - 1 - lookback), -1):
        o, h, l, c, _ = S.weekly(k)
        if _close(o[-1], ohlc["open"], tol) and _close(h[-1], ohlc["high"], tol) and _close(l[-1], ohlc["low"], tol) \
                and _close(c[-1], ohlc["close"], tol):
            return k
    return None


def match_hourly(daily, hourly, ohlc, tol=0.015):
    """Daily index of the session that contains an hourly bar matching the hourly legend (close within 1.5 %, see notes)."""
    if len(hourly) == 0:
        return None
    for j in range(len(hourly) - 1, -1, -1):
        r = hourly.iloc[j]
        if all(_close(r[c], ohlc[c], tol) for c in ("open", "high", "low", "close")):
            day = pd.Timestamp(r["timestamp"]).tz_convert("America/New_York").tz_localize(None).normalize()
            k = np.searchsorted(daily["date"].values.astype("datetime64[ns]"), np.datetime64(day, "ns"), side="right") - 1
            return int(k) if k >= 0 else None
    return None


def _empty_hourly():
    return pd.DataFrame({"timestamp": pd.to_datetime([], utc=True), "open": [], "high": [], "low": [], "close": [], "volume": []})


# ------------------------------------------------------------------------------------------------ model inputs
def features_at(daily, k):
    """The 33 numeric inputs at daily index k (needs k >= 125)."""
    if k < FV.W - 1:
        raise ValueError(f"need at least {FV.W} daily bars before the as-of date, have {k + 1}")
    a = [daily[c].values.astype(float) for c in ("open", "high", "low", "close", "volume")]
    return FV.from_series(*a, [k])[DEPLOY].iloc[0].values.astype(float)


def render_at(daily, hourly, k):
    """The three binary chart images at daily index k, exactly as the training renderer draws them."""
    S = RN.StockWindows(daily[["date", "open", "high", "low", "close", "volume"]], hourly if len(hourly) else _empty_hourly())
    (w, d, h), (nw, nd, nh) = S.triplet(int(k))
    return {"W": w, "D": d, "H": h, "n_valid": {"W": int(nw), "D": int(nd), "H": int(nh)}}


def triplet_png(trip, scale=2):
    """One PNG (bytes) with the three model-view charts side by side, white bars on black."""
    from PIL import Image, ImageDraw
    panels = []
    for v, label in (("W", "1W (104 weeks)"), ("D", "1D (126 days)"), ("H", "1H (140 hours)")):
        a = (trip[v] * 255).astype(np.uint8)
        im = Image.fromarray(a, "L").resize((a.shape[1] * scale, a.shape[0] * scale), Image.NEAREST).convert("RGB")
        canvas = Image.new("RGB", (im.width, im.height + 22), (17, 17, 17)); canvas.paste(im, (0, 22))
        ImageDraw.Draw(canvas).text((6, 4), label, fill=(200, 200, 200))
        panels.append(canvas)
    W = sum(p.width for p in panels) + 12 * (len(panels) - 1); H = max(p.height for p in panels)
    out = Image.new("RGB", (W, H), (0, 0, 0)); x = 0
    for p in panels:
        out.paste(p, (x, 0)); x += p.width + 12
    return out


# ------------------------------------------------------------------------------------------------ image embeddings (only if a chosen model uses images)
def to_input(img01, mean, std):
    import torch, torch.nn.functional as F
    t = torch.from_numpy(img01.astype(np.float32))[None, None].repeat(1, 3, 1, 1)
    t = F.interpolate(t, size=(224, 224), mode="bilinear", align_corners=False, antialias=True)
    return (t - torch.tensor(mean).view(1, 3, 1, 1)) / torch.tensor(std).view(1, 3, 1, 1)


_BACKBONE_CACHE = {}


def embed_views(trip, spec, root=None):
    """Backbone embeddings of the three views (CPU), with the training preprocessing. A fine-tuned backbone (spec has
    'weights_file') is built from its architecture and loads its weights from the bundle folder `root`."""
    import torch, timm
    key = spec.get("name", spec["timm_name"])
    if key not in _BACKBONE_CACHE:
        torch.manual_seed(int(spec.get("seed", 0)))
        if spec.get("weights_file"):
            from safetensors.torch import load_file
            m = timm.create_model(spec["timm"], pretrained=False, num_classes=0, **spec.get("kwargs", {}))
            m.load_state_dict(load_file(os.path.join(root or ".", spec["weights_file"]), device="cpu"))
        else:
            m = timm.create_model(spec["timm_name"], pretrained=bool(spec.get("pretrained", True)), num_classes=0, **spec.get("kwargs", {}))
        _BACKBONE_CACHE[key] = m.eval()
    m = _BACKBONE_CACHE[key]
    out = {}
    with torch.no_grad():
        for v in VIEWS:
            out[v] = m(to_input(trip[v], spec["mean"], spec["std"])).float().numpy()[0]
    return out


# ------------------------------------------------------------------------------------------------ frozen model bundle
class Bundle:
    """Loads config.json + arrays.npz + per-target tree files written by the notebook, and predicts one row or a matrix."""

    def __init__(self, path):
        self.path = path
        self.cfg = json.load(open(os.path.join(path, "config.json")))
        self.A = dict(np.load(os.path.join(path, "arrays.npz")))
        self.heads = {}
        for t, spec in self.cfg["targets"].items():
            if spec["head"] == "xgb":
                import xgboost as xgb
                b = xgb.Booster(); b.load_model(os.path.join(path, spec["file"])); self.heads[t] = b
            elif spec["head"] == "lgbm":
                import lightgbm as lgb
                self.heads[t] = lgb.Booster(model_file=os.path.join(path, spec["file"]))

    # -- design matrix for one target from standardized numbers and (optional) PCA'd image embeddings
    def num_block(self, x_num):
        x = np.asarray(x_num, float).reshape(-1, len(self.cfg["features"]))
        x = np.where(np.isfinite(x), x, self.A["num_fill"])
        return (x - self.A["num_mean"]) / self.A["num_std"]

    @staticmethod
    def parse_fs(fs):
        """'NUM' -> (False, None, ()); 'IMG:b' -> (False, b, WDH); 'NUM+IMG:b' -> (True, b, WDH); 'NUM+IMG:b:W' -> (True, b, (W,))."""
        if fs == "NUM":
            return True, None, ()
        parts = fs.split(":")
        views = (parts[2],) if len(parts) > 2 else VIEWS
        return parts[0] == "NUM+IMG", parts[1], tuple(views)

    def img_block(self, backbone, views, emb_by_view):
        """Raw embeddings {view: (n, dim)} -> PCA (frozen) -> standardised (frozen) -> clipped, exactly as in training."""
        parts = []
        for v in views:
            key = f"{backbone}|{v}"
            e = np.asarray(emb_by_view[v], np.float64).reshape(-1, self.A[f"pca_mean|{key}"].shape[0])
            parts.append((e - self.A[f"pca_mean|{key}"]) @ self.A[f"pca_comp|{key}"])
        tag = f"{backbone}|{''.join(views)}"
        z = (np.hstack(parts) - self.A[f"img_mean|{tag}"]) / self.A[f"img_std|{tag}"]
        return np.clip(z, -self.cfg["img_clip"], self.cfg["img_clip"])

    def design(self, target, x_num_std, emb_by_backbone=None, _cache=None):
        use_num, b, views = self.parse_fs(self.cfg["targets"][target]["feature_set"])
        if b is None:
            return x_num_std
        key = (b, views)
        if _cache is not None and key in _cache:
            z = _cache[key]
        else:
            z = self.img_block(b, views, emb_by_backbone[b])
            if _cache is not None:
                _cache[key] = z
        return np.hstack([x_num_std, z]) if use_num else z

    def raw_predict(self, target, X):
        spec = self.cfg["targets"][target]
        if spec["head"] == "linear":
            z = X @ self.A[f"coef|{target}"] + float(self.A[f"intercept|{target}"])
            return z if spec["task"] == "reg" else sigmoid(z)
        if spec["head"] == "xgb":
            return np.asarray(self.heads[target].inplace_predict(np.ascontiguousarray(X, dtype=np.float32)), float)
        return np.asarray(self.heads[target].predict(X), float)

    def calibrate(self, target, p):
        c = self.cfg["targets"][target].get("calibrator")
        return np.interp(p, c["x"], c["y"]) if c else p

    def needed_backbones(self):
        return sorted({self.parse_fs(s["feature_set"])[1] for s in self.cfg["targets"].values()} - {None})

    def predict_matrix(self, x_num, emb_by_backbone=None):
        """x_num: (n, 33) raw features; emb_by_backbone: {backbone: {view: (n, dim) raw embeddings}}.
        Returns arrays: logrv20, vol_ann, vol_ann_lo, vol_ann_hi, p_out5, p_dd10 (probabilities calibrated)."""
        xs = self.num_block(x_num)
        cache = {}
        v = self.raw_predict("V", self.design("V", xs, emb_by_backbone, cache))
        q = self.cfg["targets"]["V"]
        out = {"logrv20": v, "vol_ann": ann_vol(v), "vol_ann_lo": ann_vol(v + q["res_q10"]), "vol_ann_hi": ann_vol(v + q["res_q90"])}
        for t, k in (("O5", "p_out5"), ("D10", "p_dd10")):
            out[k] = self.calibrate(t, self.raw_predict(t, self.design(t, xs, emb_by_backbone, cache)))
        return out

    def predict_row(self, x_num, emb_by_backbone=None):
        """x_num: 33 raw features; emb_by_backbone: {backbone: {view: raw embedding}} for backbones the bundle needs."""
        m = self.predict_matrix(np.asarray(x_num, float).reshape(1, -1),
                                {b: {v: np.asarray(e).reshape(1, -1) for v, e in d.items()} for b, d in (emb_by_backbone or {}).items()})
        out = {k: float(np.asarray(v)[0]) for k, v in m.items()}
        out["decision"], out["reasons"] = decide(out, self.cfg["rule"])
        return out


# ------------------------------------------------------------------------------------------------ Monte Carlo price paths
MC_DEFAULT = dict(t_df=5.0, low_k=1.0)


def _unit_t(rng, df, size):
    """Student-t shocks scaled to unit variance (df >= 100 means normal)."""
    if df is None or df >= 100:
        return rng.standard_normal(size)
    return rng.standard_t(df, size) * math.sqrt((df - 2.0) / df)


def mc_unit_paths(n, t_df, rng, horizon=HORIZON_V, low_k=1.0):
    """Path shapes with a realised 20-day volatility of exactly 1 (sqrt of the sum of squared daily log returns = 1, the same
    definition as the logrv20 label). Returns (cum, low), both (n, horizon): the cumulative log return after each session,
    and that session's intraday low, drawn from the exact minimum of a Brownian bridge between consecutive closes whose
    variance is low_k times the daily variance (low_k is calibrated on validation data so simulated drawdowns match real ones)."""
    z = _unit_t(rng, t_df, (n, horizon))
    z /= np.sqrt((z ** 2).sum(1, keepdims=True))
    cum = np.cumsum(z, 1)
    prev = np.hstack([np.zeros((n, 1)), cum[:, :-1]])
    low = 0.5 * (prev + cum - np.sqrt((cum - prev) ** 2 - 2.0 * low_k / horizon * np.log(rng.random((n, horizon)))))
    return cum, low


def tilt_weights(hit, ret, p_target, iters=80):
    """The smallest change to equal path weights (minimum relative entropy) such that the weighted share of paths touching a 10%
    drawdown equals p_target while the mean 20-day return stays what it was (no directional view is added).
    Weights are proportional to exp(a * hit + b * ret); (a, b) are found by damped Newton steps. Returns (weights, converged)."""
    h = np.asarray(hit, float); r = np.asarray(ret, float)
    r = (r - r.mean()) / (r.std() + 1e-12)
    target = np.array([float(p_target), 0.0])
    ab = np.zeros(2)

    def moments(ab_):
        z = ab_[0] * h + ab_[1] * r
        w = np.exp(z - z.max()); w /= w.sum()
        return w, np.array([w @ h, w @ r])
    w, m = moments(ab)
    for _ in range(iters):
        g = m - target
        if np.abs(g).max() < 1e-6:
            return w, True
        dh, dr = h - m[0], r - m[1]
        J = np.array([[w @ (dh * dh), w @ (dh * dr)], [w @ (dh * dr), w @ (dr * dr)]]) + 1e-10 * np.eye(2)
        step, t = np.linalg.solve(J, g), 1.0
        while t > 1e-4:
            w2, m2 = moments(ab - t * step)
            if np.abs(m2 - target).sum() < np.abs(g).sum():
                ab, w, m = ab - t * step, w2, m2
                break
            t *= 0.5
        else:
            break
    return w, bool(np.abs(m - target).max() < 1e-4)


def residual_sampler(vspec, mc=None):
    """Draws of the volatility model's error (log scale): from the validation quantile table if the bundle has one, else a
    normal with the validation median and variance."""
    q = (mc or {}).get("res_quantiles") or vspec.get("res_quantiles")
    if q:
        qs = np.asarray(q, float); grid = np.linspace(0.0, 1.0, len(qs))
        return lambda rng, n: np.interp(rng.random(n), grid, qs)
    mid, s = float(vspec.get("res_q50", 0.0)), math.sqrt(float(vspec.get("s2", 0.1)))
    return lambda rng, n: mid + s * rng.standard_normal(n)


def wquantile(x, w, qs):
    """Weighted quantiles of x (n,) or column-wise of x (n, k)."""
    x = np.asarray(x, float)
    if x.ndim == 2:
        return np.column_stack([wquantile(x[:, j], w, qs) for j in range(x.shape[1])])
    o = np.argsort(x); cw = np.cumsum(w[o]); cw /= cw[-1]
    return np.interp(qs, cw - 0.5 * w[o] / w.sum(), x[o])


def simulate_paths(p0, logrv20, vspec, mc=None, n_paths=5000, p_dd_target=None, seed=0, horizon=HORIZON_V, n_show=40):
    """Monte Carlo price paths over the model's 20-session horizon.
    * Each path's volatility = the model's forecast x its validated error (drawn from the validation error distribution), so a
      less accurate model gives a wider spread of paths.
    * Daily shocks are fat-tailed (Student-t, tail shape chosen on validation data) and scaled so the path's realised volatility
      equals its drawn volatility (the label's own definition). No drift: the direction model has no reliable edge.
    * Intraday lows come from the exact Brownian-bridge minimum (range calibrated on validation data), because the drawdown
      label uses daily lows.
    * p_dd_target: reweight the paths (smallest possible change, mean return unchanged) so the share touching a 10% drawdown
      equals the calibrated drawdown model's probability.
    Returns a dict of arrays and summary numbers."""
    mc = {**MC_DEFAULT, **(mc or {})}
    rng = np.random.default_rng(seed)
    eps = residual_sampler(vspec, mc)(rng, n_paths)
    rv = np.exp(float(logrv20) + eps)
    cum, low = mc_unit_paths(n_paths, mc["t_df"], rng, horizon, mc["low_k"])
    prices = float(p0) * np.exp(np.hstack([np.zeros((n_paths, 1)), rv[:, None] * cum]))
    lows = float(p0) * np.exp(rv[:, None] * low)
    worst = lows.min(1) / float(p0) - 1
    hit = worst <= -0.10
    p_sim = float(hit.mean())
    ret = prices[:, -1] / float(p0) - 1
    w = np.full(n_paths, 1.0 / n_paths)
    tilted = False
    if p_dd_target is not None and 0 < p_sim < 1:
        w2, ok = tilt_weights(hit, ret, float(np.clip(p_dd_target, 1e-4, 1 - 1e-4)))
        if ok:
            w, tilted = w2, True
    qs = [0.05, 0.25, 0.5, 0.75, 0.95]
    fan = wquantile(prices, w, qs)                                  # (5, horizon + 1)
    q05 = float(wquantile(ret, w, [0.05])[0])
    tail = ret <= q05
    show = rng.choice(n_paths, size=min(n_show, n_paths), replace=False, p=w / w.sum())
    return dict(p0=float(p0), days=np.arange(horizon + 1), fan=fan, fan_q=qs, sample_paths=prices[show], ret=ret, w=w,
                p_dd_sim=p_sim, p_dd=float(np.sum(w * hit)), tilted=tilted, ess=float(1.0 / np.sum(w ** 2)),
                ret_q05=q05, ret_q50=float(wquantile(ret, w, [0.5])[0]), ret_q95=float(wquantile(ret, w, [0.95])[0]),
                p_up=float(np.sum(w * (ret > 0))), p_down10=float(np.sum(w * (ret <= -0.10))),
                es95=float(-np.sum(w[tail] * ret[tail]) / max(np.sum(w[tail]), 1e-12)), var95=-q05,
                price_q05=float(p0) * (1 + q05), price_q95=float(p0) * (1 + float(wquantile(ret, w, [0.95])[0])),
                mean_ret=float(np.sum(w * ret)), n_paths=int(n_paths), t_df=float(mc["t_df"]), low_k=float(mc["low_k"]))


def decide(out, rule):
    """Pre-registered NO TRADE rule (frozen on validation data). NO TRADE is the default."""
    reasons = []
    risk_high = out["vol_ann"] >= rule["vol_ann_q80"] or out["p_dd10"] >= rule["p_dd10_q80"]
    if out["vol_ann"] >= rule["vol_ann_q80"]:
        reasons.append(f"Predicted volatility is in the top 20 % of the validation period (>= {rule['vol_ann_q80']:.0%} a year).")
    if out["p_dd10"] >= rule["p_dd10_q80"]:
        reasons.append(f"Chance of a 10 % drawdown is in the top 20 % of the validation period (>= {rule['p_dd10_q80']:.0%}).")
    conf = abs(out["p_out5"] - 0.5)
    if not rule["direction_usable"]:
        reasons.append("The direction model showed no reliable edge on held-out data, so direction is not used for a call.")
        return "NO TRADE", reasons
    if conf < rule["conf_threshold"]:
        reasons.append("Direction confidence is below the threshold chosen on validation data.")
    if risk_high or conf < rule["conf_threshold"]:
        return "NO TRADE", reasons
    side = "above" if out["p_out5"] > 0.5 else "below"
    reasons.append("Direction confidence passes the validated threshold, and risk is not elevated.")
    return f"Risk within normal range; 5-day outlook {side} the median stock (low-confidence signal)", reasons
''')
print('wrote', 'tfcv_core.py')

In [ ]:
# writes tfcv_train.py (used by this notebook and, later, by the dashboard)
open(os.path.join(WORK, 'tfcv_train.py'), 'w', encoding='utf-8').write(r'''"""TriFrame CV - training-side code for the Colab notebook (loader, embeddings, PCA, heads, metrics, selection).
Data is read only through triframe_loader.ImageStore (sealed); the fast reader below is checked against
ImageStore.triplet on random rows before it is used."""
import os, json, time, math, gc, re
import numpy as np
import pandas as pd
import tf_render as RN

VIEWS = ("W", "D", "H")
_SHAPES = {"W": (128, 3 * 104), "D": (128, 3 * 126), "H": (128, 3 * 140)}
_OFF = {"W": 0, "D": 128 * 312, "H": 128 * (312 + 378)}

# ===================================================================================== 1. fast triplet reader
class FastTriplets:
    """Vectorised version of ImageStore.triplet for whole tickers at a time (same shards, same r1.1 patch overlay)."""

    def __init__(self, st):
        self.st = st

    def read_packed(self, g):
        """g: rows of ONE ticker from the store's index (columns ticker, sample_id, src, shard_row, patched).
        Returns the packed 1-bit triplets (n, TRIPLET_BYTES) in g's order."""
        t = g["ticker"].iloc[0]
        n = len(g)
        buf = np.empty((n, RN.TRIPLET_BYTES), np.uint8)
        pm = g["patched"].values.astype(bool)
        if (~pm).any():
            src = g["src"].values
            for s in np.unique(src[~pm]):
                sel = (~pm) & (src == s)
                buf[sel] = np.asarray(self.st._shard(s, t)[g["shard_row"].values[sel]])
        if pm.any():
            prow = self.st._patch.loc[g["sample_id"].values[pm], "patch_row"].values.astype(int)
            buf[pm] = np.asarray(self.st._shard("cache_patch_r11", t)[prow])
        return buf

    def read(self, g):
        """g: rows of ONE ticker from the store's index. Returns {view: (n, 128, width) uint8 0/1 images}."""
        n = len(g)
        bits = np.unpackbits(self.read_packed(g), axis=1)[:, :RN.TRIPLET_BITS]
        return {v: bits[:, _OFF[v]:_OFF[v] + _SHAPES[v][0] * _SHAPES[v][1]].reshape(n, *_SHAPES[v]) for v in VIEWS}

    def parity(self, rows, n=300, seed=0):
        """Assert the fast reader equals ImageStore.triplet on n random rows (patched rows included)."""
        rng = np.random.default_rng(seed)
        pick = rows.iloc[rng.choice(len(rows), min(n, len(rows)), replace=False)]
        patched = rows[rows["patched"]]
        if len(patched):
            pick = pd.concat([pick, patched.iloc[rng.choice(len(patched), min(50, len(patched)), replace=False)]])
        bad = 0
        for t, g in pick.groupby("ticker"):
            fast = self.read(g)
            for j, sid in enumerate(g["sample_id"].values):
                ref = self.st.triplet(sid)
                bad += sum(int(not np.array_equal(fast[v][j], ref[v])) for v in VIEWS)
        assert bad == 0, f"fast reader differs from ImageStore.triplet on {bad} views"
        return len(pick)


# ===================================================================================== 2. backbones and embeddings
BACKBONES = {
    "resnet18": dict(timm="resnet18"),
    "efficientnet_b0": dict(timm="efficientnet_b0"),
    "vit_base_patch16_224": dict(timm="vit_base_patch16_224"),
    "convnext_tiny": dict(timm="convnext_tiny"),
    "swin_tiny_patch4_window7_224": dict(timm="swin_tiny_patch4_window7_224"),
    "vit_small_patch14_dinov2": dict(timm="vit_small_patch14_dinov2", kwargs={"img_size": 224}),
    "mobilenetv3_large_100": dict(timm="mobilenetv3_large_100"),
    "resnet18_random_s0": dict(timm="resnet18", pretrained=False, seed=0),
    "resnet18_random_s1": dict(timm="resnet18", pretrained=False, seed=1),
    "resnet18_random_s2": dict(timm="resnet18", pretrained=False, seed=2),
}
PRETRAINED = [b for b, s in BACKBONES.items() if s.get("pretrained", True)]
RANDOM = [b for b in BACKBONES if b not in PRETRAINED]

# plain-language facts for the model cards (family, what the network is, who introduced it)
BACKBONE_INFO = {
    "resnet18": ("CNN", "ResNet-18", "18-layer residual convolutional network (He et al., 2016)"),
    "efficientnet_b0": ("CNN", "EfficientNet-B0", "compound-scaled convolutional network built for efficiency (Tan & Le, 2019)"),
    "vit_base_patch16_224": ("Transformer", "ViT-B/16", "Vision Transformer: the image is cut into 16x16 patches that attend to each other (Dosovitskiy et al., 2021)"),
    "convnext_tiny": ("CNN", "ConvNeXt-Tiny", "a ResNet modernised with transformer-era design choices (Liu et al., 2022)"),
    "swin_tiny_patch4_window7_224": ("Transformer", "Swin-T", "hierarchical transformer that attends within shifted windows (Liu et al., 2021)"),
    "vit_small_patch14_dinov2": ("Transformer", "DINOv2 ViT-S/14", "Vision Transformer trained without labels by self-distillation (Oquab et al., 2023)"),
    "mobilenetv3_large_100": ("CNN", "MobileNetV3-Large", "lightweight network designed for phones (Howard et al., 2019)"),
    "resnet18_random_s0": ("Control", "ResNet-18 (random weights, seed 0)", "same architecture as ResNet-18 but never trained: a control"),
    "resnet18_random_s1": ("Control", "ResNet-18 (random weights, seed 1)", "same architecture as ResNet-18 but never trained: a control"),
    "resnet18_random_s2": ("Control", "ResNet-18 (random weights, seed 2)", "same architecture as ResNet-18 but never trained: a control"),
}


def pretrain_data(timm_name, pretrained=True):
    """The pretraining dataset behind a timm weight tag, in words."""
    if not pretrained:
        return "none (random weights)"
    tag = timm_name.split(".", 1)[1] if "." in timm_name else ""
    if "lvd142m" in tag:
        return "LVD-142M, 142 million unlabelled images (self-supervised)"
    if "in21k_ft_in1k" in tag:
        return "ImageNet-21k, then fine-tuned on ImageNet-1k"
    if "in12k_ft_in1k" in tag:
        return "ImageNet-12k, then fine-tuned on ImageNet-1k"
    if "in1k" in tag:
        return "ImageNet-1k (1.28 million labelled photos, 1,000 classes)"
    return tag or "ImageNet"


def display_name(name):
    """'ft_convnext_tiny' -> 'ConvNeXt-Tiny (fine-tuned)'."""
    base = base_name(name)
    nm = BACKBONE_INFO.get(base, ("", base, ""))[1]
    return f"{nm} (fine-tuned)" if name.startswith(FT_PREFIX) else nm


# fine-tuned backbones are named "ft_<backbone>" and load their weights from FT_WEIGHTS (filled by the notebook's Stage 5F)
FT_PREFIX = "ft_"
FT_WEIGHTS = {}


def base_name(name):
    return name[len(FT_PREFIX):] if name.startswith(FT_PREFIX) else name


def build_backbone(name, device, factory=None):
    """Returns (model, mean, std, record). factory(name, spec) may replace timm (used only for the local CPU test).
    A name 'ft_<backbone>' builds <backbone> and loads its fine-tuned weights from FT_WEIGHTS[name]."""
    import torch
    base = base_name(name)
    ft = name != base
    spec = BACKBONES[base]
    if factory is not None:
        m, mu, sd, rec = factory(base, spec, device)
    else:
        import timm
        torch.manual_seed(int(spec.get("seed", 0)))
        m = timm.create_model(spec["timm"], pretrained=bool(spec.get("pretrained", True)) and not ft, num_classes=0, **spec.get("kwargs", {}))
        m = m.to(device).eval()
        cfg = timm.data.resolve_data_config({}, model=m)
        pc = getattr(m, "pretrained_cfg", {}) or {}
        tag = pc.get("tag") if spec.get("pretrained", True) else None
        rec = dict(name=base, timm=spec["timm"], timm_name=(f"{spec['timm']}.{tag}" if tag else spec["timm"]),
                   pretrained=bool(spec.get("pretrained", True)), seed=int(spec.get("seed", 0)), kwargs=spec.get("kwargs", {}),
                   mean=list(cfg["mean"]), std=list(cfg["std"]), hf_hub_id=pc.get("hf_hub_id"), timm_version=timm.__version__)
        mu = torch.tensor(cfg["mean"], device=device).view(1, 3, 1, 1)
        sd = torch.tensor(cfg["std"], device=device).view(1, 3, 1, 1)
    if ft:
        from safetensors.torch import load_file
        m.load_state_dict(load_file(FT_WEIGHTS[name], device="cpu"))
        m = m.to(device).eval()
        rec = dict(rec, name=name, base=base, finetuned=True, weights_file=os.path.basename(FT_WEIGHTS[name]))
    rec["n_params"] = int(sum(p.numel() for p in m.parameters()))
    return m, mu, sd, rec


def throughput(m, mu, sd, device, n_img=256, batch=64):
    """Inference speed (images per second) of a backbone on this device, with the same preprocessing and precision as the
    embedding stage."""
    import torch
    x = torch.rand(batch, 3, 224, 224, device=device)
    cuda = str(device).startswith("cuda")
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16, enabled=cuda):
        m((x - mu) / sd)                                           # warm-up
        if cuda:
            torch.cuda.synchronize()
        t0 = time.time(); done = 0
        while done < n_img:
            m((x - mu) / sd); done += batch
        if cuda:
            torch.cuda.synchronize()
    return done / max(time.time() - t0, 1e-9)


def extract_embeddings(st, rows, names, out_dir, device, chunk_tickers=40, batch=384, factory=None,
                       on_chunk=None, have_chunk=None, log=print):
    """Embeds every row x view x backbone. Work is cut into chunks of tickers; each finished chunk is one .npz
    (sample_id + one float16 array per 'backbone|view'). Finished chunks are skipped, so the run can resume.
    on_chunk(path) is called after each chunk (e.g. upload to Hugging Face); have_chunk(name) can fetch a chunk saved
    in an earlier session (returns a local path or None)."""
    import torch, torch.nn.functional as F
    os.makedirs(out_dir, exist_ok=True)
    tickers = list(dict.fromkeys(rows["ticker"].tolist()))
    chunks = [tickers[i:i + chunk_tickers] for i in range(0, len(tickers), chunk_tickers)]
    models, records = {}, {}
    for nm in names:
        m, mu, sd, rec = build_backbone(nm, device, factory)
        with torch.no_grad():
            rec["emb_dim"] = int(m(torch.zeros(2, 3, 224, 224, device=device)).shape[1])
        rec["infer_img_per_s"] = float(throughput(m, mu, sd, device))
        models[nm] = (m, mu, sd); records[nm] = rec
    reader = FastTriplets(st)
    use_amp = str(device).startswith("cuda")
    checked = set()
    t0 = time.time(); done_rows = 0
    for ci, ch in enumerate(chunks):
        fname = f"chunk_{ci:04d}.npz"; path = os.path.join(out_dir, fname)
        if not os.path.exists(path) and have_chunk is not None:
            got = have_chunk(fname)
            if got and os.path.exists(got) and got != path:
                os.replace(got, path)
        if os.path.exists(path):
            continue
        sub = rows[rows["ticker"].isin(ch)]
        sub = pd.concat([sub[sub["ticker"] == t] for t in ch if (sub["ticker"] == t).any()])
        out = {}
        pos = 0
        for t, g in sub.groupby("ticker", sort=False):
            imgs = reader.read(g)
            n = len(g)
            for v in VIEWS:
                for b0 in range(0, n, batch):
                    x = torch.from_numpy(np.ascontiguousarray(imgs[v][b0:b0 + batch])).to(device)
                    x = F.interpolate(x.float().unsqueeze(1), size=(224, 224), mode="bilinear", align_corners=False, antialias=True)
                    if v not in checked:
                        assert float(x.max()) > 0.5, f"blank chart batch in view {v}"
                    x3 = x.expand(-1, 3, -1, -1)
                    for nm, (m, mu, sd) in models.items():
                        xn = (x3 - mu) / sd
                        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                            e = m(xn)
                        e = e.float()
                        if (nm, v) not in checked:
                            assert float(xn.std()) > 0.2, f"{nm}/{v}: input contrast too low after normalisation"
                            assert torch.isfinite(e).all(), f"{nm}/{v}: non-finite embeddings"
                            assert float(e.std(0).mean()) > 1e-4, f"{nm}/{v}: embeddings do not vary across charts"
                            checked.add((nm, v))
                        key = f"{nm}|{v}"
                        if key not in out:
                            out[key] = np.empty((len(sub), e.shape[1]), np.float16)
                        out[key][pos + b0:pos + b0 + e.shape[0]] = e.cpu().numpy().astype(np.float16)
                checked.add(v)
            pos += n
        tmp = path + ".tmp.npz"
        np.savez(tmp, sample_id=np.asarray(sub["sample_id"].tolist(), dtype="U"), **out)    # fixed-width text, no pickle
        os.replace(tmp, path)
        done_rows += len(sub)
        el = time.time() - t0
        log(f"   chunk {ci + 1}/{len(chunks)}: {len(sub):,} rows, {done_rows / max(el, 1e-9):,.0f} rows/s this session")
        if on_chunk is not None:
            on_chunk(path)
    return records


# ===================================================================================== 2b. fine-tuning (end to end, multi-task)
def packed_rows(st, rows):
    """Packed 1-bit chart triplets (n, TRIPLET_BYTES) for index rows, in the order of rows (about 17.8 kB per chart set)."""
    reader = FastTriplets(st)
    out = np.empty((len(rows), RN.TRIPLET_BYTES), np.uint8)
    tick = rows["ticker"].values
    for t in pd.unique(tick):
        sel = np.flatnonzero(tick == t)
        out[sel] = reader.read_packed(rows.iloc[sel])
    return out


def views_from_packed(buf, device, resize=True):
    """Packed triplets (B, TRIPLET_BYTES) -> {view: (B, 1, 224, 224) float tensor in [0, 1]} on device, resized exactly as
    in extract_embeddings (bilinear, antialiased). resize=False returns the raw 0/1 images (B, 1, 128, width)."""
    import torch, torch.nn.functional as F
    b = torch.from_numpy(np.ascontiguousarray(buf)).to(device)
    shifts = torch.arange(7, -1, -1, device=device, dtype=torch.uint8)
    bits = ((b.unsqueeze(-1) >> shifts) & 1).reshape(b.shape[0], -1)[:, :RN.TRIPLET_BITS]
    out = {}
    for v in VIEWS:
        h, w = _SHAPES[v]
        x = bits[:, _OFF[v]:_OFF[v] + h * w].reshape(-1, 1, h, w).float()
        out[v] = F.interpolate(x, size=(224, 224), mode="bilinear", align_corners=False, antialias=True) if resize else x
    return out


def _ft_net(backbone, emb_dim, n_num, hidden=256, drop=0.2):
    """The fine-tuning network: ONE backbone embeds each of the three views; the three embeddings and the 33 standardised
    numbers feed a small shared head with three outputs (volatility, direction logit, drawdown logit)."""
    import torch, torch.nn as nn

    class TriFrameNet(nn.Module):
        def __init__(self):
            super().__init__()
            self.backbone = backbone
            self.trunk = nn.Sequential(nn.LayerNorm(3 * emb_dim + n_num), nn.Linear(3 * emb_dim + n_num, hidden), nn.GELU(), nn.Dropout(drop))
            self.out = nn.Linear(hidden, 3)

        def forward(self, xs, xnum):
            n = xnum.shape[0]
            e = self.backbone(torch.cat(xs, 0)).float()
            e = torch.cat(e.split(n, 0), 1)
            return self.out(self.trunk(torch.cat([e, xnum], 1)))
    return TriFrameNet()


def _ft_losses(out, ys):
    """Per-task losses with missing labels masked: MSE on standardised volatility, plain binary cross-entropy on the two
    probability targets (no label smoothing, no focal loss: project rule)."""
    import torch, torch.nn.functional as F
    terms = []
    for k, y in enumerate(ys):
        m = torch.isfinite(y)
        if not bool(m.any()):
            terms.append(out[:, k].sum() * 0.0); continue
        p = out[m, k].float()
        terms.append(F.mse_loss(p, y[m]) if k == 0 else F.binary_cross_entropy_with_logits(p, y[m]))
    return terms


def finetune(name, packed, xnum, ys, train_idx, val_idx, device, epochs=4, batch=64, lr_backbone=3e-5, lr_head=1e-3,
             weight_decay=0.05, patience=1, seed=11, factory=None, log=print):
    """Fine-tunes one pretrained backbone end to end on TRAINING rows only, with early stopping on the VALIDATION loss.
    packed: (n, TRIPLET_BYTES) chart sets; xnum: (n, 33) standardised numbers; ys: (yV, yO5, yD10) arrays of length n (NaN =
    missing); train_idx / val_idx index into those arrays. No augmentation (flips or crops would change what a chart means).
    Returns (backbone with the best-epoch weights, eval mode), mean, std, record, history (DataFrame), and the network's own
    validation predictions at the best epoch {V: logrv20, O5: prob, D10: prob}."""
    import torch
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    m, mu, sd, rec = build_backbone(name, device, factory)
    with torch.no_grad():
        dim = int(m(torch.zeros(2, 3, 224, 224, device=device)).shape[1])
    net = _ft_net(m, dim, xnum.shape[1]).to(device)
    yv = np.asarray(ys[0], float)
    v_mu, v_sd = float(np.nanmean(yv[train_idx])), float(np.nanstd(yv[train_idx]))
    Yt = [(yv - v_mu) / v_sd, np.asarray(ys[1], float), np.asarray(ys[2], float)]
    cuda = str(device).startswith("cuda")
    amp_dtype = torch.bfloat16 if (cuda and torch.cuda.is_bf16_supported()) else torch.float16
    scaler = torch.amp.GradScaler("cuda", enabled=cuda and amp_dtype == torch.float16)
    head_params = list(net.trunk.parameters()) + list(net.out.parameters())
    opt = torch.optim.AdamW([{"params": m.parameters(), "lr": lr_backbone}, {"params": head_params, "lr": lr_head}],
                            weight_decay=weight_decay)
    steps = epochs * math.ceil(len(train_idx) / batch); warm = max(1, int(0.05 * steps))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warm) * (0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * min(1.0, s / steps)))))

    def prep(ii):
        vs = views_from_packed(packed[ii], device)
        xs = [(vs[v].expand(-1, 3, -1, -1) - mu) / sd for v in VIEWS]
        yb = [torch.as_tensor(Yt[k][ii], dtype=torch.float32, device=device) for k in range(3)]
        return xs, torch.as_tensor(xnum[ii], dtype=torch.float32, device=device), yb

    def predict(idx):
        net.eval(); outs = []
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=cuda):
            for b0 in range(0, len(idx), 2 * batch):
                xs, xn, _ = prep(idx[b0:b0 + 2 * batch])
                outs.append(net(xs, xn).float().cpu().numpy())
        return np.concatenate(outs) if outs else np.zeros((0, 3))

    def val_loss(o):
        tot = 0.0
        for k in range(3):
            y = Yt[k][val_idx]; ok = np.isfinite(y)
            if ok.any():
                p = o[ok, k]
                tot += float(np.mean((p - y[ok]) ** 2)) if k == 0 else float(np.mean(np.logaddexp(0, p) - y[ok] * p))
        return tot

    hist, best, bad, best_state, best_out = [], np.inf, 0, None, None
    t_start = time.time(); n_img = 0
    for ep in range(1, epochs + 1):
        net.train(); t0 = time.time(); run = []
        order = rng.permutation(train_idx)
        for b0 in range(0, len(order), batch):
            ii = order[b0:b0 + batch]
            if len(ii) < 2:
                continue
            xs, xn, yb = prep(ii)
            with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=cuda):
                out = net(xs, xn)
            loss = sum(_ft_losses(out, yb))
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            run.append(float(loss.detach())); n_img += 3 * len(ii)
        o = predict(val_idx)
        vl = val_loss(o)
        yva = yv[val_idx]; okv = np.isfinite(yva)
        r2v = r2(yva[okv], o[okv, 0] * v_sd + v_mu) if okv.sum() > 2 else float("nan")
        yd = Yt[2][val_idx]; okd = np.isfinite(yd)
        aucd = auc(yd[okd], o[okd, 2]) if okd.sum() > 2 and 0 < np.nanmean(yd[okd]) < 1 else float("nan")
        hist.append(dict(backbone=name, epoch=ep, train_loss=float(np.mean(run)) if run else float("nan"), val_loss=vl,
                         val_R2_V=r2v, val_AUC_D10=aucd, seconds=time.time() - t0))
        log(f"   {name} epoch {ep}/{epochs}: train loss {hist[-1]['train_loss']:.4f} | val loss {vl:.4f} | "
            f"val R2 (volatility) {r2v:.3f} | val AUC (drawdown) {aucd:.3f} | {time.time() - t0:.0f}s")
        if vl < best - 1e-5:
            best, bad, best_out = vl, 0, o
            best_state = {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}
        else:
            bad += 1
            if bad > patience:
                break
    m.load_state_dict(best_state)
    m.eval()
    best_ep = int(min(hist, key=lambda h: h["val_loss"])["epoch"])
    secs = time.time() - t_start
    rec = dict(rec, ft=dict(epochs_run=len(hist), best_epoch=best_ep, minutes=secs / 60, train_img_per_s=n_img / max(secs, 1e-9),
                            n_train=int(len(train_idx)), n_val=int(len(val_idx)), lr_backbone=lr_backbone, lr_head=lr_head,
                            batch=batch, weight_decay=weight_decay, v_mean=v_mu, v_std=v_sd))
    e2e = {"V": best_out[:, 0] * v_sd + v_mu, "O5": 1 / (1 + np.exp(-best_out[:, 1])), "D10": 1 / (1 + np.exp(-best_out[:, 2]))}
    return m, mu, sd, rec, pd.DataFrame(hist), e2e


def save_backbone(m, path):
    """Backbone weights as .safetensors (plain tensors, no pickle)."""
    from safetensors.torch import save_file
    save_file({k: v.detach().cpu().contiguous().clone() for k, v in m.state_dict().items()}, path)


def assemble(out_dir, key, order_ids):
    """Rebuild one 'backbone|view' matrix (float32) in the order of order_ids from the chunk files."""
    files = sorted(f for f in os.listdir(out_dir) if f.startswith("chunk_") and f.endswith(".npz"))
    ids, parts = [], []
    for f in files:
        z = np.load(os.path.join(out_dir, f))
        ids.append(z["sample_id"]); parts.append(z[key])
    ids = np.concatenate(ids); X = np.concatenate(parts).astype(np.float32)
    pos = pd.Series(np.arange(len(ids)), index=ids)
    miss = ~pd.Index(order_ids).isin(pos.index)
    assert not miss.any(), f"{int(miss.sum())} rows have no embedding for {key}"
    return X[pos.loc[order_ids].values]


def pca_fit_transform(X, train_mask, k, seed=0, device="cpu"):
    """PCA fitted on training rows only. Returns (Z float32, mean float64, components float64, explained ratio)."""
    import torch
    mu = X[train_mask].mean(0).astype(np.float64)
    q = int(min(k + 10, X.shape[1]))
    torch.manual_seed(seed)
    Xt = torch.from_numpy((X[train_mask] - mu).astype(np.float32)).to(device)
    U, S, V = torch.pca_lowrank(Xt, q=q, center=False, niter=4)
    comp = V[:, :k].double().cpu().numpy()
    tot = float((Xt.double() ** 2).sum().cpu()); expl = float((S[:k].double() ** 2).sum().cpu()) / max(tot, 1e-12)
    del Xt, U, S, V
    Z = ((X - mu) @ comp).astype(np.float32)
    return Z, mu, comp, expl


# ===================================================================================== 3. heads
def standardize(Xtr):
    mu = np.nanmean(Xtr, 0); sd = np.nanstd(Xtr, 0); sd[~np.isfinite(sd) | (sd < 1e-12)] = 1.0
    return mu, sd


def ridge_path(Xtr, ytr, Xva, alphas):
    """Ridge with an unpenalised intercept for each alpha (sklearn convention). Returns {alpha: (coef, intercept, val_pred)}."""
    Xtr = np.asarray(Xtr, np.float64); xm = Xtr.mean(0); ym = float(np.mean(ytr))
    Xc = Xtr - xm
    G = Xc.T @ Xc; b = Xc.T @ (np.asarray(ytr, np.float64) - ym)
    out = {}
    for a in alphas:
        w = np.linalg.solve(G + a * np.eye(G.shape[0]), b)
        out[a] = (w, ym - xm @ w, np.asarray(Xva, np.float64) @ w + (ym - xm @ w))
    return out


def logit_path(Xtr, ytr, Xva, Cs, device="cpu", max_iter=300):
    """L2 logistic regression (sklearn C convention, unpenalised intercept), fitted by full-batch L-BFGS along the C path
    from strong to weak regularisation with warm starts. Returns {C: (coef, intercept, val_prob)}."""
    import torch
    dt = torch.float32 if str(device).startswith("cuda") else torch.float64
    X = torch.as_tensor(np.asarray(Xtr), dtype=dt, device=device); y = torch.as_tensor(np.asarray(ytr), dtype=dt, device=device)
    Xv = torch.as_tensor(np.asarray(Xva), dtype=dt, device=device)
    n, d = X.shape
    w = torch.zeros(d, dtype=dt, device=device, requires_grad=True); b = torch.zeros(1, dtype=dt, device=device, requires_grad=True)
    out = {}
    for C in sorted(Cs):
        lam = 1.0 / (C * n)
        opt = torch.optim.LBFGS([w, b], lr=1.0, max_iter=max_iter, tolerance_grad=1e-9, tolerance_change=1e-12,
                                history_size=20, line_search_fn="strong_wolfe")

        def closure():
            opt.zero_grad()
            loss = torch.nn.functional.binary_cross_entropy_with_logits(X @ w + b, y) + 0.5 * lam * (w * w).sum()
            loss.backward()
            return loss
        opt.step(closure)
        with torch.no_grad():
            pv = torch.sigmoid(Xv @ w + b).double().cpu().numpy()
        out[C] = (w.detach().double().cpu().numpy().copy(), float(b.detach().cpu()), pv)
    return out


TREE_GRID = [dict(max_depth=3, learning_rate=0.05, min_child_weight=100),
             dict(max_depth=5, learning_rate=0.05, min_child_weight=100),
             dict(max_depth=7, learning_rate=0.05, min_child_weight=300),
             dict(max_depth=5, learning_rate=0.10, min_child_weight=1000)]


def tree_backend():
    try:
        import xgboost  # noqa
        return "xgb"
    except Exception:
        return "lgbm"


def tree_fit(Xtr, ytr, Xva, yva, task, cfg, gpu, seed=11, n_estimators=None):
    """One boosted-tree head. With Xva/yva: early stopping on validation. With n_estimators: fixed size (refit).
    Returns (model, best_n, val_pred or None)."""
    be = tree_backend()
    early = n_estimators is None
    n_est = 2000 if early else int(n_estimators)
    if be == "xgb":
        import xgboost as xgb
        kw = dict(n_estimators=n_est, learning_rate=cfg["learning_rate"], max_depth=cfg["max_depth"],
                  min_child_weight=cfg["min_child_weight"], subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                  tree_method="hist", device="cuda" if gpu else "cpu", random_state=seed, n_jobs=-1)
        if early:
            kw["early_stopping_rounds"] = 100
        if task == "reg":
            m = xgb.XGBRegressor(objective="reg:squarederror", eval_metric="rmse", **kw)
        else:
            m = xgb.XGBClassifier(objective="binary:logistic", eval_metric="logloss", **kw)
        if early:
            m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False); best = int(m.best_iteration) + 1
        else:
            m.fit(Xtr, ytr, verbose=False); best = n_est
    else:
        import lightgbm as lgb
        kw = dict(n_estimators=n_est, learning_rate=cfg["learning_rate"], max_depth=cfg["max_depth"], num_leaves=min(2 ** cfg["max_depth"], 127),
                  min_child_samples=int(cfg["min_child_weight"]), subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                  reg_lambda=1.0, random_state=seed, n_jobs=-1, verbose=-1)
        m = lgb.LGBMRegressor(**kw) if task == "reg" else lgb.LGBMClassifier(**kw)
        if early:
            m.fit(Xtr, ytr, eval_set=[(Xva, yva)], callbacks=[lgb.early_stopping(100, verbose=False)])
            best = int(m.best_iteration_ or n_est)
        else:
            m.fit(Xtr, ytr); best = n_est
    pv = None
    if Xva is not None:
        pv = m.predict(Xva) if task == "reg" else m.predict_proba(Xva)[:, 1]
    return m, best, pv


def tree_save(model, path):
    if tree_backend() == "xgb":
        model.get_booster().save_model(path); return "xgb"
    model.booster_.save_model(path); return "lgbm"


def tree_predict(model, X, task):
    return model.predict(X) if task == "reg" else model.predict_proba(X)[:, 1]


# ===================================================================================== 4. metrics and month-block bootstrap
def r2(y, p):
    y = np.asarray(y, float); p = np.asarray(p, float)
    return float(1 - ((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum())


def month_codes(dates):
    m = pd.to_datetime(pd.Series(dates)).dt.to_period("M").astype(str).values
    u = np.unique(m)
    return np.searchsorted(u, m), len(u)


def wm_r2(y, p, c):
    """Within-month R2: demean target and prediction by calendar month, best scalar rescale (cross-sectional skill only)."""
    y = np.asarray(y, float); p = np.asarray(p, float); K = int(c.max()) + 1
    n = np.maximum(np.bincount(c, minlength=K), 1)
    yd = y - (np.bincount(c, y, minlength=K) / n)[c]; pd_ = p - (np.bincount(c, p, minlength=K) / n)[c]
    ss = (pd_ ** 2).sum(); b = (yd * pd_).sum() / ss if ss > 0 else 0.0
    return float(1 - ((yd - b * pd_) ** 2).sum() / (yd ** 2).sum())


def qlike(y_log, p_log, s2):
    """QLIKE on variance: RV = exp(2 y), forecast F = exp(2 p + 2 s2) (log-normal bias correction, s2 from validation)."""
    rv = np.exp(2 * np.asarray(y_log, float)); f = np.exp(2 * np.asarray(p_log, float) + 2 * s2)
    q = rv / f
    return float(np.mean(q - np.log(q) - 1))


def auc(y, p):
    from sklearn.metrics import roc_auc_score
    return float(roc_auc_score(np.asarray(y, int), np.asarray(p, float)))


def logloss(y, p):
    p = np.clip(np.asarray(p, float), 1e-6, 1 - 1e-6); y = np.asarray(y, float)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


def brier(y, p):
    return float(np.mean((np.asarray(p, float) - np.asarray(y, float)) ** 2))


def rmse(y, p):
    return float(np.sqrt(np.mean((np.asarray(y, float) - np.asarray(p, float)) ** 2)))


def spearman(y, p):
    return float(pd.Series(np.asarray(y, float)).rank().corr(pd.Series(np.asarray(p, float)).rank()))


def avg_precision(y, p):
    from sklearn.metrics import average_precision_score
    return float(average_precision_score(np.asarray(y, int), np.asarray(p, float)))


def accuracy(y, p):
    return float(((np.asarray(p, float) > 0.5) == (np.asarray(y, float) > 0.5)).mean())


def board_metrics(task, y, p, dates, base=None, s2=None):
    """The leaderboard's metric set. Regression: R2, within-month R2, QLIKE, MAE, RMSE, Spearman. Classification: AUC,
    average precision, accuracy, log loss, Brier skill score, ECE."""
    y = np.asarray(y, float); p = np.asarray(p, float)
    if task == "reg":
        c, _ = month_codes(dates)
        s2 = float(np.var(y - p)) if s2 is None else s2
        return dict(R2=r2(y, p), wmR2=wm_r2(y, p, c), QLIKE=qlike(y, p, s2), MAE=float(np.mean(np.abs(y - p))), RMSE=rmse(y, p),
                    Spearman=spearman(y, p))
    base = float(np.mean(y)) if base is None else base
    return dict(AUC=auc(y, p), AP=avg_precision(y, p), ACC=accuracy(y, p), logloss=logloss(y, p),
                BSS=1 - brier(y, p) / brier(y, np.full(len(y), base)), ECE=ece(y, p))


# ===================================================================================== 4b. Monte Carlo calibration (notebook side)
def mc_universal(n, t_df, res_sampler, seed=0, horizon=20, low_k=1.0):
    """Volatility-free draws of the dashboard's Monte Carlo model (tfcv_core.simulate_paths):
    U = 20-day log return per unit of forecast volatility, D = lowest intraday log price per unit of forecast volatility.
    For a forecast x_hat (log 20-day volatility): log return = exp(x_hat) * U and log(low / start) = exp(x_hat) * D."""
    import tfcv_core as C
    rng = np.random.default_rng(seed)
    eps = res_sampler(rng, n)
    cum, low = C.mc_unit_paths(n, t_df, rng, horizon, low_k)
    s = np.exp(eps)
    return s * cum[:, -1], s * low.min(1)


def _tilt_rows(H, R, p, iters=40):
    """tfcv_core.tilt_weights for many rows at once: H (rows, draws) drawdown hits, R (rows, draws) 20-day returns,
    p (rows,) target drawdown chances. Keeps each row's mean return; returns the weights (rows, draws)."""
    h = H.astype(np.float64)
    r = (R - R.mean(1, keepdims=True)) / (R.std(1, keepdims=True) + 1e-12)
    a = np.zeros(len(p)); b = np.zeros(len(p))
    for _ in range(iters):
        z = a[:, None] * h + b[:, None] * r
        w = np.exp(z - z.max(1, keepdims=True)); w /= w.sum(1, keepdims=True)
        mh, mr = (w * h).sum(1), (w * r).sum(1)
        g1, g2 = mh - p, mr
        if max(np.abs(g1).max(), np.abs(g2).max()) < 1e-6:
            break
        dh, dr = h - mh[:, None], r - mr[:, None]
        j11, j12, j22 = (w * dh * dh).sum(1) + 1e-10, (w * dh * dr).sum(1), (w * dr * dr).sum(1) + 1e-10
        det = j11 * j22 - j12 ** 2
        a = np.clip(a - np.clip((j22 * g1 - j12 * g2) / det, -2, 2), -30, 30)
        b = np.clip(b - np.clip((-j12 * g1 + j11 * g2) / det, -2, 2), -30, 30)
    return w


def mc_scores(U, D, xhat, logret, dd_hit, p_model=None, levels=(0.5, 0.8, 0.9, 0.95), chunk=200, n_tilt=6000):
    """How well the simulation describes real outcomes. For each row: the probability integral transform (PIT) of the realised
    20-day log return, and the simulated chance of a 10% drawdown. With p_model (the calibrated drawdown model), the paths
    are reweighted per row exactly as the dashboard's 'match the drawdown model' option does (drawdown share = p_model, mean
    return unchanged). Returns {coverage at each level, mean absolute coverage error, PIT histogram, drawdown Brier}."""
    xhat = np.asarray(xhat, float); scale = np.exp(xhat)
    u_row = np.asarray(logret, float) / scale
    c_row = math.log(0.9) / scale
    if p_model is None:
        pit = np.searchsorted(np.sort(U), u_row, side="right") / len(U)
        p_dd = np.searchsorted(np.sort(D), c_row, side="right") / len(D)
    else:
        Ut, Dt = U[:n_tilt], D[:n_tilt]
        pit = np.empty(len(xhat)); p_dd = np.asarray(p_model, float).copy()
        for a in range(0, len(xhat), chunk):
            sl = slice(a, a + chunk)
            H = Dt[None, :] <= c_row[sl, None]
            R = np.expm1(scale[sl, None] * Ut[None, :])
            w = _tilt_rows(H, R, np.clip(p_dd[sl], 1e-4, 1 - 1e-4))
            pit[sl] = (w * (Ut[None, :] <= u_row[sl, None])).sum(1)
    cov = {f"{int(l * 100)}%": float(np.mean((pit >= (1 - l) / 2) & (pit <= 1 - (1 - l) / 2))) for l in levels}
    err = float(np.mean([abs(cov[f"{int(l * 100)}%"] - l) for l in levels]))
    hist = np.histogram(pit, bins=10, range=(0, 1))[0] / max(len(pit), 1)
    ok = np.isfinite(dd_hit)
    return dict(coverage=cov, coverage_error=err, pit_hist=hist.tolist(), mean_pit=float(np.mean(pit)),
                dd_brier=brier(np.asarray(dd_hit)[ok], p_dd[ok]), dd_rate=float(np.mean(np.asarray(dd_hit)[ok])), dd_mean_pred=float(np.mean(p_dd[ok])))


def ece(y, p, bins=10):
    y = np.asarray(y, float); p = np.asarray(p, float)
    e = np.clip((p * bins).astype(int), 0, bins - 1); tot = 0.0
    for k in range(bins):
        m = e == k
        if m.any():
            tot += m.mean() * abs(p[m].mean() - y[m].mean())
    return float(tot)


def rank_ic(dates, p, x, min_names=20):
    """Mean cross-sectional Spearman correlation per decision date, and a month-block t-statistic."""
    df = pd.DataFrame({"d": pd.to_datetime(pd.Series(dates)).values, "p": np.asarray(p, float), "x": np.asarray(x, float)})
    g = df.groupby("d")
    ic = g.apply(lambda z: z["p"].rank().corr(z["x"].rank()) if len(z) >= min_names else np.nan, include_groups=False).dropna()
    if len(ic) < 3:
        return float("nan"), float("nan")
    mon = ic.index.to_period("M"); mm = ic.groupby(mon).mean()
    t = mm.mean() / (mm.std(ddof=1) / np.sqrt(len(mm))) if len(mm) > 2 else float("nan")
    return float(ic.mean()), float(t)


class MonthBoot:
    """Month-block bootstrap on one evaluation set: resample calendar months with replacement, keep all their rows.
    The same resamples are used for every metric and every model comparison on that set."""

    def __init__(self, dates, n=1000, seed=42):
        self.c, self.K = month_codes(dates)
        rows = [np.flatnonzero(self.c == k) for k in range(self.K)]
        rng = np.random.default_rng(seed)
        self.idx = [np.concatenate([rows[k] for k in rng.integers(0, self.K, self.K)]) for _ in range(n)]

    def ci(self, fn, *arrays, uses_months=False):
        v = [fn(*[a[i] for a in arrays], self.c[i]) if uses_months else fn(*[a[i] for a in arrays]) for i in self.idx]
        return [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))], np.asarray(v)

    def diff(self, fn, y, p1, p2, uses_months=False):
        if uses_months:
            v = [fn(y[i], p1[i], self.c[i]) - fn(y[i], p2[i], self.c[i]) for i in self.idx]
        else:
            v = [fn(y[i], p1[i]) - fn(y[i], p2[i]) for i in self.idx]
        v = np.asarray(v)
        return [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))], v


def holm(pvals):
    """Holm step-down adjusted p-values (dict in, dict out)."""
    items = sorted(pvals.items(), key=lambda kv: kv[1]); m = len(items); adj = {}; run = 0.0
    for i, (k, p) in enumerate(items):
        run = max(run, min(1.0, (m - i) * p)); adj[k] = run
    return adj


def wilson_lower(k, n, z=1.96):
    if n == 0:
        return 0.0
    ph = k / n; den = 1 + z * z / n
    return float((ph + z * z / (2 * n) - z * math.sqrt(ph * (1 - ph) / n + z * z / (4 * n * n))) / den)


def isotonic_arrays(p, y):
    from sklearn.isotonic import IsotonicRegression
    ir = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0).fit(np.asarray(p, float), np.asarray(y, float))
    return {"x": ir.X_thresholds_.tolist(), "y": ir.y_thresholds_.tolist()}


def abstention_table(p, y, thresholds=None, coverages=(1.0, 0.5, 0.25, 0.10)):
    """Accuracy of direction calls among the most confident predictions. thresholds: {coverage: confidence threshold}
    fixed on validation (pass None on validation itself to compute them)."""
    p = np.asarray(p, float); y = np.asarray(y, int); conf = np.abs(p - 0.5); rows = []; thr = {}
    for c in coverages:
        t = 0.0 if c == 1.0 else (float(np.quantile(conf, 1 - c)) if thresholds is None else thresholds[c])
        thr[c] = t
        keep = conf >= t; n = int(keep.sum()); k = int(((p[keep] > 0.5) == y[keep]).sum())
        rows.append(dict(coverage_target=c, coverage_realised=n / len(p), n=n, accuracy=k / max(n, 1), wilson_lower=wilson_lower(k, n)))
    return pd.DataFrame(rows), thr
''')
print('wrote', 'tfcv_train.py')

In [ ]:
# writes screen_reader.py (used by this notebook and, later, by the dashboard)
open(os.path.join(WORK, 'screen_reader.py'), 'w', encoding='utf-8').write(r'''"""Reads ticker candidates, timeframe and the last-bar O/H/L/C legend from a TradingView-style chart screenshot.
Tested on 66 real TradingView dark-theme screenshots (22 tickers x 1W/1D/1H): timeframe 100 %, legend O/H/L/C 100 %,
true ticker among the candidates 66/66. The app confirms a candidate by matching the legend against market data."""
import re
import numpy as np
from PIL import Image

_OCR = None
TF_MAP = {"1W": "1W", "W": "1W", "1D": "1D", "D": "1D", "1H": "1H", "60": "1H"}
STOP = {"USD", "SELL", "BUY", "VOL", "SMA", "EMA", "NASDAQ", "NYSE", "AMEX", "ARCA", "BATS", "OTC", "TRADE", "SAVE", "ALERT",
        "PRE", "POST", "MACD", "RSI", "LOG", "AUTO", "INC", "CORP", "LTD", "CO", "PLC", "THE", "AND", "OF", "CLOSE", "OPEN",
        "HIGH", "LOW", "ADJ", "EXT", "UTC"}


def _ocr():
    global _OCR
    if _OCR is None:
        from rapidocr_onnxruntime import RapidOCR
        _OCR = RapidOCR()
    return _OCR


def _num(s):
    try:
        return float(s.replace(",", ""))
    except Exception:
        return None


def _rows(items):
    """Group OCR items (box, text, score) into text rows by vertical centre, each row sorted left to right."""
    boxes = []
    for box, text, _ in items:
        ys = [p[1] for p in box]; xs = [p[0] for p in box]
        boxes.append((sum(ys) / 4, max(ys) - min(ys), min(xs), text.strip()))
    boxes.sort()
    rows, cur = [], []
    for b in boxes:
        if cur and abs(b[0] - cur[-1][0]) > 0.6 * max(b[1], cur[-1][1], 1):
            rows.append(sorted(cur, key=lambda z: z[2])); cur = []
        cur.append(b)
    if cur:
        rows.append(sorted(cur, key=lambda z: z[2]))
    return [[z[3] for z in r if z[3]] for r in rows]


def _toolbar_candidates(rows):
    out = []
    for r in rows[:3]:
        for j, t in enumerate(r):
            if re.search(r"[OHLC0][\d,]+\.\d", t) or "·" in t:            # legend values or the name line
                continue
            for tok in re.findall(r"[A-Za-z][A-Za-z\.\-]{0,5}", t):
                up = tok.upper().strip(".-")
                if not (1 <= len(up) <= 5) or up in STOP or up in TF_MAP:
                    continue
                if sum(c.islower() for c in tok) > 1:                      # ordinary words ("Replay", "Indicators")
                    continue
                if len(up) == 1 and j == 0 and len(r) > 1:                 # the account avatar at the far left
                    continue
                up = up.replace(".", "-")
                if up not in out:
                    out.append(up)
    return out


def _price_tag_candidates(img):
    """The ticker in the price-axis tag on the right ("BRKR 58.84"), for screenshots taken without the toolbar."""
    w, h = img.size
    right = img.crop((int(w * 0.72), 0, w, h))
    if right.size[0] < 800:
        right = right.resize((right.size[0] * 2, right.size[1] * 2), Image.LANCZOS)
    res, _ = _ocr()(np.asarray(right))
    out = []
    for box, text, _ in sorted(res or [], key=lambda r: min(p[1] for p in r[0])):
        tok = text.strip()
        if re.fullmatch(r"[A-Z][A-Z\.\-]{0,5}", tok) and tok not in STOP and tok.replace(".", "-") not in out:
            out.append(tok.replace(".", "-"))
    return out


def read_header(img, deep=False):
    """img: PIL.Image or path. Returns dict(candidates, timeframe, ohlc, name, header_text).
    deep=True also reads the price-axis tag (slower; used when the toolbar candidates cannot be confirmed)."""
    if not isinstance(img, Image.Image):
        img = Image.open(img)
    img = img.convert("RGB")
    w, h = img.size
    top = img.crop((0, 0, int(w * 0.75), max(60, int(h * 0.12))))
    if top.size[0] < 1600:
        top = top.resize((top.size[0] * 2, top.size[1] * 2), Image.LANCZOS)
    res, _ = _ocr()(np.asarray(top))
    rows = _rows(res or [])
    joined = " | ".join(" ".join(r) for r in rows)

    tf = None
    m = re.search(r"[·•]\s*(1W|1D|1H|60)\s*[·•]", joined, re.I)
    if m:
        tf = TF_MAP.get(m.group(1).upper())
    if tf is None and rows:
        for t in rows[0]:
            if t.strip().upper() in TF_MAP:
                tf = TF_MAP[t.strip().upper()]; break

    ohlc = None
    flat = re.sub(r"\s+", "", joined)
    m = re.search(r"[O0]([\d,]*\d\.\d+)H([\d,]+\.\d+)L([\d,]+\.\d+)C([\d,]+\.\d+)", flat)
    if m:
        vals = [_num(x) for x in m.groups()]
        if all(v is not None for v in vals) and vals[1] >= max(vals[0], vals[3]) * 0.999 and vals[2] <= min(vals[0], vals[3]) * 1.001:
            ohlc = dict(zip(("open", "high", "low", "close"), vals))

    name = None
    m = re.search(r"([A-Za-z][^|·•]{2,80}?)\s*[·•]\s*(?:1W|1D|1H|60)\b", joined, re.I)
    if m:
        name = re.sub(r"^[^A-Za-z]+", "", m.group(1)).strip()

    cands = _toolbar_candidates(rows)
    if deep or not cands:
        cands += [c for c in _price_tag_candidates(img) if c not in cands]
    return dict(candidates=cands, timeframe=tf, ohlc=ohlc, name=name, header_text=joined[:300])
''')
print('wrote', 'screen_reader.py')

In [ ]:
# writes dashboard_logic.py (used by this notebook and, later, by the dashboard)
open(os.path.join(WORK, 'dashboard_logic.py'), 'w', encoding='utf-8').write(r'''"""Dashboard logic shared by the Streamlit app (streamlit_app.py) and the Gradio app (app.py).

Flow per company: three TradingView screenshots (1W, 1D, 1H) -> read ticker candidates, timeframe and last-bar legend from each
image -> confirm the ticker by matching the legend against Yahoo Finance bars -> rebuild the exact model inputs (33 numeric
features + the three training-style charts) as of the screenshot's date -> frozen model bundle -> outputs, NO TRADE decision,
risk tier, plain-language explanation and a Monte Carlo simulation of the next 20 sessions. Up to four companies are compared."""
import math
import os
import time
import traceback

import numpy as np
import pandas as pd
from PIL import Image

import tfcv_core as C
from screen_reader import read_header

TITLE = "Computer Vision with Deep Learning Project Dashboard"
MODEL_REPO_DEFAULT = "KrishnaHarshaTangirala/triframe-cv-model"
CHECK_COLUMNS = ["slot", "ticker candidates", "timeframe", "legend read"]
MAX_PIXELS = 40_000_000                 # refuse absurdly large uploads before OCR touches them
MARKET_TTL = 600                        # seconds a ticker's market data is reused (spares Yahoo, speeds up repeat clicks)
_MARKET = {}
SLOTS = ("1W", "1D", "1H")

LIMITS = """
*Limits: the models were trained on survivor stocks from one data vendor (stocks that later failed are missing, so risk may be
understated); live inputs come from Yahoo Finance. Research output, not investment advice. NO TRADE is the default decision.*
"""

# risk tiers on the risk score = the larger of (volatility / its NO TRADE threshold) and (drawdown chance / its threshold);
# 1.0 is the NO TRADE risk line (the top 20 % of the model's readings in the validation years)
TIERS = (("Very high", 1.5, "#b71c1c"), ("High", 1.0, "#e53935"), ("Moderate", 0.75, "#f39c12"), ("Low", 0.0, "#2e9d5b"))

BACKBONE_NAMES = {
    "resnet18": "ResNet-18", "efficientnet_b0": "EfficientNet-B0", "vit_base_patch16_224": "ViT-B/16",
    "convnext_tiny": "ConvNeXt-Tiny", "swin_tiny_patch4_window7_224": "Swin-T", "vit_small_patch14_dinov2": "DINOv2 ViT-S/14",
    "mobilenetv3_large_100": "MobileNetV3-Large",
}


# ------------------------------------------------------------------------------------------------ model bundle
def load_bundle(token=None, model_repo=None):
    """Load the frozen bundle from BUNDLE_DIR (local folder) or from the private Hugging Face model repo."""
    path = os.environ.get("BUNDLE_DIR")
    if not path:
        from huggingface_hub import snapshot_download
        path = snapshot_download(model_repo or os.environ.get("MODEL_REPO", MODEL_REPO_DEFAULT), token=token or os.environ.get("HF_TOKEN"))
    return C.Bundle(path)


def _fmt_pct(x, d=0):
    return "n/a" if x is None or not np.isfinite(x) else f"{100 * x:.{d}f}%"


def _metric(cfg, path, default=None):
    cur = cfg.get("metrics", {})
    for p in path:
        if not isinstance(cur, dict) or p not in cur:
            return default
        cur = cur[p]
    return cur if not (isinstance(cur, float) and not np.isfinite(cur)) else default


def _years(cfg, split):
    a, b = cfg.get("splits", {}).get(split, ("", ""))
    if not a:
        return {"train": "2001-2016", "val": "2017-2019", "test": "2020-2022"}[split]
    last = int(b[:4]) - 1 if b[5:7] == "01" else int(b[:4])
    return f"{a[:4]}-{last}"


def backbone_label(b):
    """'ft_convnext_tiny' -> 'ConvNeXt-Tiny (fine-tuned)'; 'swin_tiny_patch4_window7_224' -> 'Swin-T (frozen)'."""
    ft = b.startswith("ft_")
    base = b[3:] if ft else b
    return f"{BACKBONE_NAMES.get(base, base)} ({'fine-tuned' if ft else 'frozen'})"


def pretrain_words(timm_name):
    tag = timm_name.split(".", 1)[1] if "." in timm_name else ""
    for k, w in (("lvd142m", "LVD-142M (self-supervised)"), ("in21k_ft_in1k", "ImageNet-21k + 1k"),
                 ("in12k_ft_in1k", "ImageNet-12k + 1k"), ("in1k", "ImageNet-1k")):
        if k in tag:
            return w
    return "ImageNet"


def models_in_use(cfg):
    """One row per output: what is predicted, the inputs, the network behind the chart features, the head, and its test score."""
    q = {"V": "Volatility over the next 20 trading days", "D10": "Chance of a 10%+ drawdown within 20 trading days",
         "O5": "Chance of beating the median stock over 5 trading days"}
    score = {"V": lambda: f"R² {_metric(cfg, ['test', 'V', 'R2'], float('nan')):.2f}",
             "D10": lambda: f"AUC {_metric(cfg, ['test', 'D10', 'AUC'], float('nan')):.2f}",
             "O5": lambda: f"AUC {_metric(cfg, ['test', 'O5', 'AUC'], float('nan')):.3f}"}
    rows = []
    for t in ("V", "D10", "O5"):
        s = cfg["targets"][t]
        use_num, b, views = C.Bundle.parse_fs(s["feature_set"])
        if b is None:
            inputs, net = "33 numbers from the last 126 daily bars", "none (numbers only)"
        else:
            vw = "weekly + daily + hourly charts" if len(views) == 3 else f"{'/'.join(views)} chart"
            inputs = ("33 numbers + " if use_num else "") + vw
            net = backbone_label(b) + f", pretrained on {pretrain_words(cfg.get('backbones', {}).get(b, {}).get('timm_name', ''))}"
        head = {"linear": "linear (ridge / logistic)", "xgb": "gradient-boosted trees (XGBoost)",
                "lgbm": "gradient-boosted trees (LightGBM)"}.get(s["head"], s["head"])
        rows.append({"Output": q[t], "Inputs": inputs, "Vision network": net, "Head": head,
                     "Test score": score[t]()})
    return pd.DataFrame(rows)


def describe(B):
    cfg = B.cfg
    d = cfg.get("data", {})
    scope = (f"The models in use come from a **run on {d.get('tickers_used', '?')} of {d.get('universe', '?')} stocks** "
             f"({d.get('rows', 0):,} chart sets)."
             if d.get("tickers_used") and d.get("universe") and d["tickers_used"] != d["universe"] else
             f"The models were trained on {d.get('universe', '?')} stocks.")
    return f"""
**TriFrame** asks whether chart images at three time scales (weekly, daily and hourly) add information about a US stock's
next-20-day volatility, drawdown risk and 5-day relative direction beyond plain numbers computed from the same price history.
Seven ImageNet-pretrained vision networks were compared with a numerical baseline, using chronological training, validation and
test periods and a sealed final holdout. {scope}

**How to use:** choose how many companies to analyse (1 to 4). For each one, upload three TradingView screenshots of the same
stock: weekly (1W), daily (1D) and hourly (1H), or just type the ticker. The app reads the ticker and the last bar of each
picture, checks them against market data, rebuilds the exact chart images and numbers the models were trained on, and reports
risk, a plain-language explanation and a Monte Carlo simulation of the next 20 trading days for every company.
"""


# ------------------------------------------------------------------------------------------------ market data
def market_data(ticker):
    """Daily + regular-hours hourly bars for a ticker (cached for MARKET_TTL seconds, retried on transient failures)."""
    now = time.time()
    hit = _MARKET.get(ticker)
    if hit and now - hit[0] < MARKET_TTL:
        return hit[1], hit[2]
    last = None
    for attempt, wait in enumerate((0, 1.5, 4)):
        if wait:
            time.sleep(wait)
        try:
            d, h = C.fetch_yahoo(ticker)
            d = C.drop_incomplete_session(d)
            if len(d) < C.FV.W + 5:
                raise ValueError(f"only {len(d)} daily bars for {ticker}")
            _MARKET[ticker] = (now, d, h)
            return d, h
        except ValueError as e:                  # unknown ticker or too little history: one quick retry, then give up
            last = e
            if attempt >= 1:
                break
        except Exception as e:                   # network / rate limit: keep retrying
            last = e
    raise last


def confirm(cands, typed, reads):
    """First candidate whose market data match at least one screenshot legend. A ticker typed by the user is accepted
    on the latest completed session even if no legend matches (for example a screenshot taken mid-session).
    Returns (ticker, daily index, how, daily, hourly) or (None,)*5."""
    for tk in cands:
        try:
            d, h = market_data(tk)
        except Exception:
            continue
        found = {}
        if (reads.get("1D") or {}).get("ohlc"):
            found["1D"] = C.match_daily(d, reads["1D"]["ohlc"])
        if (reads.get("1W") or {}).get("ohlc"):
            found["1W"] = C.match_weekly(d, reads["1W"]["ohlc"])
        if (reads.get("1H") or {}).get("ohlc"):
            found["1H"] = C.match_hourly(d, h, reads["1H"]["ohlc"])
        for slot in ("1D", "1W", "1H"):
            if found.get(slot) is not None:
                return tk, found[slot], f"matched the {slot} screenshot's last bar", d, h
        if tk == typed:
            why = "the screenshots' last bars did not match market data" if found else "no screenshot legend was readable"
            return tk, len(d) - 1, f"latest completed session; {why}", d, h
    return None, None, None, None, None


# ------------------------------------------------------------------------------------------------ one company
def _check_image(im):
    if im is None:
        return None
    if not isinstance(im, Image.Image):
        im = Image.open(im)
    if im.size[0] * im.size[1] > MAX_PIXELS:
        raise ValueError("That image is too large; please upload a normal screenshot.")
    return im.convert("RGB")


def analyse_company(B, images, ticker_text=""):
    """images: {"1W": file or PIL image or None, "1D": ..., "1H": ...}. Returns a result dict:
    ok, message, ticker, asof, how, out (model outputs + decision + reasons), p0 (as-of close), charts (PIL), checks (DataFrame)."""
    cfg = B.cfg
    checks = []
    frame = lambda: pd.DataFrame(checks, columns=CHECK_COLUMNS) if checks else None
    res = dict(ok=False, message="", ticker=None, asof=None, how=None, out=None, p0=None, charts=None, checks=None)
    try:
        imgs = {s: _check_image((images or {}).get(s)) for s in SLOTS}
        typed = (ticker_text or "").strip().upper().replace(".", "-")
        if all(v is None for v in imgs.values()) and not typed:
            res["message"] = "Upload the three screenshots (or type a ticker) and press Analyse."
            return res
        reads = {}
        for slot, im in imgs.items():
            if im is None:
                checks.append([slot, "-", "-", "no image"]); continue
            r = read_header(im)
            reads[slot] = r
            note = "ok" if r["timeframe"] in (None, slot) else f"looks like a {r['timeframe']} chart"
            lg = "-" if not r["ohlc"] else f"O {r['ohlc']['open']:g} H {r['ohlc']['high']:g} L {r['ohlc']['low']:g} C {r['ohlc']['close']:g}"
            checks.append([slot, ", ".join(r["candidates"][:4]) or "-", r["timeframe"] or "-", f"{lg} ({note})"])

        cands = [typed] if typed else []
        for s in ("1D", "1W", "1H"):
            for c in (reads.get(s) or {}).get("candidates", []):
                if c not in cands:
                    cands.append(c)
        tk, k, how, d, h = confirm(cands, typed, reads)
        if tk is None:                                           # slower pass: also read the price-axis tag
            deep = []
            for im in imgs.values():
                if im is not None:
                    deep += [c for c in read_header(im, deep=True)["candidates"] if c not in cands and c not in deep]
            tk, k, how, d, h = confirm(deep, typed, reads)
        if tk is None:
            res.update(message="Could not confirm the ticker from the screenshots. Type the ticker in the box and try again. "
                               "(If a valid ticker keeps failing, Yahoo Finance may be rate-limiting this server; try again in a few minutes.)",
                       checks=frame())
            return res

        x = C.features_at(d, k)
        trip = C.render_at(d, h, k)
        emb = {b: C.embed_views(trip, cfg["backbones"][b], B.path) for b in B.needed_backbones()}
        out = B.predict_row(x, emb)
        res.update(ok=True, ticker=tk, asof=pd.Timestamp(d["date"].iloc[k]).date(), how=how, out=out,
                   p0=float(d["close"].iloc[k]), charts=C.triplet_png(trip), checks=frame())
        return res
    except ValueError as e:
        res.update(message=f"{e}", checks=frame())
        return res
    except Exception as e:
        traceback.print_exc()
        res.update(message=f"Something went wrong: {type(e).__name__}: {e}", checks=frame())
        return res


# ------------------------------------------------------------------------------------------------ risk tier, Monte Carlo
def _percentile(cfg, key, value):
    """Where a reading falls among the model's own validation-period readings, in percent (None if the bundle has no table)."""
    rq = cfg.get("risk_quantiles") or {}
    if key not in rq or not rq.get("levels"):
        return None
    return float(100 * np.interp(value, rq[key], rq["levels"]))


def _pct_words(pct, levels_top=95, levels_bottom=5):
    if pct is None:
        return ""
    if pct >= levels_top:
        return f"in the top {100 - levels_top}% of the model's readings"
    if pct <= levels_bottom:
        return f"in the bottom {levels_bottom}% of the model's readings"
    return f"higher than about {pct:.0f}% of the model's readings"


def risk_profile(out, cfg):
    """Risk tier from the two risk readings, each divided by its NO TRADE threshold (1.0 = the top 20 % of validation readings)."""
    rule = cfg["rule"]
    vr, dr = out["vol_ann"] / rule["vol_ann_q80"], out["p_dd10"] / rule["p_dd10_q80"]
    score = max(vr, dr)
    tier, color = next((n, c) for n, lo, c in TIERS if score >= lo)
    return dict(score=float(score), tier=tier, color=color, vol_ratio=float(vr), dd_ratio=float(dr),
                vol_pct=_percentile(cfg, "vol_ann", out["vol_ann"]), dd_pct=_percentile(cfg, "p_dd10", out["p_dd10"]),
                flagged=bool(score >= 1.0))


def run_mc(res, cfg, n_paths=5000, match_dd=True, seed=7):
    """Monte Carlo simulation of the next 20 sessions for one analysed company (see tfcv_core.simulate_paths)."""
    out = res["out"]
    return C.simulate_paths(res["p0"], out["logrv20"], cfg["targets"]["V"], mc=cfg.get("mc"), n_paths=int(n_paths),
                            p_dd_target=out["p_dd10"] if match_dd else None, seed=int(seed))


def _one_in(p):
    if p >= 0.95:
        return "close to certain by the model's reckoning"
    if p >= 0.5:
        return "more likely than not"
    if p < 0.04:
        return "less than 1 in 25"
    return f"about 1 in {max(2, round(1 / p))}"


# ------------------------------------------------------------------------------------------------ plain-language explanation
def explain(res, cfg, sim=None):
    """The detailed, plain-language reading of one company's outputs. Returns sections (title + labelled paragraphs), the
    summary table rows and the primary takeaway. Every number comes from the model, the bundle or the simulation."""
    out, tk = res["out"], res["ticker"]
    rule, rp = cfg["rule"], risk_profile(res["out"], cfg)
    v, lo, hi = out["vol_ann"], out["vol_ann_lo"], out["vol_ann_hi"]
    day, month = v / math.sqrt(252), v * math.sqrt(20 / 252)
    ref = cfg.get("reference", {})
    yv, yt, ytr = _years(cfg, "val"), _years(cfg, "test"), _years(cfg, "train")

    # 1. volatility
    vol_level = {"Very high": "very high", "High": "high", "Moderate": "moderate", "Low": "on the calm side"}[
        next(n for n, lo_, _ in TIERS if rp["vol_ratio"] >= lo_)]
    comp = "For comparison, broad US stock indices such as the S&P 500 have historically run at roughly 15-20% a year"
    if ref.get("vol_ann_median_train"):
        comp += f", and the typical (median) stock in this project's training data ran at about {ref['vol_ann_median_train']:.0%}"
    pct = f" It is {_pct_words(rp['vol_pct'])} in the validation years ({yv})." if rp["vol_pct"] is not None else ""
    r2t, r2h = _metric(cfg, ["test", "V", "R2"]), _metric(cfg, ["holdout", "V|all", "R2"])
    acc = (f"On the {yt} test years the volatility model explained {r2t:.0%} of the variation in (log) volatility across stocks "
           f"and dates (R² {r2t:.2f})" if r2t is not None else "The volatility model was scored on held-out years")
    acc += f"; on the sealed 2023-2026 holdout it explained {r2h:.0%} (R² {r2h:.2f})." if r2h is not None else "."
    acc += (" That is a solid score for this task: volatility is persistent (calm stocks tend to stay calm and wild ones wild), "
            "which makes it one of the more forecastable quantities in finance. It forecasts the size of the moves, not their direction.")
    s1 = dict(title=f"1. Expected volatility: {v:.0%} a year (likely {lo:.0%}-{hi:.0%})", items=[
        ("What it means", f"The model expects {tk}'s price to move at an annualised rate of about {v:.0%} over the next 20 trading "
                          f"days (about one month). In everyday terms that is a typical daily move of about ±{day:.1%} and a typical "
                          f"one-month move of about ±{month:.0%}. That is {vol_level} for a single US stock. {comp}.{pct}"),
        ("The range", f"{lo:.0%}-{hi:.0%} is an 80% prediction range. In the validation years ({yv}) the actual volatility landed "
                      f"inside a range this wide around the forecast about 8 times out of 10: the low end is a calm month, the high "
                      f"end a turbulent one. About 1 time in 10 it was calmer than {lo:.0%}, and about 1 time in 10 wilder than {hi:.0%}."),
        ("How accurate the model is", acc)])

    # 2. drawdown
    p, base = out["p_dd10"], cfg.get("base_rates", {}).get("D10")
    ratio = p / base if base else None
    rel = ("well above" if ratio >= 1.3 else "above" if ratio >= 1.1 else "close to" if ratio >= 0.9 else "below") if ratio else ""
    auc_d, ece_d = _metric(cfg, ["test", "D10", "AUC"]), _metric(cfg, ["test", "D10", "ECE"])
    ctx = (f"Across all stocks and dates in the training years ({ytr}) a drop like this happened {base:.0%} of the time, so this reading "
           f"is {ratio:.1f}x the typical rate ({rel} normal). " if base else "")
    ctx += (f"The NO TRADE rule treats {rule['p_dd10_q80']:.0%} or more as high risk (the top 20% of the model's readings in the "
            f"validation years).")
    cal = (f"The model's raw scores were calibrated on the validation years ({yv}) with isotonic regression, so a stated chance "
           f"matched how often the drop really happened in that period.")
    if auc_d is not None:
        cal += (f" On the {yt} test years it ranked a stock that went on to drop 10% above one that did not {auc_d:.0%} of the time "
                f"(AUC {auc_d:.2f}; 50% would be guessing), a useful but far from perfect separation.")
    if ece_d is not None:
        cal += f" Its stated chances were off by {100 * ece_d:.1f} percentage points on average in those years."
    s2 = dict(title=f"2. Chance of a 10%+ drawdown within 20 trading days: {p:.0%}", items=[
        ("What it means", f"The model gives a {p:.0%} chance ({_one_in(p)}) that at some point in the next 20 trading days {tk} "
                          f"trades at least 10% below the next session's opening price, even if it recovers afterwards."),
        ("The context", ctx), ("The calibration", cal)])

    # 3. direction
    q = out["p_out5"]
    auc_o, ci_o, hold_o = _metric(cfg, ["test", "O5", "AUC"]), _metric(cfg, ["test", "O5", "AUC_CI"]), _metric(cfg, ["holdout", "O5|all", "AUC"])
    gap = abs(q - 0.5)
    lean = ("exactly at" if gap < 0.005 else ("slightly " if gap < 0.03 else "" if gap < 0.1 else "well ") + ("above" if q > 0.5 else "below"))
    if not rule["direction_usable"]:
        take = "Treat this as a coin flip."
        if auc_o is not None:
            take += f" On the {yt} test years the direction model's AUC was {auc_o:.3f}"
            take += f" (95% range {ci_o[0]:.3f}-{ci_o[1]:.3f}, which includes 0.5)" if ci_o else ""
            take += f", and {hold_o:.3f} on the holdout" if hold_o is not None else ""
            take += ": no reliable edge."
        take += " That is why the dashboard never turns this number into a buy or sell call."
    else:
        take = (f"The direction model passed its validation and test checks, but the edge is small (test AUC {auc_o:.3f}); "
                "it is shown as a low-confidence signal only when risk is not elevated.")
    s3 = dict(title=f"3. Chance of beating the median stock over 5 trading days: {q:.0%}", items=[
        ("What it means", f"A {q:.0%} probability that {tk}'s return over the next 5 trading days beats the return of the median "
                          f"(middle) US stock over the same days. {q:.0%} is {lean} 50%."),
        ("The takeaway", take)])

    sections = [s1, s2, s3]
    if sim is not None:
        cov = (cfg.get("mc") or {}).get("coverage", {}).get("test", {})
        check = (f" On the {yt} test years the simulation's 90% band contained {cov['90%']:.0%} of the real 20-day outcomes."
                 if cov.get("90%") is not None else "")
        sections.append(dict(title=f"4. Monte Carlo simulation of the next 20 trading days ({sim['n_paths']:,} paths)", items=[
            ("How it works", "Each simulated path draws a volatility from the model's forecast and its measured error, then draws "
                             f"fat-tailed daily moves (Student-t, {sim['t_df']:g} degrees of freedom) with exactly that volatility. "
                             "There is no upward or downward drift, because the direction model has no reliable edge."
                             + (" The paths are reweighted so the share that touches a 10% drawdown equals the drawdown model's "
                                f"calibrated {p:.0%} (raw simulation: {sim['p_dd_sim']:.0%})." if sim["tilted"] else "") + check),
            ("What it shows", f"Starting from ${sim['p0']:,.2f}, the middle 90% of simulated 20-day outcomes range from "
                              f"{sim['ret_q05']:+.0%} to {sim['ret_q95']:+.0%} (${sim['price_q05']:,.2f} to ${sim['price_q95']:,.2f}). "
                              f"In the worst 5% of paths the average loss is {sim['es95']:.0%}. {sim['p_down10']:.0%} of paths end "
                              f"10% or more lower, and {sim['p_up']:.0%} end higher than they started."),
            ("How to read it", "This is a picture of the range of plausible outcomes given the predicted risk, not a forecast of "
                               "where the price will go.")]))

    tier_note = {"Very high": "Very high: far above the high-risk line", "High": "High: above the NO TRADE risk line",
                 "Moderate": "Moderate: close to, but below, the high-risk line", "Low": "Low: well below the high-risk line"}[rp["tier"]]
    rows = [
        ("Short-term direction (5 days)", f"{q:.0%} chance to beat the median stock",
         "Neutral: no reliable 5-day edge" if not rule["direction_usable"] else "Weak signal only"),
        ("Volatility (20 days)", f"{v:.0%} a year (±{month:.0%} a month)", f"{vol_level.capitalize()} swings expected"),
        ("Drawdown risk (20 days)", f"{p:.0%} chance of a 10%+ drop", f"{ratio:.1f}x the typical rate" if ratio else "-"),
        ("Overall risk tier", f"{rp['tier']} (risk score {rp['score']:.2f})", tier_note),
        ("Decision (pre-registered rule)", out["decision"], "; ".join(x.rstrip(".") for x in out["reasons"][:2]) + "."),
    ]
    if sim is not None:
        rows.insert(3, ("Simulated 20-day range", f"{sim['ret_q05']:+.0%} to {sim['ret_q95']:+.0%}",
                        f"worst-5% average {-sim['es95']:+.0%}"))

    if rp["tier"] in ("High", "Very high"):
        tk_ = (f"{tk} is in the {rp['tier'].lower()}-risk zone right now: expected swings of about ±{month:.0%} over the next month and a "
               f"{p:.0%} chance of a 10% drop" + (f" ({ratio:.1f}x the typical rate)" if ratio else "") + ", while the 5-day "
               "direction signal is no better than a coin flip. The risk side of this reading is the reliable part; the direction "
               "side is not.")
    elif rp["tier"] == "Moderate":
        tk_ = (f"{tk}'s risk is moderate: swings of about ±{month:.0%} a month and a {p:.0%} chance of a 10% drop put it close to, "
               "but below, the high-risk line. The 5-day direction signal offers no reliable edge.")
    else:
        tk_ = (f"{tk} is on the calmer side right now: swings of about ±{month:.0%} a month and a {p:.0%} chance of a 10% drop. "
               "Lower risk is not a buy signal, though: the 5-day direction signal still offers no reliable edge.")
    tk_ += f" Under the pre-registered rule the decision is {out['decision']}."
    return dict(sections=sections, summary=pd.DataFrame(rows, columns=["Metric", "Current signal", "What it implies"]),
                takeaway=tk_, risk=rp)


def report_markdown(res, cfg, sim=None):
    """One company's full report as markdown (used by the Gradio app and as a text fallback)."""
    if not res.get("ok"):
        return res.get("message") or "No result."
    ex = explain(res, cfg, sim)
    out = res["out"]
    md = [f"### {res['ticker']} — analysis as of {res['asof']} ({res['how']})", f"**Decision: {out['decision']}** · "
          f"**Risk tier: {ex['risk']['tier']}** (score {ex['risk']['score']:.2f})", ""]
    for s in ex["sections"]:
        md.append(f"#### {s['title']}")
        md += [f"- **{a}:** {b}" for a, b in s["items"]]
        md.append("")
    md += ["#### Summary", "| Metric | Current signal | What it implies |", "|---|---|---|"]
    md += [f"| {a} | {b} | {c} |" for a, b, c in ex["summary"].itertuples(index=False)]
    md += ["", f"**Primary takeaway:** {ex['takeaway']}", "", LIMITS]
    return "\n".join(md)


def analyse(B, img_w, img_d, img_h, ticker_text=""):
    """Single-company analysis, returns (markdown, PIL image of the three rebuilt charts or None, DataFrame of checks or None)."""
    res = analyse_company(B, {"1W": img_w, "1D": img_d, "1H": img_h}, ticker_text)
    if not res["ok"]:
        return res["message"], None, res["checks"]
    sim = run_mc(res, B.cfg)
    return report_markdown(res, B.cfg, sim), res["charts"], res["checks"]


# ------------------------------------------------------------------------------------------------ several companies
def compare(results, cfg, sims=None):
    """Ranking of the analysed companies from highest to lowest risk, and a one-paragraph verdict."""
    rows = []
    for i, r in enumerate(results):
        if not r.get("ok"):
            continue
        rp, out = risk_profile(r["out"], cfg), r["out"]
        row = {"Ticker": r["ticker"], "Risk tier": rp["tier"], "Risk score": round(rp["score"], 2),
               "Volatility (a year)": out["vol_ann"], "Chance of 10%+ drawdown": out["p_dd10"],
               "Chance to beat median (5 days)": out["p_out5"], "Decision": out["decision"], "As of": str(r["asof"])}
        if sims is not None and sims[i] is not None:
            row["Simulated 20-day range"] = f"{sims[i]['ret_q05']:+.0%} to {sims[i]['ret_q95']:+.0%}"
        rows.append(row)
    if not rows:
        return None, ""
    tab = pd.DataFrame(rows).sort_values("Risk score", ascending=False).reset_index(drop=True)
    tab.insert(0, "Rank (1 = riskiest)", range(1, len(tab) + 1))
    if len(tab) == 1:
        r = tab.iloc[0]
        return tab, f"**{r['Ticker']}** is in the **{r['Risk tier'].lower()}** risk tier (risk score {r['Risk score']:.2f})."
    hi_, lo_ = tab.iloc[0], tab.iloc[-1]
    txt = (f"**Highest risk: {hi_['Ticker']}** ({hi_['Risk tier']}, volatility {hi_['Volatility (a year)']:.0%} a year, "
           f"{hi_['Chance of 10%+ drawdown']:.0%} chance of a 10% drawdown). **Lowest risk: {lo_['Ticker']}** ({lo_['Risk tier']}, "
           f"volatility {lo_['Volatility (a year)']:.0%} a year, {lo_['Chance of 10%+ drawdown']:.0%} chance of a 10% drawdown).")
    flagged = tab[tab["Risk score"] >= 1.0]["Ticker"].tolist()
    txt += (f" Above the NO TRADE risk line: {', '.join(flagged)}." if flagged else " None is above the NO TRADE risk line.")
    txt += (" The direction signal has no reliable edge for any of them, so the ranking is about risk, not about which "
            "stock will do better." if not cfg["rule"]["direction_usable"] else "")
    return tab, txt
''')
print('wrote', 'dashboard_logic.py')

In [ ]:
# writes tfcv_report.py (used by this notebook and, later, by the dashboard)
open(os.path.join(WORK, 'tfcv_report.py'), 'w', encoding='utf-8').write(r'''"""TriFrame CV - the plain-language PDF project report (about 5-6 pages) and its figures.
Used by the notebook's Stage 11, from the values computed in the run (the same function made the 60-stock pilot's report).
Every number in the report comes from the dict R passed to build_pdf; nothing is typed in by hand."""
import os
import datetime as dt
from xml.sax.saxutils import escape

import numpy as np

BLUE, ORANGE, GREY, RED, GREEN = "#3b6ea5", "#e67e22", "#9e9e9e", "#c0392b", "#2e9d5b"


# ------------------------------------------------------------------------------------------------ figures (matplotlib, no pyplot state)
def _fig(w, h):
    from matplotlib.figure import Figure
    return Figure(figsize=(w, h), dpi=150)


def fig_pipeline(path, ft=True):
    """The architecture as a flow diagram."""
    from matplotlib.patches import FancyBboxPatch
    fig = _fig(11, 4.4); ax = fig.add_axes([0, 0, 1, 1]); ax.set_xlim(0, 11); ax.set_ylim(0, 4.4); ax.axis("off")

    def box(x, y, w, h, text, color, fs=8.2):
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.04,rounding_size=0.08", fc=color, ec="#33415c", lw=1))
        ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=fs)

    def arrow(x0, y0, x1, y1):
        ax.annotate("", xy=(x1, y1), xytext=(x0, y0), arrowprops=dict(arrowstyle="-|>", color="#33415c", lw=1.3))
    # top row, left to right: pictures -> renderer -> vision network -> embeddings -> head
    top, h = 3.05, 1.2
    box(0.1, top, 1.65, h, "3 chart pictures\nweekly 104 bars\ndaily 126 bars\nhourly 140 bars", "#dde8f5")
    box(2.05, top, 1.75, h, "Renderer\n128-px black and\nwhite images (same\nfor training and\nthe dashboard)", "#dde8f5")
    box(4.1, top, 2.05, h, "Pretrained vision network\n(one of 7, shared by\nthe 3 views)\n" + ("frozen OR fine-tuned" if ft else "frozen"), "#fde9d4")
    box(6.45, top, 1.75, h, "Embeddings\n(384-768 numbers\nper picture)\n-> PCA to 128 each", "#fde9d4")
    box(8.5, top, 2.4, h, "Head, tuned on validation:\nridge / logistic regression\nor boosted trees (XGBoost),\none per target", "#e3f1e6")
    for a, b in ((1.75, 2.05), (3.8, 4.1), (6.15, 6.45), (8.2, 8.5)):
        arrow(a, top + h / 2, b, top + h / 2)
    # the 33 numbers join at the head
    box(6.45, 1.75, 1.75, 0.95, "33 numbers from the\nlast 126 daily bars", "#dde8f5")
    arrow(8.2, 2.22, 9.0, top)
    # bottom row, right to left: calibration -> outputs -> NO TRADE rule -> Monte Carlo
    bot, hb = 0.15, 1.15
    box(8.5, bot, 2.4, hb, "Calibration (isotonic,\nvalidation years):\n'30%' means 30%", "#e3f1e6")
    box(5.3, bot, 2.9, hb, "Outputs: 20-day volatility,\nchance of a 10% drawdown,\nchance to beat the median", "#e3f1e6")
    box(2.65, bot, 2.35, hb, "NO TRADE rule\n(fixed before the\ntest years)", "#f6d5d5")
    box(0.1, bot, 2.25, hb, "Monte Carlo:\n20-day price paths\n+ risk tier", "#f6d5d5")
    arrow(9.7, top, 9.7, bot + hb)
    for a, b in ((8.5, 8.2), (5.3, 5.0), (2.65, 2.35)):
        arrow(a, bot + hb / 2, b, bot + hb / 2)
    fig.savefig(path, bbox_inches="tight"); return path


def fig_timeline(path, splits_years):
    """Train / validation / test / holdout on a time axis. splits_years: [(name, first_year, last_year + 1, colour)]."""
    fig = _fig(11, 1.5); ax = fig.add_axes([0.03, 0.32, 0.94, 0.55])
    for name, a, b, c in splits_years:
        ax.barh(0, b - a, left=a, color=c, edgecolor="white", height=0.8)
        ax.text((a + b) / 2, 0, f"{name}\n{a}-{b - 1}", ha="center", va="center", fontsize=9, color="white", fontweight="bold")
    ax.set_yticks([]); ax.set_xlim(splits_years[0][1], splits_years[-1][2]); ax.spines[["left", "right", "top"]].set_visible(False)
    ax.set_xticks(range(splits_years[0][1], splits_years[-1][2] + 1, 2)); ax.tick_params(labelsize=8)
    fig.savefig(path, bbox_inches="tight"); return path


def fig_example(path, seed=3):
    """What the model sees: the renderer applied to SYNTHETIC prices (a random walk), so no licensed data appears in the report."""
    import pandas as pd
    import tfcv_core as C
    rng = np.random.default_rng(seed)
    days = pd.bdate_range("2021-01-04", periods=700)
    r = rng.standard_t(4, len(days)) * 0.012 + 0.0003
    close = 100 * np.exp(np.cumsum(r)); op = close * np.exp(rng.normal(0, 0.004, len(days)))
    hi = np.maximum(op, close) * np.exp(np.abs(rng.normal(0, 0.006, len(days)))); lo = np.minimum(op, close) * np.exp(-np.abs(rng.normal(0, 0.006, len(days))))
    daily = pd.DataFrame({"date": days, "open": op, "high": hi, "low": lo, "close": close, "volume": rng.integers(1e5, 1e6, len(days)).astype(float)})
    rows = []
    for d, o, c in zip(days[-40:], op[-40:], close[-40:]):
        path_ = np.linspace(o, c, 8) * np.exp(np.cumsum(rng.normal(0, 0.003, 8)))
        for j in range(7):
            ts = pd.Timestamp(d).tz_localize("America/New_York") + pd.Timedelta(hours=9 + j, minutes=30)
            a, b = path_[j], path_[j + 1]
            rows.append(dict(timestamp=ts.tz_convert("UTC"), open=a, high=max(a, b) * 1.002, low=min(a, b) * 0.998, close=b, volume=1e4))
    hourly = pd.DataFrame(rows)
    trip = C.render_at(daily, hourly, len(daily) - 1)
    fig = _fig(11, 1.9)
    for i, (v, lab) in enumerate((("W", "weekly: 104 bars"), ("D", "daily: 126 bars"), ("H", "hourly: 140 bars"))):
        ax = fig.add_axes([0.005 + i * 0.333, 0.02, 0.32, 0.8]); ax.imshow(trip[v], cmap="gray", aspect="auto", interpolation="nearest")
        ax.set_xticks([]); ax.set_yticks([]); ax.set_title(lab, fontsize=9)
    fig.savefig(path, bbox_inches="tight", facecolor="white"); return path


def fig_bars(path, names, values, ci=None, ref=None, ref_label="numbers only", title="", colors=None, xlabel=""):
    """Horizontal bars with optional CIs and a reference line."""
    fig = _fig(9, 0.32 * len(names) + 1.2); ax = fig.add_axes([0.32, 0.18, 0.64, 0.7])
    yy = np.arange(len(names)); v = np.asarray(values, float)
    err = None if ci is None else np.vstack([v - np.asarray(ci)[:, 0], np.asarray(ci)[:, 1] - v])
    ax.barh(yy, v, color=colors or BLUE, xerr=err, capsize=2)
    if ref is not None:
        ax.axvline(ref, color=RED, ls="--", lw=1.2, label=f"{ref_label} ({ref:.3f})"); ax.legend(fontsize=7, loc="lower right")
    ax.set_yticks(yy); ax.set_yticklabels(names, fontsize=8); ax.set_title(title, fontsize=9); ax.set_xlabel(xlabel, fontsize=8)
    ax.tick_params(axis="x", labelsize=8)
    fig.savefig(path, bbox_inches="tight"); return path


# ------------------------------------------------------------------------------------------------ the PDF
def _fonts():
    """A Unicode font (DejaVu ships with matplotlib) so ², ±, ≥ and – print correctly."""
    from reportlab.pdfbase import pdfmetrics
    from reportlab.pdfbase.ttfonts import TTFont
    import matplotlib
    d = os.path.join(os.path.dirname(matplotlib.__file__), "mpl-data", "fonts", "ttf")
    try:
        pdfmetrics.registerFont(TTFont("DejaVu", os.path.join(d, "DejaVuSans.ttf")))
        pdfmetrics.registerFont(TTFont("DejaVu-Bold", os.path.join(d, "DejaVuSans-Bold.ttf")))
        pdfmetrics.registerFont(TTFont("DejaVu-Oblique", os.path.join(d, "DejaVuSans-Oblique.ttf")))
        from reportlab.lib.fonts import addMapping
        addMapping("DejaVu", 0, 0, "DejaVu"); addMapping("DejaVu", 1, 0, "DejaVu-Bold"); addMapping("DejaVu", 0, 1, "DejaVu-Oblique")
        addMapping("DejaVu", 1, 1, "DejaVu-Bold")
        return "DejaVu", "DejaVu-Bold"
    except Exception:
        return "Helvetica", "Helvetica-Bold"


def _f(x, d=3):
    return "n/a" if x is None or (isinstance(x, float) and not np.isfinite(x)) else f"{x:.{d}f}"


def _pct(x, d=0):
    return "n/a" if x is None or (isinstance(x, float) and not np.isfinite(x)) else f"{100 * x:.{d}f}%"


def build_pdf(R, out_path):
    """R: dict with the run's values (see make_pilot_report.py and the notebook's Stage 11 for how it is filled).
    Required keys: title_run, version, n_stocks, universe, n_rows, splits, cards, n_configs, chosen, metrics, hypotheses,
    no_trade, dashboard, figures. Optional: finetune, mc, vs_pilot, controls, holdout_note."""
    from reportlab.lib import colors
    from reportlab.lib.enums import TA_LEFT
    from reportlab.lib.pagesizes import letter
    from reportlab.lib.styles import ParagraphStyle
    from reportlab.lib.units import inch
    from reportlab.platypus import Image, KeepTogether, PageBreak, Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle
    from PIL import Image as PILImage

    FONT, BOLD = _fonts()
    st = dict(
        title=ParagraphStyle("t", fontName=BOLD, fontSize=19, leading=23, textColor=colors.HexColor("#1d2533"), spaceAfter=4),
        sub=ParagraphStyle("s", fontName=FONT, fontSize=10, leading=13, textColor=colors.HexColor("#555555"), spaceAfter=10),
        h1=ParagraphStyle("h1", fontName=BOLD, fontSize=13.5, leading=17, textColor=colors.HexColor("#1f4e79"), spaceBefore=10, spaceAfter=5,
                          keepWithNext=1),
        body=ParagraphStyle("b", fontName=FONT, fontSize=9.6, leading=13.4, alignment=TA_LEFT, spaceAfter=5),
        bullet=ParagraphStyle("bl", fontName=FONT, fontSize=9.6, leading=13.4, leftIndent=12, bulletIndent=2, spaceAfter=2.5),
        cell=ParagraphStyle("c", fontName=FONT, fontSize=8.2, leading=10.4),
        cellb=ParagraphStyle("cb", fontName=BOLD, fontSize=8.2, leading=10.4, textColor=colors.white),
        box=ParagraphStyle("x", fontName=FONT, fontSize=9.8, leading=13.6, backColor=colors.HexColor("#eef4fb"), borderPadding=7,
                           borderColor=colors.HexColor("#b9cde6"), borderWidth=0.6, spaceBefore=4, spaceAfter=10),
        small=ParagraphStyle("sm", fontName=FONT, fontSize=7.8, leading=10, textColor=colors.HexColor("#555555")))
    E = lambda s: escape(str(s))
    P = lambda s, k="body": Paragraph(s, st[k])
    story = []

    def bullets(items):
        for it in items:
            story.append(Paragraph(it, st["bullet"], bulletText="•"))

    def table(rows, widths, header=True):
        data = [[Paragraph(str(c), st["cellb" if (header and i == 0) else "cell"]) for c in r] for i, r in enumerate(rows)]
        t = Table(data, colWidths=[w * inch for w in widths], repeatRows=1 if header else 0)
        t.setStyle(TableStyle([("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#1f4e79") if header else colors.white),
                               ("GRID", (0, 0), (-1, -1), 0.4, colors.HexColor("#c8d3e0")),
                               ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#f4f7fb")]),
                               ("VALIGN", (0, 0), (-1, -1), "TOP"), ("TOPPADDING", (0, 0), (-1, -1), 2.5), ("BOTTOMPADDING", (0, 0), (-1, -1), 2.5)]))
        story.append(t); story.append(Spacer(1, 6))

    def image(key, width_in, caption=None, head=None):
        p = R.get("figures", {}).get(key)
        if not p or not os.path.exists(p):
            if head:
                story.append(P(head, "h1"))
            return
        w, h = PILImage.open(p).size
        items = ([P(head, "h1")] if head else []) + [Image(p, width=width_in * inch, height=width_in * inch * h / w)]
        if caption:
            items.append(Paragraph(caption, st["small"]))
        story.append(KeepTogether(items)); story.append(Spacer(1, 6))

    M, NT, H = R["metrics"], R["no_trade"], R["hypotheses"]
    sp = R["splits"]
    v3 = R.get("version") == "v3"

    # ---------------------------------------------------------------- page 1: summary
    story.append(P("TriFrame: can pictures of stock charts help predict risk?", "title"))
    story.append(P(f"Computer Vision with Deep Learning project · end-to-end report · {E(R['title_run'])} · generated "
                   f"{dt.date.today().isoformat()}", "sub"))
    images_add = "did not add a significant improvement" if not H["H1"][0] else "added a small but significant improvement"
    direction = ("there is no reliable 5-day direction signal" if not M.get("direction_usable") else "the 5-day direction signal is weak")
    story.append(P(
        f"<b>In one paragraph.</b> We asked whether pictures of a stock's price chart at three time scales (weekly, daily and hourly) "
        f"help a deep-learning model predict three things about the coming weeks: how much the price will swing (<b>volatility</b>), "
        f"whether it will fall 10% or more at some point (<b>drawdown</b>), and whether it will beat the typical stock over the next "
        f"5 days (<b>direction</b>). We tested {len([c for c in R['cards'] if c.get('family') != 'Control'])} pretrained vision models"
        f"{' (each both frozen and fine-tuned)' if R.get('finetune') else ''} on {R['n_stocks']} US stocks ({R['n_rows']:,} chart sets, "
        f"2001-2022), always against a simple baseline that uses only 33 numbers computed from the same prices. <b>The answer:</b> risk "
        f"is predictable (volatility R² {_f(M['test_V_R2'], 2)} on the unseen 2020-2022 test years), but the chart pictures {images_add} "
        f"over the numbers there, and {direction}, so the system's default decision is <b>NO TRADE</b>.", "box"))
    rows = [["Target", "Model chosen", "Test score (2020-2022)", "Sealed holdout (2023-2026)", "In plain words"]]
    for c in R["chosen"]:
        rows.append([c["target"], c["model"], c["test"], c.get("holdout", "not opened"), c["plain"]])
    table(rows, [1.45, 1.75, 1.15, 1.05, 2.1])
    if v3:
        story.append(P("<b>What is new in this version (v3)</b>"))
        bullets([f"Trained on <b>{R['n_stocks']} stocks</b> instead of 60 (the new sample contains the pilot's 60, so the two runs compare like for like).",
                 "Every pretrained model is <b>fine-tuned</b> on the charts as well as used frozen, and both compete under the same rule.",
                 "A <b>leaderboard</b> scores every model on six metrics per target, with confidence intervals and charts.",
                 "A <b>Monte Carlo simulation</b> of the next 20 trading days, calibrated on validation data and checked on the test years.",
                 "The dashboard analyses <b>up to four companies at once</b>, ranks them by risk and explains every number in plain words."])

    # ---------------------------------------------------------------- steps 1-2: data and splits
    story.append(P("Steps 1-2: the data and how it was split", "h1"))
    story.append(P(
        "Each example, a <b>chart set</b>, is one stock on one day: three black-and-white chart pictures drawn from its price history "
        "(104 weekly bars, 126 daily bars and 140 hourly bars) plus 33 ordinary numbers (returns, price ranges, volatility measures) "
        "computed from the last 126 daily bars. The labels are what happened next: the realised volatility over the next 20 trading "
        "days, whether the price traded 10% below the next day's open within 20 days, and whether the stock beat the median stock over "
        f"5 days. This run uses <b>{R['n_stocks']} of {R['universe']} US stocks</b>"
        + (" (chosen at random, and including the 60 stocks of the earlier pilot)" if R.get("pilot_included") else " (chosen at random)")
        + f", {R['n_rows']:,} chart sets in total."))
    story.append(P(
        "Time is never shuffled. Models learn from the <b>training</b> years, are tuned and selected on the <b>validation</b> years, and "
        "are scored once on the <b>test</b> years. 2023 onward is a sealed <b>holdout</b> that can be opened only once, by the owner, "
        "after everything is frozen. A 20-day gap at each boundary stops any label from leaking into the next period."))
    image("timeline", 6.9)
    table([["Period", "Years", "Chart sets", "Stocks", "Used for"]] + [[k, v[0], f"{v[1]:,}" if v[1] else "-", v[2] or "-", v[3]] for k, v in sp.items()],
          [1.0, 1.0, 1.0, 0.8, 3.7])
    image("example", 6.9, "What a model sees: the three pictures of one chart set (drawn here from synthetic prices). Each picture is "
                          "scaled to fill its panel, so it shows the shape of the moves, not their size in dollars.")

    # ---------------------------------------------------------------- step 3: pretrained models
    story.append(P("Step 3: the pretrained vision models", "h1"))
    story.append(P(
        "A <b>pretrained</b> model is a neural network that has already learned to recognise edges, shapes and textures from millions of "
        "ordinary photos. We reuse that knowledge instead of training a network from scratch. We compared seven well-known designs: "
        "convolutional networks (CNNs), which scan the picture with small filters, and transformers, which split it into patches that "
        "'pay attention' to each other. Three ResNet-18s with <b>random, untrained weights</b> serve as a control: if they did as well, "
        "pretraining would not be doing anything."))
    cols = ["Model", "Type", "Pretrained on", "Parameters (millions)"] + (["Fine-tuning"] if R.get("finetune") else [])
    rows = [cols]
    for c in R["cards"]:
        rows.append([c["model"], c["family"], c["pretrained_on"], _f(c.get("params_m"), 1)] + ([c.get("finetuning", "")] if R.get("finetune") else []))
    table(rows, [1.7, 0.9, 2.3, 1.1] + ([1.4] if R.get("finetune") else []))
    story.append(P(
        "<b>Frozen or fine-tuned?</b> 'Frozen' means the network's own weights are left exactly as they came from pretraining; we only "
        "read its output (an <b>embedding</b>, a list of a few hundred numbers summarising the picture) and train small models on top. "
        "'Fine-tuned' means the network's weights are also adjusted on our charts. "
        + (R["finetune"]["intro"] if R.get("finetune") else
           "<b>This run used frozen networks only; no fine-tuning was done.</b> Version 3 of the notebook adds fine-tuning (Stage 5F).")))

    # ---------------------------------------------------------------- step 4: architecture
    image("pipeline", 7.0, head="Step 4: the architecture")
    bullets(["The renderer draws the three charts as 128-pixel-high black-and-white pictures; the dashboard uses the very same code.",
             "One vision network turns each picture into an embedding; <b>PCA</b> compresses each embedding to 128 numbers "
             "(fitted on training years only).",
             "The three compressed embeddings and the 33 numbers go into a <b>head</b>: ridge or logistic regression, or "
             "gradient-boosted decision trees (XGBoost). One head per target.",
             "Probabilities are <b>calibrated</b> on the validation years so that '30%' really means 30%.",
             "A <b>NO TRADE rule</b>, fixed before the test years were opened, turns the outputs into a decision, and a "
             "<b>Monte Carlo simulation</b> turns the volatility forecast into a fan of possible price paths."])

    # ---------------------------------------------------------------- step 5: training and tuning
    story.append(P("Step 5: training and hyper-parameter tuning", "h1"))
    story.append(P(
        f"We tried <b>{R['n_configs']:,} model configurations</b> and logged every one: ridge strengths, logistic regularisation, tree "
        "depth, learning rate and the number of trees (by early stopping), for each target, each vision network and each combination of "
        "inputs (images only, numbers + all three timeframes, numbers + one timeframe). Every choice was made on the validation years "
        "only, and all preprocessing (gap filling, scaling, PCA) was fitted on the training years only. <b>The selection rule was written "
        "down before the test years were opened:</b> per target the best validation score wins, and a model that uses chart pictures "
        "wins only if its improvement over the numbers-only model is clearly above zero (its 95% confidence interval excludes 0)."))

    # ---------------------------------------------------------------- step 6: results
    story.append(P("Step 6: results - which models won, and how good they are", "h1"))
    rows = [["Target", "Inputs", "Vision network", "Head", "Validation", "Test (95% CI)"]]
    for c in R["chosen"]:
        rows.append([c["target"], c["inputs"], c["network"], c["head"], c["val"], c["test_ci"]])
    table(rows, [1.2, 1.3, 1.45, 1.2, 0.9, 1.25])
    image("leaderboard", 7.0, R.get("leaderboard_caption"))
    hyp = [f"<b>Do chart pictures add to the numbers? {'Yes' if H['H1'][0] else 'No'}.</b> {H['H1'][1]}",
           f"<b>Do the vision models differ from each other? {'Yes, a little' if H['H2'][0] else 'No'}.</b> {H['H2'][1]}",
           f"<b>Does waiting for confident direction calls help? {'Yes' if H['H3'][0] else 'No'}.</b> {H['H3'][1]}"]
    if R.get("controls"):
        hyp.append(f"<b>Sanity checks.</b> {R['controls']}")
    bullets(hyp)

    # ---------------------------------------------------------------- step 7: fine-tuning
    story.append(P("Step 7: did we fine-tune, and did it help?", "h1"))
    if R.get("finetune"):
        F = R["finetune"]
        story.append(P(F["verdict"]))
        if F.get("rows"):
            table([["Model", "Frozen R²", "Fine-tuned R²", "Change (95% CI)", "Training time"]] + F["rows"], [2.0, 1.0, 1.1, 1.7, 1.2])
        image("finetune", 7.0)
    else:
        story.append(P(
            "<b>No.</b> In this run every vision network was <b>frozen</b>: the pretrained weights were used as they are, and only the small "
            "heads on top were trained. Fine-tuning a network means continuing its training on our own charts so its features adapt to "
            "price patterns. Version 3 of the notebook does exactly that for all seven pretrained networks (training years only, early "
            "stopping on validation, no image augmentation because flipping or cropping a chart changes its meaning) and lets the "
            "fine-tuned networks compete with the frozen ones under the same selection rule, so the question 'is fine-tuning worth it?' "
            "gets a measured answer, with confidence intervals and the extra training time."))

    # ---------------------------------------------------------------- step 8: decision + Monte Carlo
    story.append(P("Step 8: from predictions to a decision, and the Monte Carlo simulation", "h1"))
    story.append(P(
        f"<b>NO TRADE rule.</b> A stock is flagged as high risk when its predicted volatility is at least {_pct(NT['vol_q80'])} a year "
        f"or its drawdown chance at least {_pct(NT['pdd_q80'])} (the top 20% of the model's readings in the validation years). "
        f"On the test years the flag picked out {_pct(NT['share_flagged'])} of cases, and those cases really were riskier: realised "
        f"volatility {_pct(NT['realised_vol_flagged'])} vs {_pct(NT['realised_vol_other'])} a year, and 10% drawdowns "
        f"{_pct(NT['dd10_rate_flagged'])} vs {_pct(NT['dd10_rate_other'])} of the time. A direction call would only ever be shown if "
        f"the direction model had passed its validation <i>and</i> test checks; "
        f"{'it did not, so every decision is NO TRADE' if not M.get('direction_usable') else 'it passed, but the edge is small'}."))
    if R.get("mc"):
        mc = R["mc"]
        story.append(P(
            f"<b>Monte Carlo.</b> For each stock the dashboard simulates thousands of 20-day price paths. Each path draws a volatility "
            f"from the model's forecast and its measured error on validation data, then fat-tailed daily moves (Student-t with "
            f"{mc['t_df']:g} degrees of freedom, chosen on validation) with exactly that volatility; daily lows reach below the closes by "
            f"a range also fitted on validation. There is no drift, because the direction model has no edge. "
            + (f"By default the paths are reweighted, as little as possible and without changing the average return, so the share that "
               f"touches a 10% drawdown equals the calibrated drawdown model's chance. " if mc.get("tilt_default") else "")
            + f"<b>Check on the test years:</b> the simulation's 90% band contained {_pct(mc['test_cov90'])} of the real 20-day outcomes "
              f"(validation: {_pct(mc['val_cov90'])}) and its 50% band {_pct(mc['test_cov50'])}."))
        image("mc", 7.0)
    else:
        story.append(P("<b>Monte Carlo.</b> Added in version 3: a simulation of 20-day price paths driven by the volatility forecast "
                       "and its measured error, calibrated on validation data and checked on the test years."))

    # ---------------------------------------------------------------- step 9: dashboard
    story.append(P("Step 9: the dashboard - which models it uses", "h1"))
    D = R["dashboard"]
    multi = v3 or D.get("multi")
    story.append(P(
        "The <i>Computer Vision with Deep Learning Project Dashboard</i> runs the frozen models from the private model repository. "
        + ("You choose 1 to 4 companies and upload three TradingView screenshots (1W, 1D, 1H) for each. " if multi else
           "You upload three TradingView screenshots of one stock (1W, 1D, 1H). ")
        + "The app reads the ticker and the last bar of every screenshot, confirms them against market data, rebuilds exactly the "
          "inputs the models were trained on, and shows the outputs with charts and a plain-language explanation"
        + (", a risk tier, a Monte Carlo fan of price paths and, for several companies, a ranking from riskiest to safest." if multi else ".")
        + (f" Live at {E(D['url'])}." if D.get("url") else "")))
    rows = [["Output", "Inputs", "Vision network", "Head", "Test score"]] + [[r["Output"], r["Inputs"], r["Vision network"], r["Head"], r["Test score"]] for r in D["models"]]
    table(rows, [1.9, 1.5, 1.9, 1.3, 0.75])

    if R.get("vs_pilot"):
        n0 = len(story)
        story.append(P(f"Does more data help? {R['n_stocks']} stocks vs the 60-stock pilot", "h1"))
        story.append(P(R["vs_pilot"]["text"]))
        table([["Target", "Metric", "Pilot model (60 stocks)", "This model, same 60 stocks", "This model, all stocks"]] + R["vs_pilot"]["rows"],
              [1.6, 0.8, 1.5, 1.7, 1.5])
        story[n0:] = [KeepTogether(story[n0:])]

    # ---------------------------------------------------------------- limits + glossary
    story.append(P("Limits", "h1"))
    bullets([E(x) for x in R.get("limits", [])])
    story.append(P("Glossary", "h1"))
    table([["Term", "Meaning"],
           ["R²", "Share of the ups and downs in the target that the model explains. 1 = perfect, 0 = no better than always guessing the average."],
           ["AUC", "Chance that the model ranks a case where the event happened above one where it did not. 0.5 = coin flip, 1 = perfect."],
           ["Brier skill score", "How much better the probabilities are than always predicting the average rate (0 = no better)."],
           ["Calibration", "Whether stated chances come true as often as stated (30% events happen about 30% of the time)."],
           ["Embedding / PCA", "The numbers a vision network outputs for a picture / a way to compress them to the most informative 128."],
           ["Month-block bootstrap", "Confidence intervals made by resampling whole calendar months, because stocks move together within a month."],
           ["Holdout", "The most recent data (2023 onward), sealed until all choices are frozen and then opened once."]],
          [1.5, 5.6])

    def footer(canvas, doc):
        canvas.saveState(); canvas.setFont(FONT, 7.5); canvas.setFillColor(colors.HexColor("#777777"))
        canvas.drawString(0.6 * inch, 0.45 * inch, f"TriFrame CV · {R['title_run']} · research output, not investment advice")
        canvas.drawRightString(7.9 * inch, 0.45 * inch, f"page {doc.page}"); canvas.restoreState()
    doc = SimpleDocTemplate(out_path, pagesize=letter, leftMargin=0.6 * inch, rightMargin=0.6 * inch, topMargin=0.55 * inch,
                            bottomMargin=0.7 * inch, title="TriFrame CV project report", author="Krishna Harsha Tangirala")
    doc.build(story, onFirstPage=footer, onLaterPages=footer)
    return out_path
''')
print('wrote', 'tfcv_report.py')

In [ ]:
# writes app.py (used by this notebook and, later, by the dashboard)
open(os.path.join(WORK, 'app.py'), 'w', encoding='utf-8').write(r'''"""Gradio version of the dashboard (kept for Hugging Face Spaces; the Streamlit app streamlit_app.py is the primary front end).
Both use dashboard_logic.py, so they behave identically."""
import os

import gradio as gr

import dashboard_logic as DL

B = DL.load_bundle()                       # BUNDLE_DIR, or MODEL_REPO + HF_TOKEN from the environment / Space secrets


def analyse(img_w, img_d, img_h, ticker_text):
    return DL.analyse(B, img_w, img_d, img_h, ticker_text)


with gr.Blocks(title=DL.TITLE) as demo:
    gr.Markdown(f"# {DL.TITLE}")
    gr.Markdown(DL.describe(B))
    with gr.Row():
        w = gr.Image(type="pil", label="1 Week chart screenshot (1W)")
        dd = gr.Image(type="pil", label="1 Day chart screenshot (1D)")
        hh = gr.Image(type="pil", label="1 Hour chart screenshot (1H)")
    with gr.Row():
        tk = gr.Textbox(label="Ticker (optional; detected from the screenshots if left empty)", scale=3)
        btn = gr.Button("Analyse", variant="primary", scale=1)
    out_md = gr.Markdown()
    out_img = gr.Image(label="What the model sees: the three charts rebuilt from market data", type="pil")
    out_tab = gr.Dataframe(label="Screenshot checks")
    btn.click(analyse, inputs=[w, dd, hh, tk], outputs=[out_md, out_img, out_tab])

if __name__ == "__main__":
    demo.launch()
''')
print('wrote', 'app.py')

In [ ]:
# ==================================== shared helpers (imports, metrics per target, fitting with tuning) ====================================
import importlib
import tfcv_core as C, tfcv_train as T
importlib.reload(C); importlib.reload(T)
from triframe_loader import ImageStore

RES, FIG = os.path.join(WORK, "results"), os.path.join(WORK, "figures")
EMB_DIR, EMB_FT_DIR = os.path.join(WORK, f"emb_{RUN_TAG}"), os.path.join(WORK, f"emb_ft_{RUN_TAG}")
PCA_DIR = os.path.join(WORK, f"pca_{RUN_TAG}")


def emb_dir(b):
    """Embedding chunks of a backbone: frozen ones in EMB_DIR, fine-tuned ones ('ft_<backbone>') in EMB_FT_DIR."""
    return EMB_FT_DIR if b.startswith(T.FT_PREFIX) else EMB_DIR
TARGETS = {"V": "reg", "O5": "clf", "D10": "clf"}
TARGET_NAME = {"V": "20-day volatility (logrv20)", "O5": "beats the median stock over 5 days (y_out5)",
               "D10": "drawdown of 10 % or more within 20 days (dd20_le_10)"}
PRIMARY = {"V": ("R2", +1), "O5": ("logloss", -1), "D10": ("brier", -1)}    # +1: higher is better, -1: lower is better
LEDGER, VAL_PRED, FITS = [], {}, {}


def ledger(**kw):
    LEDGER.append(dict(utc=dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"), run=RUN_TAG, **kw))


def eval_target(t, y, p, dates, s2=None, xret=None, base=None):
    """All metrics for one target on one evaluation set (no confidence intervals)."""
    if TARGETS[t] == "reg":
        c, _ = T.month_codes(dates)
        s2 = float(np.var(y - p)) if s2 is None else s2
        return dict(n=len(y), R2=T.r2(y, p), wmR2=T.wm_r2(y, p, c), QLIKE=T.qlike(y, p, s2), MAE=float(np.mean(np.abs(y - p))))
    out = dict(n=len(y), AUC=T.auc(y, p), logloss=T.logloss(y, p), brier=T.brier(y, p), ECE=T.ece(y, p))
    if base is not None:
        out["BSS"] = 1 - out["brier"] / T.brier(y, np.full(len(y), base))
    if xret is not None:
        out["rankIC"], out["rankIC_t"] = T.rank_ic(dates, p, xret)
    return out


def fit_one(t, fs, X, heads=("linear", "tree"), stage="?"):
    """Fit on TRAIN, tune on VAL (ridge alpha / logistic C / tree grid with early stopping), keep the best per head."""
    m_tr, m_va = TR & OK[t], VA & OK[t]
    ytr, yva = Y[t][m_tr], Y[t][m_va]
    reg = TARGETS[t] == "reg"
    score = (lambda y, p: T.r2(y, p)) if reg else (lambda y, p: -T.logloss(y, p))     # higher is better
    if "linear" in heads:
        path = T.ridge_path(X[m_tr], ytr, X[m_va], ALPHAS) if reg else T.logit_path(X[m_tr], ytr, X[m_va], CS, device=DEVICE)
        sc = {k: score(yva, v[2]) for k, v in path.items()}
        for k, s in sc.items():
            ledger(stage=stage, target=t, feature_set=fs, head="linear", param=float(k), val_score=s)
        kbest = max(sc, key=sc.get)
        w, b, p = path[kbest]
        grid = ALPHAS if reg else CS
        FITS[(t, fs, "linear")] = dict(param=float(kbest), coef=w, intercept=float(b),
                                       at_edge=bool(kbest in (grid[0], grid[-1])), n_features=X.shape[1])
        VAL_PRED[(t, fs, "linear")] = p
    if "tree" in heads:
        best = None
        for cfg in T.TREE_GRID:
            m, n, p = T.tree_fit(X[m_tr], ytr, X[m_va], yva, TARGETS[t], cfg, gpu=(DEVICE == "cuda"), seed=SEED)
            s = score(yva, p)
            ledger(stage=stage, target=t, feature_set=fs, head="tree", param=json.dumps({**cfg, "n_estimators": n}), val_score=s)
            if best is None or s > best[0]:
                best = (s, cfg, n, m, p)
        FITS[(t, fs, "tree")] = dict(cfg=best[1], n_estimators=best[2], model=best[3], n_features=X.shape[1])
        VAL_PRED[(t, fs, "tree")] = best[4]


def val_table(t, keys=None):
    m = VA & OK[t]
    rows = []
    for (tt, fs, hd), p in VAL_PRED.items():
        if tt != t or (keys is not None and (tt, fs, hd) not in keys):
            continue
        rows.append(dict(target=t, feature_set=fs, head=hd,
                         **eval_target(t, Y[t][m], p, DATES[m], xret=XRET[m] if t == "O5" else None, base=BASE.get(t))))
    return pd.DataFrame(rows)


_IMG_CACHE = {}


def img_matrix(b, views=("W", "D", "H")):
    """PCA'd image features of one backbone (views concatenated), standardised on TRAIN rows and clipped at +-10."""
    key = (b, tuple(views))
    if key not in _IMG_CACHE:
        if len(_IMG_CACHE) > 6:
            _IMG_CACHE.clear(); gc.collect()
        Z = np.hstack([np.load(os.path.join(PCA_DIR, f"{b}__{v}.npy")) for v in views]).astype(np.float32)
        mu, sd = T.standardize(Z[TR])
        _IMG_CACHE[key] = (np.clip((Z - mu) / sd, -10, 10).astype(np.float32), mu, sd)
    return _IMG_CACHE[key]


def design(fs):
    """Design matrix for a feature-set name: NUM | IMG:<backbone> | NUM+IMG:<backbone> | NUM+IMG:<backbone>:<view>."""
    if fs == "NUM":
        return XN
    parts = fs.split(":")
    if parts[0] == "IMG":
        return img_matrix(parts[1])[0]
    views = (parts[2],) if len(parts) > 2 else ("W", "D", "H")
    return np.hstack([XN, img_matrix(parts[1], views)[0]])


def save_df(df_, name, push_it=True):
    path = os.path.join(RES, name)
    df_.to_csv(path, index=False)
    if push_it:
        push(path, f"results_{RUN_TAG}/{name}")
    return path


print("Modules loaded. Targets:", ", ".join(f"{k} = {v}" for k, v in TARGET_NAME.items()))

## Stage 1 - load the data
Sealed development rows (2001-10-22 to 2022-12-01) with features and labels. In QUICK mode a fixed random sample of stocks is used: the pilot's 60 plus 60 more.

In [ ]:
# ==================================== STAGE 1: load the data (979 stocks, three timeframes) ====================================
t0 = time.time()
st = ImageStore(who=f"tfcv_e2e_{RUN_TAG}", root=DST + "/")            # sealed: rows dated 2023-01-03 onward are invisible here
idx = st.index()
df = st.join_features(idx).merge(st.derived_labels()[["sample_id", "xret5_med", "dd20_pct", "dd20_le_10"]],
                                  on="sample_id", how="left", validate="one_to_one")
all_tickers = sorted(df["ticker"].unique())
PILOT_SET = set()
if QUICK:
    # the pilot's exact 60 stocks (same draw as the pilot run), plus QUICK_TICKERS - 60 more drawn from the rest
    n_pilot = min(PILOT_TICKERS, QUICK_TICKERS)
    pilot = np.random.default_rng(SEED).choice(all_tickers, n_pilot, replace=False)
    PILOT_SET = set(pilot) if n_pilot == PILOT_TICKERS else set()
    rest = sorted(set(all_tickers) - set(pilot))
    extra = np.random.default_rng(SEED + 1).choice(rest, QUICK_TICKERS - n_pilot, replace=False) if QUICK_TICKERS > n_pilot else []
    keep = sorted(set(pilot) | set(extra))
    df = df[df["ticker"].isin(keep)]
df = df.sort_values(["ticker", "date"]).reset_index(drop=True)

raw_span = "n/a"
if os.path.isdir(f"{DST}/final/1day"):
    _d = pd.read_parquet(f"{DST}/final/1day/{all_tickers[0]}.parquet", columns=["date"])
    raw_span = f"{_d['date'].min().date()} to {_d['date'].max().date()} ({(_d['date'].max() - _d['date'].min()).days / 365.25:.1f} years)"
yr = pd.to_datetime(df["date"]).dt.year
s1 = pd.DataFrame({"rows": df.groupby(yr).size(), "tickers": df.groupby(yr)["ticker"].nunique(),
                   "full_1W_window": df.groupby(yr)["n_valid_W"].apply(lambda s: float((s == 104).mean())),
                   "full_1D_window": df.groupby(yr)["n_valid_D"].apply(lambda s: float((s == 126).mean())),
                   "full_1H_window": df.groupby(yr)["n_valid_H"].apply(lambda s: float((s == 140).mean()))}).reset_index(names="year")
print(f"Universe: {len(all_tickers)} US stocks in the data ({df['ticker'].nunique()} used in this run"
      + (f", including all {len(PILOT_SET)} pilot stocks" if PILOT_SET else "") + ").")
print(f"Raw daily/hourly bars (first ticker): {raw_span}.")
print(f"Chart sets loaded: {len(df):,} rows, decision dates {pd.to_datetime(df['date']).min().date()} to {pd.to_datetime(df['date']).max().date()}.")
print("Each row = one stock on one decision date: a 104-bar weekly chart, a 126-bar daily chart, a 140-bar hourly chart,")
print("33 numeric features from the last 126 daily bars, and the labels. Rows from 2023-01-03 on stay sealed until Stage 10.")
print(f"Label base rates: beats median over 5 days {df['y_out5'].mean():.3f} | drawdown >= 10 % in 20 days {df['dd20_le_10'].mean():.3f}")
print(s1.round(3).to_string(index=False))
save_df(s1, "stage1_rows_by_year.csv")
print(f"Stage 1 done in {time.time() - t0:.0f}s")

## Stage 2 - split
A row joins a split only if its 20-day label window ends inside that split, so no label leaks across a boundary.

In [ ]:
# ==================================== STAGE 2: chronological split with a 20-day purge ====================================
d_, e_ = pd.to_datetime(df["date"]), pd.to_datetime(df["end20"])
split = np.full(len(df), "purged", dtype=object)
for name, (a, b) in SPLITS.items():
    m = ((d_ >= a) & (d_ < b) & (e_ < b)).values        # a row belongs to a split only if its 20-day label also ends inside it
    split[m] = name
df["split"] = split
n_purged = int((df["split"] == "purged").sum())
df = df[df["split"] != "purged"].reset_index(drop=True)
TR, VA, TE = (df["split"].values == s for s in ("train", "val", "test"))
for a_, b_ in (("train", "val"), ("val", "test")):
    assert pd.to_datetime(df.loc[df.split == a_, "end20"]).max() < pd.to_datetime(df.loc[df.split == b_, "date"]).min(), f"{a_}/{b_} overlap"
s2_tab = df.groupby("split").agg(rows=("sample_id", "size"), tickers=("ticker", "nunique"), first=("date", "min"), last=("date", "max"),
                                 vol_mean=("logrv20", "mean"), up5_rate=("y_out5", "mean"), dd10_rate=("dd20_le_10", "mean"))
s2_tab = s2_tab.loc[["train", "val", "test"]].reset_index()
print(s2_tab.to_string(index=False))
print(f"Purged {n_purged:,} rows whose 20-day label window crossed a split boundary. Holdout (from {HOLDOUT_START}) is not loaded.")
save_df(s2_tab, "stage2_splits.csv")
spath = os.path.join(RES, f"split_ids_{RUN_TAG}.parquet")
df[["sample_id", "ticker", "date", "split"]].to_parquet(spath, index=False)
push(spath, f"results_{RUN_TAG}/split_ids.parquet")

## Stage 3 - numeric baselines
The bar every image model has to clear. Volatility is scored by R² on log volatility (with QLIKE and within-month R² alongside); direction and drawdown by log loss and Brier score (with AUC, rank IC and calibration error).

In [ ]:
# ==================================== STAGE 3: numeric baselines (the bar every image model must clear) ====================================
t0 = time.time()
FEATS = C.DEPLOY                                            # 33 features from the last 126 daily bars (project rule)
Xraw = df[FEATS].to_numpy(np.float64)
num_fill = np.nanmedian(Xraw[TR], 0); num_fill = np.where(np.isfinite(num_fill), num_fill, 0.0)
Xf = np.where(np.isfinite(Xraw), Xraw, num_fill)
num_mean, num_std = T.standardize(Xf[TR])
XN = ((Xf - num_mean) / num_std).astype(np.float32)
Y = {"V": df["logrv20"].to_numpy(float), "O5": df["y_out5"].to_numpy(float), "D10": df["dd20_le_10"].to_numpy(float)}
OK = {t: np.isfinite(Y[t]) for t in Y}
BASE = {t: float(np.mean(Y[t][TR & OK[t]])) for t in ("O5", "D10")}
DATES, XRET = df["date"].values, df["xret5_med"].to_numpy(float)

# M0: training mean / base rate; HAR-range OLS for volatility; then linear and tree heads on the 33 numbers
VAL_PRED[("V", "M0", "constant")] = np.full(int((VA & OK["V"]).sum()), float(np.mean(Y["V"][TR & OK["V"]])))
for t in ("O5", "D10"):
    VAL_PRED[(t, "M0", "constant")] = np.full(int((VA & OK[t]).sum()), BASE[t])
HAR = [f"x_{k}_{w}" for k in ("cc", "park", "gk") for w in (1, 5, 22)] + ["x_ewma_win"]
har_ix = [FEATS.index(c) for c in HAR]
m_tr, m_va = TR & OK["V"], VA & OK["V"]
w_, b_, p_ = T.ridge_path(XN[m_tr][:, har_ix], Y["V"][m_tr], XN[m_va][:, har_ix], [1e-6])[1e-6]
VAL_PRED[("V", "HAR", "ols")] = p_
for t in TARGETS:
    fit_one(t, "NUM", XN, heads=("linear", "tree"), stage="3_numeric")
    tab = val_table(t)
    print(f"\n{TARGET_NAME[t]} - validation {SPLITS['val'][0][:4]}-{int(SPLITS['val'][1][:4]) - 1}:")
    print(tab.drop(columns=["target"]).round(4).to_string(index=False))
    save_df(tab, f"stage3_numeric_{t}.csv")
print(f"\nStage 3 done in {time.time() - t0:.0f}s")

## Stage 4 - frozen-backbone embeddings (GPU)
The fast chart reader is first checked against `ImageStore.triplet`. Each finished chunk of stocks is saved and pushed to Hugging Face, so a disconnect only loses the chunk in progress. Size and speed of every network are recorded for the model cards.

In [ ]:
# ==================================== STAGE 4: frozen-backbone embeddings for every chart (the GPU-heavy step) ====================================
t0 = time.time()
NAMES = BACKBONE_NAMES or list(T.BACKBONES)
rows_e = st._idx.set_index("sample_id").loc[df["sample_id"].values].reset_index()      # store rows in df order
reader = T.FastTriplets(st)
n_chk = reader.parity(rows_e, n=200)
print(f"Fast chart reader matches ImageStore.triplet on {n_chk} rows (including patched hourly charts).")

manifest = dict(names=NAMES, chunk_tickers=CHUNK_TICKERS, n_rows=len(rows_e),
                tickers_sha=hashlib.sha256(",".join(dict.fromkeys(rows_e["ticker"])).encode()).hexdigest())
mpath = os.path.join(EMB_DIR, "manifest.json")
if os.path.exists(mpath):
    assert json.load(open(mpath)) == manifest, "embedding manifest changed (backbones, rows or chunk size); use a new WORK folder"
json.dump(manifest, open(mpath, "w"), indent=1)
push(mpath, f"emb_{RUN_TAG}/manifest.json")


def chunk_store(local_dir, repo_dir):
    """(have_chunk, keep_chunk) for one embedding folder: fetch a chunk finished in an earlier Colab session (Drive copy first,
    then Hugging Face) so the run resumes; keep each new chunk on Hugging Face (or Drive if embeddings are not uploaded)."""
    drive = os.path.join(DRIVE_DIR, repo_dir)

    def have_chunk(fname):
        if os.path.exists(os.path.join(drive, fname)):
            shutil.copy(os.path.join(drive, fname), os.path.join(local_dir, fname))
            return os.path.join(local_dir, fname)
        if UPLOAD and UPLOAD_EMBEDDINGS:
            try:
                got = hf_hub_download(RUNS_REPO, f"{repo_dir}/{fname}", repo_type="dataset", local_dir=os.path.join(WORK, "_dl"), token=HF_TOKEN)
                shutil.move(got, os.path.join(local_dir, fname))
                return os.path.join(local_dir, fname)
            except Exception:
                return None
        return None

    def keep_chunk(path):
        if UPLOAD and UPLOAD_EMBEDDINGS:
            push(path, f"{repo_dir}/{os.path.basename(path)}")
        else:
            os.makedirs(drive, exist_ok=True)
            shutil.copy(path, os.path.join(drive, os.path.basename(path)))
    return have_chunk, keep_chunk


have_chunk, keep_chunk = chunk_store(EMB_DIR, f"emb_{RUN_TAG}")
records = T.extract_embeddings(st, rows_e, NAMES, EMB_DIR, DEVICE, chunk_tickers=CHUNK_TICKERS, batch=BATCH,
                               factory=globals().get("FACTORY"), on_chunk=keep_chunk, have_chunk=have_chunk)
bpath = os.path.join(RES, "backbones.json")
json.dump(records, open(bpath, "w"), indent=1)
push(bpath, f"results_{RUN_TAG}/backbones.json")
print(f"Embeddings ready for {len(NAMES)} backbones x 3 views x {len(rows_e):,} charts in {(time.time() - t0) / 60:.1f} min.")
print(pd.DataFrame([{"backbone": k, "timm weights": v["timm_name"], "parameters (M)": v["n_params"] / 1e6, "embedding size": v["emb_dim"],
                     "images/s on this GPU": v["infer_img_per_s"]} for k, v in records.items()]).round(1).to_string(index=False))

## Stage 5 - combine the timeframes and tune
PCA per backbone and view (fitted on training rows), then: images only; numbers + all three views (linear and trees); numbers + one view (the single-timeframe condition for H1).

In [ ]:
# ==================================== STAGE 5: PCA, then image and combined heads with hyper-parameter tuning ====================================
t0 = time.time()
PCA_EXPL = {}
order_ids = df["sample_id"].values


def pca_backbones(names):
    """PCA per backbone and view, fitted on training rows only (resumable: finished files are kept)."""
    for b in names:
        for v in T.VIEWS:
            f, fm = os.path.join(PCA_DIR, f"{b}__{v}.npy"), os.path.join(PCA_DIR, f"{b}__{v}_meta.npz")
            if not (os.path.exists(f) and os.path.exists(fm)):
                Xe = T.assemble(emb_dir(b), f"{b}|{v}", order_ids)
                Z, mu, comp, expl = T.pca_fit_transform(Xe, TR, min(PCA_DIM, Xe.shape[1]), seed=SEED, device=DEVICE)
                np.save(f, Z.astype(np.float32)); np.savez(fm, mean=mu, comp=comp, expl=expl)
                del Xe, Z; gc.collect()
            PCA_EXPL[f"{b}|{v}"] = float(np.load(fm)["expl"])
            push(fm, f"pca_{RUN_TAG}/{os.path.basename(fm)}")


def fit_backbones(names, stage="5"):
    """Images only (linear), numbers + all three views (linear + trees) and, for frozen pretrained backbones, numbers + one view."""
    for b in names:
        tb = time.time()
        for t in TARGETS:
            fit_one(t, f"IMG:{b}", design(f"IMG:{b}"), heads=("linear",), stage=f"{stage}_images_only")
            heads = ("linear", "tree") if (T.base_name(b) in T.PRETRAINED or b == "resnet18_random_s0") else ("linear",)
            fit_one(t, f"NUM+IMG:{b}", design(f"NUM+IMG:{b}"), heads=heads, stage=f"{stage}_numbers_plus_images")
        if b in T.PRETRAINED:
            for v in T.VIEWS:                               # single-timeframe conditions (H1 breakdown)
                for t in ("V", "O5"):
                    fit_one(t, f"NUM+IMG:{b}:{v}", design(f"NUM+IMG:{b}:{v}"), heads=("linear",), stage=f"{stage}_single_view")
        print(f"   {b}: fitted in {time.time() - tb:.0f}s")


def refresh_val_tables():
    global VAL_TABLES
    VAL_TABLES = {t: val_table(t) for t in TARGETS}
    for t in TARGETS:
        save_df(VAL_TABLES[t], f"stage5_validation_{t}.csv")
    save_df(pd.DataFrame(LEDGER), "ledger.csv")


pca_backbones(NAMES)
print(f"PCA ({PCA_DIM} components per view, fitted on training rows) - variance kept, mean over views:")
print(pd.Series({b: np.mean([PCA_EXPL[f'{b}|{v}'] for v in T.VIEWS]) for b in NAMES}).round(3).to_string())
fit_backbones(NAMES)
refresh_val_tables()
print(f"\nStage 5 done in {(time.time() - t0) / 60:.1f} min. Configurations tried so far (all logged): {len(LEDGER):,}")

## Stage 5F - fine-tuning (new in v3)
**Until v2 every backbone was frozen.** Here each of the 7 pretrained backbones is fine-tuned end to end: one network embeds the weekly, daily and hourly chart, a small head adds the 33 numbers and predicts volatility, direction and drawdown together. Training years only, AdamW with a cosine schedule, early stopping on the validation loss, no augmentation (a flipped chart means something else). The fine-tuned embeddings then go through the same PCA and heads as the frozen ones and compete under the same selection rule. The cell reports, per backbone, frozen vs fine-tuned validation scores with month-block confidence intervals, the network's own predictions, the training curves and the GPU time, so *is fine-tuning worth it?* gets a measured answer. Resumable per backbone.

In [ ]:
# ==================================== STAGE 5F: fine-tune the pretrained backbones end to end (v3) ====================================
# Until v2 the backbones were FROZEN: ImageNet features were extracted once and only the small heads were trained. Here each
# pretrained backbone is trained further on the charts themselves (training years only; early stopping on validation; no
# augmentation). Its fine-tuned embeddings then go through the same PCA and heads and compete under the same selection rule.
import matplotlib; import matplotlib.pyplot as plt
t0 = time.time()
FT_NAMES, FT_REC, FT_HIST, FT_E2E = [], {}, [], {}
if FINETUNE:
    bases = list(T.PRETRAINED) if FT_BACKBONES == "all" else [b for b in FT_BACKBONES if b in T.PRETRAINED]
    FT_NAMES = [T.FT_PREFIX + b for b in bases]
    rng_ft = np.random.default_rng(SEED)
    tr_rows, va_rows = np.flatnonzero(TR), np.flatnonzero(VA)
    if len(tr_rows) > FT_MAX_TRAIN_ROWS:
        tr_rows = np.sort(rng_ft.choice(tr_rows, FT_MAX_TRAIN_ROWS, replace=False))
    if len(va_rows) > FT_MAX_VAL_ROWS:
        va_rows = np.sort(rng_ft.choice(va_rows, FT_MAX_VAL_ROWS, replace=False))
    sel_ft = np.concatenate([tr_rows, va_rows])
    rows_ft = st._idx.set_index("sample_id").loc[df["sample_id"].values[sel_ft]].reset_index()
    tp = time.time()
    PACKED = T.packed_rows(st, rows_ft)
    g0 = rows_ft[rows_ft["ticker"] == rows_ft["ticker"].iloc[0]].iloc[:16]
    raw = T.views_from_packed(PACKED[:len(g0)], "cpu", resize=False); ref = T.FastTriplets(st).read(g0)
    assert all(np.array_equal(raw[v][:, 0].numpy().astype(np.uint8), ref[v]) for v in T.VIEWS), "packed chart decoder differs from the store"
    print(f"Fine-tuning data: {len(tr_rows):,} training and {len(va_rows):,} validation chart sets packed in memory "
          f"({PACKED.nbytes / 1e9:.2f} GB, {time.time() - tp:.0f}s); decoder checked against the store.")
    xn_ft, ys_ft = XN[sel_ft], (Y["V"][sel_ft], Y["O5"][sel_ft], Y["D10"][sel_ft])
    i_tr, i_va = np.arange(len(tr_rows)), np.arange(len(tr_rows), len(sel_ft))
    FT_BATCH = 64 if GPU_GB > 30 else 32
    FT_DRIVE = os.path.join(DRIVE_DIR, f"ft_{RUN_TAG}")
    os.makedirs(FT_DRIVE, exist_ok=True)

    def ft_fetch(fname):
        local = os.path.join(WORK, "ft", fname)
        if os.path.exists(local):
            return local
        if os.path.exists(os.path.join(FT_DRIVE, fname)):
            shutil.copy(os.path.join(FT_DRIVE, fname), local); return local
        if UPLOAD:
            try:
                got = hf_hub_download(RUNS_REPO, f"ft_{RUN_TAG}/{fname}", repo_type="dataset", local_dir=os.path.join(WORK, "_dl"), token=HF_TOKEN)
                shutil.copy(got, local); return local
            except Exception:
                return None
        return None

    for b in bases:
        nm = T.FT_PREFIX + b
        wfile, mfile = f"{nm}.safetensors", f"{nm}.json"
        w_, m_ = ft_fetch(wfile), ft_fetch(mfile)
        if w_ and m_:                                           # finished in an earlier session: reuse (resumable)
            meta = json.load(open(m_))
            T.FT_WEIGHTS[nm] = w_
            print(f"{nm}: fine-tuned weights found (from an earlier session), skipping training.")
        else:
            print(f"Fine-tuning {b} (up to {FT_EPOCHS} epochs, batch {FT_BATCH} chart sets = {3 * FT_BATCH} images)...")
            m, mu, sd, rec, hist, e2e = T.finetune(b, PACKED, xn_ft, ys_ft, i_tr, i_va, DEVICE, epochs=FT_EPOCHS, batch=FT_BATCH,
                                                   lr_backbone=FT_LR_BACKBONE, lr_head=FT_LR_HEAD, seed=SEED,
                                                   factory=globals().get("FACTORY"))
            w_ = os.path.join(WORK, "ft", wfile); T.save_backbone(m, w_); T.FT_WEIGHTS[nm] = w_
            meta = dict(record=rec, history=hist.to_dict("records"), e2e_val={k: np.asarray(v).tolist() for k, v in e2e.items()},
                        val_sample_ids=df["sample_id"].values[va_rows].tolist(), weights_sha256=_sha(w_))
            m_ = os.path.join(WORK, "ft", mfile); json.dump(meta, open(m_, "w"), default=float)
            for f_ in (w_, m_):
                shutil.copy(f_, os.path.join(FT_DRIVE, os.path.basename(f_)))
                push(f_, f"ft_{RUN_TAG}/{os.path.basename(f_)}")       # private runs repo only (weights are never published)
            del m; gc.collect()
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
        FT_REC[nm] = meta["record"]; FT_HIST += meta["history"]
        FT_E2E[nm] = {k: np.asarray(v) for k, v in meta["e2e_val"].items()}
    del PACKED; gc.collect()

    # ---- embeddings of every chart from the fine-tuned backbones (own folder per set of weights, so stale chunks are never reused)
    wsig = hashlib.sha256("".join(_sha(T.FT_WEIGHTS[n]) for n in FT_NAMES).encode()).hexdigest()[:10]
    EMB_FT_DIR = os.path.join(WORK, f"emb_ft_{RUN_TAG}_{wsig}"); os.makedirs(EMB_FT_DIR, exist_ok=True)
    have_ft, keep_ft = chunk_store(EMB_FT_DIR, f"emb_ft_{RUN_TAG}_{wsig}")
    rec_ft = T.extract_embeddings(st, rows_e, FT_NAMES, EMB_FT_DIR, DEVICE, chunk_tickers=CHUNK_TICKERS, batch=BATCH,
                                  factory=globals().get("FACTORY"), on_chunk=keep_ft, have_chunk=have_ft)
    for nm in FT_NAMES:
        rec_ft[nm]["ft"] = FT_REC[nm].get("ft", {})
    records.update(rec_ft)
    json.dump(records, open(bpath, "w"), indent=1)
    push(bpath, f"results_{RUN_TAG}/backbones.json")

    # ---- the same PCA and heads as the frozen backbones
    for nm in FT_NAMES:                                      # new weights => the PCA must be refitted
        mark, cur = os.path.join(PCA_DIR, f"{nm}.weights"), _sha(T.FT_WEIGHTS[nm])
        if not os.path.exists(mark) or open(mark).read() != cur:
            for v in T.VIEWS:
                for suf in (".npy", "_meta.npz"):
                    if os.path.exists(os.path.join(PCA_DIR, f"{nm}__{v}{suf}")):
                        os.remove(os.path.join(PCA_DIR, f"{nm}__{v}{suf}"))
            open(mark, "w").write(cur)
    pca_backbones(FT_NAMES)
    _IMG_CACHE.clear()
    fit_backbones(FT_NAMES, stage="5F")
    refresh_val_tables()

    # ---- did fine-tuning help? frozen vs fine-tuned on validation (month-block bootstrap CIs on the same resamples)
    BOOT_FT = {t: T.MonthBoot(DATES[VA & OK[t]], min(N_BOOT, 500), seed=SEED) for t in TARGETS}

    def best_head(t, fs):
        tab = VAL_TABLES[t]; met, sign = PRIMARY[t]
        sub = tab[tab.feature_set == fs]
        return sub.loc[(sign * sub[met]).idxmax(), "head"]

    rows_ft_cmp = []
    for b in bases:
        nm = T.FT_PREFIX + b
        r = dict(backbone=b, model=T.display_name(b))
        for t, fn, met in (("V", T.r2, "R2"), ("D10", T.auc, "AUC"), ("O5", T.auc, "AUC")):
            y = Y[t][VA & OK[t]]
            for cond, fs in (("img", "IMG:{}"), ("num_img", "NUM+IMG:{}")):
                hf, hz = best_head(t, fs.format(b)), best_head(t, fs.format(nm))
                pf, pz = VAL_PRED[(t, fs.format(b), hf)], VAL_PRED[(t, fs.format(nm), hz)]
                r[f"{t}_{cond}_frozen"], r[f"{t}_{cond}_finetuned"] = fn(y, pf), fn(y, pz)
                if cond == "num_img":
                    ci, _ = BOOT_FT[t].diff(fn, y, pz, pf)
                    r[f"{t}_gain"], r[f"{t}_gain_lo"], r[f"{t}_gain_hi"] = fn(y, pz) - fn(y, pf), ci[0], ci[1]
            yv_ = Y[t][va_rows]; okv_ = np.isfinite(yv_)            # the network's own predictions (validation rows it was scored on)
            r[f"{t}_end_to_end"] = fn(yv_[okv_], FT_E2E[nm][t][okv_]) if okv_.sum() > 10 else np.nan
        fr = FT_REC[nm].get("ft", {})
        r.update(epochs=fr.get("epochs_run"), best_epoch=fr.get("best_epoch"), minutes=fr.get("minutes"), train_img_per_s=fr.get("train_img_per_s"))
        rows_ft_cmp.append(r)
    FT_CMP = pd.DataFrame(rows_ft_cmp)
    save_df(FT_CMP, "stage5F_finetune_vs_frozen.csv")
    save_df(pd.DataFrame(FT_HIST), "stage5F_training_history.csv")
    n_up = int((FT_CMP["V_gain"] > 0).sum()); n_sig = int((FT_CMP["V_gain_lo"] > 0).sum()); n_sig_dn = int((FT_CMP["V_gain_hi"] < 0).sum())
    d_up = int((FT_CMP["D10_gain"] > 0).sum()); d_sig = int((FT_CMP["D10_gain_lo"] > 0).sum())
    FT_VERDICT = (f"Fine-tuning raised validation volatility R² (numbers + images) for {n_up} of {len(FT_CMP)} backbones "
                  f"(significantly for {n_sig}, significantly worse for {n_sig_dn}; mean change {FT_CMP['V_gain'].mean():+.4f}) and drawdown "
                  f"AUC for {d_up} of {len(FT_CMP)} (significantly for {d_sig}). Images-only volatility R² moved from "
                  f"{FT_CMP['V_img_frozen'].mean():.3f} (frozen, average) to {FT_CMP['V_img_finetuned'].mean():.3f} (fine-tuned). "
                  f"Total fine-tuning time {FT_CMP['minutes'].sum():.0f} GPU-minutes.")
    print("\nFrozen vs fine-tuned on validation (numbers + images, best head; gain = fine-tuned minus frozen, 95% month-block CI):")
    show = FT_CMP[["model", "V_num_img_frozen", "V_num_img_finetuned", "V_gain", "V_gain_lo", "V_gain_hi", "D10_num_img_frozen",
                   "D10_num_img_finetuned", "V_img_frozen", "V_img_finetuned", "V_end_to_end", "epochs", "best_epoch", "minutes"]]
    print(show.round(4).to_string(index=False))
    print("\n" + FT_VERDICT)

    # ---- figures: training curves and frozen vs fine-tuned
    H_ = pd.DataFrame(FT_HIST)
    fig, ax = plt.subplots(1, 3, figsize=(18, 4.6))
    for i, (b, g) in enumerate(H_.groupby("backbone", sort=False)):
        ax[0].plot(g["epoch"], g["val_loss"], marker="o", label=T.display_name(b))
        ax[1].plot(g["epoch"], g["val_R2_V"], marker="o", label=T.display_name(b))
    ax[0].set_title("Validation loss per epoch (lower is better)"); ax[0].set_xlabel("epoch"); ax[0].legend(fontsize=7)
    ax[1].set_title("Validation volatility R² of the network itself"); ax[1].set_xlabel("epoch")
    x_ = np.arange(len(FT_CMP)); w_b = 0.38
    ax[2].bar(x_ - w_b / 2, FT_CMP["V_num_img_frozen"], w_b, label="frozen", color="#3b6ea5")
    ax[2].bar(x_ + w_b / 2, FT_CMP["V_num_img_finetuned"], w_b, label="fine-tuned", color="#e67e22",
              yerr=np.vstack([FT_CMP["V_gain"] - FT_CMP["V_gain_lo"], FT_CMP["V_gain_hi"] - FT_CMP["V_gain"]]), capsize=3)
    ax[2].axhline(T.r2(Y["V"][VA & OK["V"]], VAL_PRED[("V", "NUM", best_head("V", "NUM"))]), color="#c0392b", ls="--", label="numbers only")
    lo_ = min(FT_CMP[["V_num_img_frozen", "V_num_img_finetuned"]].min()) - 0.03
    ax[2].set_ylim(lo_, None); ax[2].set_xticks(x_); ax[2].set_xticklabels(FT_CMP["model"], rotation=35, ha="right", fontsize=8)
    ax[2].set_title("Volatility R² (numbers + images, validation)"); ax[2].legend(fontsize=8)
    plt.tight_layout(); fpath = os.path.join(FIG, "stage5F_finetuning.png"); plt.savefig(fpath, dpi=130); plt.show()
    push(fpath, f"figures_{RUN_TAG}/stage5F_finetuning.png")
else:
    FT_CMP, FT_VERDICT = None, "Fine-tuning was switched off (FINETUNE = False): all image models use frozen backbones."
    print(FT_VERDICT)
print(f"Stage 5F done in {(time.time() - t0) / 60:.1f} min.")

## Stage 6 - pick the model, calibrate, freeze
The selection rule and the hypothesis tests were written before any test data is read. The frozen config is the only thing later stages use.

In [ ]:
# ==================================== STAGE 6: compare, select (pre-registered rule), calibrate, freeze ====================================
import matplotlib; import matplotlib.pyplot as plt
t0 = time.time()
METRIC_FN = {"V": T.r2, "O5": T.logloss, "D10": T.brier}
PREREG = {
    "selection": "Per target the best validation score wins (V: R2, O5: log loss, D10: Brier). Candidates: numbers only, and numbers "
                 "and/or images from every pretrained backbone, frozen or fine-tuned (Stage 5F), on equal terms. A model that uses chart "
                 "images is chosen only if its validation gain over the best numbers-only model has a month-block 95% CI that excludes 0.",
    "finetune": "Fine-tuning (Stage 5F): each pretrained backbone is trained end to end on training rows only (shared backbone over the "
                "three views + 33 numbers, multi-task loss: MSE on volatility, plain cross-entropy on direction and drawdown; AdamW, "
                "cosine schedule, no augmentation), with early stopping on the validation loss; its embeddings then use the same PCA and heads.",
    "monte_carlo": "Stage 7M: the simulation's tail shape and intraday-low range are chosen on validation (return coverage, then drawdown "
                   "Brier); the dashboard matches the drawdown model by default unless that worsens the validation coverage error by "
                   "more than 1 percentage point. Checked once on the test years.",
    "H1": "On the test years, the best numbers+images model (chosen on validation among pretrained backbones) beats the numbers-only "
          "model: the 95% CI of delta-R2 (volatility) or delta-AUC (direction) lies above 0.",
    "H2": "On validation, image-only volatility R2 differs between at least two pretrained backbones (pairwise month-block "
          "bootstrap, Holm-adjusted p < 0.05).",
    "H3": "On the test years, accuracy of direction calls at 10% coverage (confidence threshold fixed on validation) exceeds "
          "accuracy at 100% coverage: the 95% CI of the difference lies above 0.",
    "no_trade": "NO TRADE unless direction is usable, confidence passes its validation threshold, and neither predicted volatility "
                "nor drawdown probability is in its validation top 20%. Direction is usable only if (a) on validation the Wilson "
                "lower bound of accuracy exceeds 0.5 at some coverage AND (b) on the test years the direction model's AUC 95% CI "
                "lies above 0.5 (validation alone is optimistic because it also chose the model).",
}
BOOT_VA = {t: T.MonthBoot(DATES[VA & OK[t]], N_BOOT, seed=SEED) for t in TARGETS}
SEL, VAL_SUMMARY = {}, {}
for t in TARGETS:
    tab = VAL_TABLES[t].copy()
    cand = tab[(tab.feature_set == "NUM") | tab.feature_set.str.startswith(("NUM+IMG:", "IMG:"))]
    cand = cand[~cand.feature_set.str.contains("random")]
    met, sign = PRIMARY[t]
    cand = cand.assign(score=sign * cand[met]).sort_values("score", ascending=False)
    best, num = cand.iloc[0], cand[cand.feature_set == "NUM"].iloc[0]
    y = Y[t][VA & OK[t]]
    key = lambda r: (t, r["feature_set"], r["head"])
    chosen, gate = num, None
    if best["feature_set"] != "NUM":
        ci, _ = BOOT_VA[t].diff(METRIC_FN[t], y, VAL_PRED[key(best)], VAL_PRED[key(num)])
        gate = dict(candidate=f"{best['feature_set']} / {best['head']}", delta=float(best[met] - num[met]), ci=ci)
        if (ci[0] > 0) if sign > 0 else (ci[1] < 0):
            chosen = best
    SEL[t] = dict(feature_set=chosen["feature_set"], head=chosen["head"], gate=gate,
                  fit={k: v for k, v in FITS[key(chosen)].items() if k not in ("model", "coef", "intercept")})
    # H1 comparison pair, fixed now: best numbers-only head and best numbers + all-three-views model (pretrained backbones)
    multi = cand[cand.feature_set.str.fullmatch(r"NUM\+IMG:[^:]+")]
    SEL[t]["h1_numbers"] = dict(feature_set="NUM", head=num["head"])
    SEL[t]["h1_images"] = dict(feature_set=multi.iloc[0]["feature_set"], head=multi.iloc[0]["head"]) if len(multi) else None
    VAL_SUMMARY[t] = {k: float(chosen[k]) for k in chosen.index if k in ("R2", "wmR2", "QLIKE", "MAE", "AUC", "logloss", "brier", "BSS", "ECE", "rankIC", "rankIC_t")}
    print(f"{t}: chosen {chosen['feature_set']} / {chosen['head']} ({met} {chosen[met]:.4f})"
          + (f"; best image model {gate['candidate']} delta {gate['delta']:+.4f}, CI [{gate['ci'][0]:+.4f}, {gate['ci'][1]:+.4f}]" if gate else ""))

# ---- H2 (validation): do pretrained backbones differ in image-only volatility R2?
yV, bV = Y["V"][VA & OK["V"]], BOOT_VA["V"]
img_r2 = {}
for b in NAMES:
    p = VAL_PRED[("V", f"IMG:{b}", "linear")]
    img_r2[b] = dict(R2=T.r2(yV, p), CI=bV.ci(T.r2, yV, p)[0])
pv = {}
pre = [b for b in NAMES if b in T.PRETRAINED]
for i, a in enumerate(pre):
    for b in pre[i + 1:]:
        _, draws = bV.diff(T.r2, yV, VAL_PRED[("V", f"IMG:{a}", "linear")], VAL_PRED[("V", f"IMG:{b}", "linear")])
        pv[f"{a} vs {b}"] = float(min(1.0, 2 * min((draws <= 0).mean(), (draws >= 0).mean())))
padj = T.holm(pv) if pv else {}
H2 = dict(supported=bool(any(p < 0.05 for p in padj.values())), n_pairs=len(pv),
          significant_pairs=[k for k, p in padj.items() if p < 0.05], image_only_R2=img_r2)
print(f"\nH2 (backbones differ, validation): {'SUPPORTED' if H2['supported'] else 'NOT SUPPORTED'} "
      f"- {len(H2['significant_pairs'])} of {len(pv)} pairs differ after Holm correction.")
print(pd.DataFrame({b: {"image-only R2": v["R2"], "lo": v["CI"][0], "hi": v["CI"][1]} for b, v in img_r2.items()}).T.round(3).to_string())

# ---- per-view (single timeframe) vs all three views, validation delta vs numbers only
rows = []
for b in pre:
    for t, met in (("V", "R2"), ("O5", "AUC")):
        tab = VAL_TABLES[t].set_index(["feature_set", "head"])
        base_ = tab.loc[("NUM", "linear"), met]
        for v in ("W", "D", "H", None):
            fs = f"NUM+IMG:{b}" + (f":{v}" if v else "")
            if (fs, "linear") in tab.index:
                rows.append(dict(backbone=b, target=t, views=v or "WDH", delta=float(tab.loc[(fs, "linear"), met] - base_)))
views_df = pd.DataFrame(rows)
save_df(views_df, "stage6_single_vs_multi_timeframe.csv")
print("\nValidation gain over numbers only (linear heads): single timeframe vs all three")
print(views_df.pivot_table(index=["target", "backbone"], columns="views", values="delta").round(4).to_string())

# ---- controls promised in the proposal: label shuffle (within each training month) and the naive majority-class baseline
rng_s = np.random.default_rng(SEED)
CONTROLS = {}
for t, fs in (("V", SEL["V"]["h1_images"]["feature_set"] if SEL["V"]["h1_images"] else "NUM"), ("O5", "NUM")):
    X = design(fs); mt, mv = TR & OK[t], VA & OK[t]
    ys = Y[t][mt].copy()
    if t == "V":                                          # volatility: shuffle across ALL training rows (a within-month shuffle
        ys = ys[rng_s.permutation(len(ys))]               # keeps month averages, which still teach a usable cross-sectional slope)
    else:                                                 # direction: within-month shuffle keeps the 50/50 base rate per period
        mon_tr, _ = T.month_codes(DATES[mt])
        for k in np.unique(mon_tr):
            ii = np.flatnonzero(mon_tr == k); ys[ii] = ys[rng_s.permutation(ii)]
    if t == "V":
        _, _, p = T.ridge_path(X[mt], ys, X[mv], [1000.0])[1000.0]
        yv = Y[t][mv]; c_, _ = T.month_codes(DATES[mv])
        CONTROLS["label_shuffle_V"] = dict(feature_set=fs, R2=T.r2(yv, p), wmR2=T.wm_r2(yv, p, c_))
    else:
        _, _, p = T.logit_path(X[mt], ys, X[mv], [0.01], device=DEVICE)[0.01]
        CONTROLS["label_shuffle_O5"] = dict(feature_set=fs, AUC=T.auc(Y[t][mv], p))
maj = int(BASE["O5"] >= 0.5)
CONTROLS["majority_class_accuracy_O5"] = float((Y["O5"][VA & OK["O5"]] == maj).mean())
print("\nControls (validation): " + "; ".join(f"{k}: " + ", ".join(f"{a} {b:.4f}" for a, b in v.items() if isinstance(b, float))
                                          if isinstance(v, dict) else f"{k}: {v:.4f}" for k, v in CONTROLS.items()))
print("   (with shuffled labels the pipeline must find nothing: volatility R2 at or below 0, within-month R2 near 0, direction AUC near 0.5)")

# ---- calibration, volatility residual band, NO TRADE rule (all from validation predictions)
def _vp(t):
    return VAL_PRED[(t, SEL[t]["feature_set"], SEL[t]["head"])]
CAL = {t: T.isotonic_arrays(_vp(t), Y[t][VA & OK[t]]) for t in ("O5", "D10")}
cal = lambda t, p: np.interp(p, CAL[t]["x"], CAL[t]["y"])
res = Y["V"][VA & OK["V"]] - _vp("V")
VRES = dict(res_q10=float(np.quantile(res, 0.10)), res_q50=float(np.quantile(res, 0.50)), res_q90=float(np.quantile(res, 0.90)), s2=float(np.var(res)),
            res_quantiles=[float(q) for q in np.quantile(res, np.linspace(0, 1, 101))])      # the error distribution the simulation draws from
pO5_va = cal("O5", _vp("O5")); yO5_va = Y["O5"][VA & OK["O5"]]
abst_va, THR = T.abstention_table(pO5_va, yO5_va)
usable = abst_va[(abst_va.coverage_target < 1) & (abst_va.wilson_lower > 0.5)]
cov_dir = float(usable.coverage_target.max()) if len(usable) else None
RULE = dict(vol_ann_q80=float(np.quantile(C.ann_vol(_vp("V")), 0.8)), p_dd10_q80=float(np.quantile(cal("D10", _vp("D10")), 0.8)),
            direction_usable=bool(len(usable)), coverage_for_direction=cov_dir,
            conf_threshold=float(THR[cov_dir]) if cov_dir else 0.5, thresholds_by_coverage={str(k): float(v) for k, v in THR.items()})
LEVELS = [round(float(q), 2) for q in np.arange(0.05, 0.96, 0.05)]                         # risk percentiles for the dashboard tiers
RISKQ = dict(levels=LEVELS, vol_ann=[float(q) for q in np.quantile(C.ann_vol(_vp("V")), LEVELS)],
             p_dd10=[float(q) for q in np.quantile(cal("D10", _vp("D10")), LEVELS)])
REFERENCE = dict(vol_ann_median_train=float(np.median(C.ann_vol(Y["V"][TR & OK["V"]]))),
                 vol_ann_q20_train=float(np.quantile(C.ann_vol(Y["V"][TR & OK["V"]]), 0.2)),
                 vol_ann_q80_train=float(np.quantile(C.ann_vol(Y["V"][TR & OK["V"]]), 0.8)),
                 dd10_rate={s_: float(np.nanmean(Y["D10"][m_ & OK["D10"]])) for s_, m_ in (("train", TR), ("val", VA))})
save_df(abst_va, "stage6_abstention_validation.csv")
print("\nAbstention on validation (calibrated direction probabilities):")
print(abst_va.round(4).to_string(index=False))
print(f"Direction usable: {RULE['direction_usable']} | NO TRADE risk thresholds: volatility >= {RULE['vol_ann_q80']:.1%} a year "
      f"or P(drawdown >= 10%) >= {RULE['p_dd10_q80']:.1%}")

# ---- figures
fig, ax = plt.subplots(1, 2, figsize=(15, 4.8))
names_ = list(img_r2)
ax[0].bar(range(len(names_)), [img_r2[b]["R2"] for b in names_], yerr=np.array([[img_r2[b]["R2"] - img_r2[b]["CI"][0] for b in names_],
          [img_r2[b]["CI"][1] - img_r2[b]["R2"] for b in names_]]), color=["#3b6ea5" if b in T.PRETRAINED else "#999999" for b in names_], capsize=3)
ax[0].axhline(VAL_SUMMARY["V"].get("R2", np.nan) if SEL["V"]["feature_set"] == "NUM" else T.r2(yV, VAL_PRED[("V", "NUM", "linear")]), color="#c0392b", ls="--", label="numbers only")
ax[0].set_xticks(range(len(names_))); ax[0].set_xticklabels(names_, rotation=40, ha="right", fontsize=8)
ax[0].set_title("Volatility R2 from chart images alone (validation; grey = random weights)"); ax[0].legend()
vd = views_df[views_df.target == "V"].pivot_table(index="backbone", columns="views", values="delta")
im = ax[1].imshow(vd.values, cmap="RdBu", vmin=-0.02, vmax=0.02, aspect="auto")
ax[1].set_xticks(range(vd.shape[1])); ax[1].set_xticklabels(vd.columns); ax[1].set_yticks(range(vd.shape[0])); ax[1].set_yticklabels(vd.index, fontsize=8)
ax[1].set_title("Validation R2 gain over numbers: single view vs all three"); plt.colorbar(im, ax=ax[1])
plt.tight_layout(); fpath = os.path.join(FIG, "stage6_validation.png"); plt.savefig(fpath, dpi=130); plt.show()
push(fpath, f"figures_{RUN_TAG}/stage6_validation.png")

# ---- freeze: everything the test and holdout stages are allowed to use
FROZEN = dict(run_tag=RUN_TAG, created_utc=dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"), splits=SPLITS,
              holdout_start=HOLDOUT_START, n_rows={s: int(m.sum()) for s, m in (("train", TR), ("val", VA), ("test", TE))},
              split_ids_sha256=_sha(os.path.join(RES, f"split_ids_{RUN_TAG}.parquet")),
              selection=SEL, calibrators=CAL, volatility_residuals=VRES, rule=RULE, prereg=PREREG, H2=H2, controls=CONTROLS,
              n_configurations_tried=len(LEDGER), backbones=NAMES, ft_backbones=FT_NAMES, pca_dim=PCA_DIM, seed=SEED,
              finetune=dict(enabled=bool(FINETUNE), epochs=FT_EPOCHS, lr_backbone=FT_LR_BACKBONE, lr_head=FT_LR_HEAD,
                            max_train_rows=FT_MAX_TRAIN_ROWS, max_val_rows=FT_MAX_VAL_ROWS),
              risk_quantiles=RISKQ, reference=REFERENCE)
FROZEN_PATH = os.path.join(WORK, "frozen", f"frozen_config_{RUN_TAG}.json")
json.dump(FROZEN, open(FROZEN_PATH, "w"), indent=1, default=float)
shutil.copy(FROZEN_PATH, os.path.join(DRIVE_DIR, os.path.basename(FROZEN_PATH)))
push(FROZEN_PATH, f"frozen/{os.path.basename(FROZEN_PATH)}")
print(f"\nFrozen: {FROZEN_PATH} (also on Drive and Hugging Face). Nothing below may change these choices.")
print(f"Stage 6 done in {time.time() - t0:.0f}s")

## Stage 6L - the model leaderboard (new in v3)
Which pretrained models were used and what they are (model cards), then every model, frozen and fine-tuned, on six metrics per target with 95% confidence intervals: R², within-month R², QLIKE, MAE, RMSE and Spearman for volatility; AUC, average precision, accuracy, log loss, Brier skill and calibration error for drawdown and direction. Charts: ranked bars, an all-metrics heatmap, and accuracy against model size and speed. Validation years only, so nothing here can change the frozen choices.

In [ ]:
# ==================================== STAGE 6L: model leaderboard - every pretrained model on every metric (validation) ====================================
# Which pretrained models were used, what they are, and how each scored on six metrics per target, frozen and fine-tuned,
# with the numbers-only baselines for reference. Validation years only: nothing here can change the frozen choices.
t0 = time.time()
recs_all = json.load(open(bpath))
LB_BOOT = {t: T.MonthBoot(DATES[VA & OK[t]], min(N_BOOT, 400), seed=SEED) for t in TARGETS}

# ---- 1. model cards
cards = []
for b in NAMES + FT_NAMES:
    rec = recs_all.get(b, {}); base = T.base_name(b)
    fam, nm, desc = T.BACKBONE_INFO.get(base, ("?", base, ""))
    ft = rec.get("ft") or {}
    cards.append({"model": T.display_name(b), "key": b, "family": fam, "what it is": desc,
                  "pretrained on": T.pretrain_data(rec.get("timm_name", ""), rec.get("pretrained", True)),
                  "parameters (M)": rec.get("n_params", np.nan) / 1e6, "embedding size": rec.get("emb_dim"),
                  "PCA variance kept": np.mean([PCA_EXPL.get(f"{b}|{v}", np.nan) for v in T.VIEWS]),
                  "inference images/s": rec.get("infer_img_per_s"), "fine-tuning": (f"{ft.get('epochs_run')} epochs (best {ft.get('best_epoch')}), "
                                                                               f"{ft.get('minutes', 0):.0f} min" if ft else "frozen")})
CARDS = pd.DataFrame(cards)
save_df(CARDS, "stage6L_model_cards.csv")
print("The vision models compared (all from timm, ImageNet-style inputs of 224 x 224; each embeds the weekly, daily and hourly chart separately):")
print(CARDS.drop(columns=["key", "what it is"]).round(3).to_string(index=False))

# ---- 2. every metric, every model, both conditions (images only / numbers + images)
PRIMARY_LB = {"V": ("R2", T.r2), "D10": ("AUC", T.auc), "O5": ("AUC", T.auc)}


def lb_row(t, label, kind, fs, head, cond):
    m = VA & OK[t]; y = Y[t][m]; p = VAL_PRED[(t, fs, head)]
    met = T.board_metrics(TARGETS[t], y, p, DATES[m], base=BASE.get(t))
    pm, fn = PRIMARY_LB[t]
    ci, _ = LB_BOOT[t].ci(fn, y, p)
    chosen = (SEL[t]["feature_set"], SEL[t]["head"]) == (fs, head)
    return dict(target=t, model=label, kind=kind, condition=cond, feature_set=fs, head=head, chosen=chosen,
                primary=pm, primary_value=met[pm], ci_lo=ci[0], ci_hi=ci[1], **met)


def best_of(t, fs):
    tab = VAL_TABLES[t]; met, sign = PRIMARY[t]
    sub = tab[tab.feature_set == fs]
    return None if sub.empty else sub.loc[(sign * sub[met]).idxmax(), "head"]


lb = []
for t in TARGETS:
    lb.append(lb_row(t, "Numbers only (33 features)", "reference", "NUM", best_of(t, "NUM"), "numbers only"))
    if t == "V":
        lb.append(lb_row(t, "HAR volatility model", "reference", "HAR", "ols", "numbers only"))
    for b in NAMES + FT_NAMES:
        kind = "fine-tuned" if b in FT_NAMES else ("control" if b in T.RANDOM else "frozen")
        lb.append(lb_row(t, T.display_name(b), kind, f"IMG:{b}", "linear", "images only"))
        h_ = best_of(t, f"NUM+IMG:{b}")
        if h_:
            lb.append(lb_row(t, T.display_name(b), kind, f"NUM+IMG:{b}", h_, "numbers + images"))
BOARD = pd.DataFrame(lb)
save_df(BOARD, "stage6L_leaderboard_validation.csv")
MET_COLS = {"V": ["R2", "wmR2", "QLIKE", "MAE", "RMSE", "Spearman"], "D10": ["AUC", "AP", "ACC", "logloss", "BSS", "ECE"],
            "O5": ["AUC", "AP", "ACC", "logloss", "BSS", "ECE"]}
for t in TARGETS:
    sub = BOARD[(BOARD.target == t) & BOARD.condition.isin(["numbers only", "numbers + images"])].copy()
    sub["model"] = sub["model"] + np.where(sub["chosen"], "  <== chosen", "")
    pm = PRIMARY_LB[t][0]
    sub = sub.sort_values("primary_value", ascending=False)
    print(f"\n{TARGET_NAME[t]} - validation {SPLITS['val'][0][:4]}-{int(SPLITS['val'][1][:4]) - 1}, numbers + images "
          f"(best head per model), sorted by {pm}:")
    print(sub[["model", "kind", "head"] + MET_COLS[t] + ["ci_lo", "ci_hi"]].round(4).to_string(index=False))
print("\nMetric guide: R2 = share of variation explained (1 is perfect, 0 = no better than the average); wmR2 = R2 within each month "
      "(stock-vs-stock skill only); QLIKE = volatility forecast loss (lower is better); MAE/RMSE = average error in log volatility; "
      "Spearman = rank correlation; AUC = chance a positive case is ranked above a negative one (0.5 = guessing); AP = average precision; "
      "ACC = accuracy at 50%; logloss = probability loss (lower is better); BSS = Brier skill vs the base rate (0 = no skill); "
      "ECE = calibration error (lower is better).")

# ---- 3. figures: primary metric per model (bars with CIs), all metrics (heatmap), accuracy vs size and speed
COLORS = {"frozen": "#3b6ea5", "fine-tuned": "#e67e22", "control": "#9e9e9e", "reference": "#c0392b"}
fig, axes = plt.subplots(1, 3, figsize=(20, 7.5))
for ax, t in zip(axes, ("V", "D10", "O5")):
    sub = BOARD[(BOARD.target == t) & (BOARD.condition == "numbers + images")].sort_values("primary_value")
    ref = BOARD[(BOARD.target == t) & (BOARD.feature_set == "NUM")].iloc[0]
    yy = np.arange(len(sub))
    ax.barh(yy, sub["primary_value"], color=[COLORS[k] for k in sub["kind"]],
            xerr=np.vstack([sub["primary_value"] - sub["ci_lo"], sub["ci_hi"] - sub["primary_value"]]), capsize=2)
    ax.axvline(ref["primary_value"], color=COLORS["reference"], ls="--", label=f"numbers only ({ref['primary_value']:.3f})")
    ax.set_yticks(yy); ax.set_yticklabels([m + ("  ★" if c else "") for m, c in zip(sub["model"], sub["chosen"])], fontsize=8)
    lo_ = min(sub["ci_lo"].min(), ref["primary_value"]) - 0.01
    ax.set_xlim(lo_, max(sub["ci_hi"].max(), ref["primary_value"]) + 0.01)
    ax.set_title(f"{TARGET_NAME[t]}\nvalidation {PRIMARY_LB[t][0]} (numbers + images, 95% CI; ★ = chosen)", fontsize=10)
    ax.legend(fontsize=8, loc="lower right")
from matplotlib.patches import Patch
fig.legend(handles=[Patch(color=c, label=k) for k, c in COLORS.items() if k != "reference"], loc="upper center", ncol=3, fontsize=9)
plt.tight_layout(rect=(0, 0, 1, 0.95)); fpath = os.path.join(FIG, "stage6L_leaderboard_validation.png"); plt.savefig(fpath, dpi=130); plt.show()
push(fpath, f"figures_{RUN_TAG}/stage6L_leaderboard_validation.png")

heat_cols = [("V", c) for c in ("R2", "wmR2", "QLIKE", "MAE")] + [("D10", c) for c in ("AUC", "AP", "BSS", "ECE")] + [("O5", c) for c in ("AUC", "ACC", "BSS")]
lower_better = {"QLIKE", "MAE", "RMSE", "logloss", "ECE"}
models_h = [m for m in BOARD[(BOARD.condition == "numbers + images") | (BOARD.feature_set == "NUM")].drop_duplicates("model")["model"]]
vals = np.full((len(models_h), len(heat_cols)), np.nan)
for i, mdl in enumerate(models_h):
    for j, (t, c) in enumerate(heat_cols):
        r_ = BOARD[(BOARD.target == t) & (BOARD.model == mdl) & BOARD.condition.isin(["numbers + images", "numbers only"])]
        if len(r_):
            vals[i, j] = r_.iloc[0][c]
score = np.zeros_like(vals)
for j, (t, c) in enumerate(heat_cols):
    col = vals[:, j]; rk = pd.Series(-col if c in lower_better else col).rank(pct=True).values
    score[:, j] = rk
fig, ax = plt.subplots(figsize=(14, 0.42 * len(models_h) + 2))
ax.imshow(score, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)
for i in range(vals.shape[0]):
    for j in range(vals.shape[1]):
        ax.text(j, i, f"{vals[i, j]:.3f}", ha="center", va="center", fontsize=7)
ax.set_xticks(range(len(heat_cols))); ax.set_xticklabels([f"{t}\n{c}" for t, c in heat_cols], fontsize=8)
ax.set_yticks(range(len(models_h))); ax.set_yticklabels(models_h, fontsize=8)
ax.set_title("All metrics, validation, numbers + images (colour = rank among models: green best; V = volatility, D10 = drawdown, O5 = direction)", fontsize=10)
plt.tight_layout(); fpath = os.path.join(FIG, "stage6L_metrics_heatmap.png"); plt.savefig(fpath, dpi=130); plt.show()
push(fpath, f"figures_{RUN_TAG}/stage6L_metrics_heatmap.png")

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
img_v = BOARD[(BOARD.target == "V") & (BOARD.condition == "images only")].set_index("feature_set")
for b in [b for b in NAMES if b in T.PRETRAINED]:
    c0 = CARDS.set_index("key").loc[b]; r0 = img_v.loc[f"IMG:{b}", "R2"]
    for k_, (axi, xcol) in enumerate(((ax[0], "parameters (M)"), (ax[1], "inference images/s"))):
        axi.scatter(c0[xcol], r0, s=70, color=COLORS["frozen"], zorder=3)
        axi.annotate(c0["model"], (c0[xcol], r0), fontsize=8, xytext=(4, 4), textcoords="offset points")
        if T.FT_PREFIX + b in FT_NAMES:
            r1 = img_v.loc[f"IMG:{T.FT_PREFIX + b}", "R2"]
            axi.annotate("", xy=(c0[xcol], r1), xytext=(c0[xcol], r0), arrowprops=dict(arrowstyle="->", color=COLORS["fine-tuned"]))
            axi.scatter(c0[xcol], r1, s=70, marker="^", color=COLORS["fine-tuned"], zorder=3)
for axi, xl in ((ax[0], "parameters (millions, log scale)"), (ax[1], "inference speed (images per second, log scale)")):
    axi.set_xscale("log"); axi.set_xlabel(xl); axi.set_ylabel("validation volatility R² from images alone")
ax[0].set_title("Bigger is not better: accuracy vs model size (circle = frozen, triangle = fine-tuned)", fontsize=10)
ax[1].set_title("Accuracy vs speed", fontsize=10)
plt.tight_layout(); fpath = os.path.join(FIG, "stage6L_efficiency.png"); plt.savefig(fpath, dpi=130); plt.show()
push(fpath, f"figures_{RUN_TAG}/stage6L_efficiency.png")
print(f"Stage 6L done in {time.time() - t0:.0f}s")

## Stage 7 - one honest test (2020-2022)
Refits the frozen choices on train + validation and reads the test years once. Also: the same leaderboard on the test years (descriptive), and the 60 vs 120 comparison on the pilot's own stocks and test rows.

In [ ]:
# ==================================== STAGE 7: one honest test of the frozen choices (2020-2022) ====================================
t0 = time.time()
marker = os.path.join(WORK, "frozen", f"test_opened_{RUN_TAG}.json")
if os.path.exists(marker) and not os.environ.get("TFCV_TEST_REUSE_REASON"):
    raise RuntimeError(f"The {RUN_TAG} test set was already evaluated ({marker}). Re-running it would turn it into a tuning set. "
                       "If you must re-run (e.g. a crash), set os.environ['TFCV_TEST_REUSE_REASON'] = '<why>' and it is logged.")
FZ = json.load(open(FROZEN_PATH))                         # only frozen choices are used below
SELF = FZ["selection"]
cal = lambda t, p: np.interp(p, FZ["calibrators"][t]["x"], FZ["calibrators"][t]["y"])
FIT_TV = TR | VA


def refit(t, fs, head, fit_mask, pred_mask):
    """Refit a frozen configuration on fit_mask rows (same hyper-parameters) and predict pred_mask rows."""
    X = design(fs); mf = fit_mask & OK[t]; f = FITS[(t, fs, head)]
    if head == "linear":
        if TARGETS[t] == "reg":
            w, b, p = T.ridge_path(X[mf], Y[t][mf], X[pred_mask], [f["param"]])[f["param"]]
        else:
            w, b, p = T.logit_path(X[mf], Y[t][mf], X[pred_mask], [f["param"]], device=DEVICE)[f["param"]]
        return dict(head="linear", coef=w, intercept=float(b)), p
    m, _, _ = T.tree_fit(X[mf], Y[t][mf], None, None, TARGETS[t], f["cfg"], gpu=(DEVICE == "cuda"), seed=SEED, n_estimators=f["n_estimators"])
    return dict(head="tree", model=m), T.tree_predict(m, X[pred_mask], TARGETS[t])


MTE = {t: TE & OK[t] for t in TARGETS}
BOOT_TE = {t: T.MonthBoot(DATES[MTE[t]], N_BOOT, seed=SEED) for t in TARGETS}
FINAL, TEST_PRED, TEST_SUMMARY = {}, {}, {}
for t in TARGETS:
    s = SELF[t]
    FINAL[t], p = refit(t, s["feature_set"], s["head"], FIT_TV, MTE[t])
    if TARGETS[t] == "clf":
        p = cal(t, p)
    TEST_PRED[t] = p
    y = Y[t][MTE[t]]; bt = BOOT_TE[t]
    met = eval_target(t, y, p, DATES[MTE[t]], s2=FZ["volatility_residuals"]["s2"] if t == "V" else None,
                      xret=XRET[MTE[t]] if t == "O5" else None, base=BASE.get(t))
    if t == "V":
        met["R2_CI"] = bt.ci(T.r2, y, p)[0]; met["wmR2_CI"] = bt.ci(T.wm_r2, y, p, uses_months=True)[0]
    else:
        met["AUC_CI"] = bt.ci(T.auc, y, p)[0]
        met["BSS_CI"] = bt.ci(lambda yy, pp: 1 - T.brier(yy, pp) / T.brier(yy, np.full(len(yy), BASE[t])), y, p)[0]
    TEST_SUMMARY[t] = met
    print(f"{t} ({s['feature_set']} / {s['head']}): " + ", ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}" for k, v in met.items()))

# ---- H1: numbers + all three chart views vs numbers only, on the test years (pre-registered pair)
H1 = {}
for t, fn, nm in (("V", T.r2, "R2"), ("O5", T.auc, "AUC")):
    pn_, pi_ = SELF[t]["h1_numbers"], SELF[t]["h1_images"]
    if pi_ is None:
        continue
    _, p_num = refit(t, pn_["feature_set"], pn_["head"], FIT_TV, MTE[t])
    _, p_img = refit(t, pi_["feature_set"], pi_["head"], FIT_TV, MTE[t])
    y = Y[t][MTE[t]]
    ci, _ = BOOT_TE[t].diff(fn, y, p_img, p_num)
    H1[t] = dict(images_model=f"{pi_['feature_set']} / {pi_['head']}", numbers_model=f"NUM / {pn_['head']}",
                 metric=nm, numbers=fn(y, p_num), images=fn(y, p_img), delta=fn(y, p_img) - fn(y, p_num), CI=ci)
H1_supported = bool(any(v["CI"][0] > 0 for v in H1.values()))
print(f"\nH1 (three timeframes of chart images add to the numbers, test years): {'SUPPORTED' if H1_supported else 'NOT SUPPORTED'}")
for t, v in H1.items():
    print(f"   {t}: {v['images_model']} vs {v['numbers_model']}: delta {v['metric']} {v['delta']:+.4f}, 95% CI [{v['CI'][0]:+.4f}, {v['CI'][1]:+.4f}]")

# ---- H3: does abstention improve accepted direction calls? (thresholds fixed on validation)
thr10 = FZ["rule"]["thresholds_by_coverage"]["0.1"]
yo, po = Y["O5"][MTE["O5"]].astype(int), TEST_PRED["O5"]
abst_te, _ = T.abstention_table(po, yo, thresholds={float(k): v for k, v in FZ["rule"]["thresholds_by_coverage"].items()})


def _gain(yy, pp):
    keep = np.abs(pp - 0.5) >= thr10
    return (((pp[keep] > 0.5) == yy[keep]).mean() if keep.any() else np.nan) - ((pp > 0.5) == yy).mean()


h3_ci, _ = BOOT_TE["O5"].ci(_gain, yo, po)
H3 = dict(gain_10pct_vs_all=float(_gain(yo, po)), CI=h3_ci, supported=bool(h3_ci[0] > 0), table=abst_te.to_dict("records"))
print(f"H3 (abstention raises accuracy of accepted calls, test years): {'SUPPORTED' if H3['supported'] else 'NOT SUPPORTED'} "
      f"- accuracy gain at 10% coverage {H3['gain_10pct_vs_all']:+.4f}, 95% CI [{h3_ci[0]:+.4f}, {h3_ci[1]:+.4f}]")
print(abst_te.round(4).to_string(index=False))

# ---- NO TRADE rule on the test years: what does the risk flag separate?
common = MTE["V"] & MTE["O5"] & MTE["D10"]
def _on(t):                                            # test predictions restricted to rows valid for all targets
    return TEST_PRED[t][common[MTE[t]]]
vol_hat, pdd = C.ann_vol(_on("V")), _on("D10")
risk = (vol_hat >= FZ["rule"]["vol_ann_q80"]) | (pdd >= FZ["rule"]["p_dd10_q80"])
real_vol, real_dd = C.ann_vol(Y["V"][common]), Y["D10"][common]
NT = dict(share_flagged=float(risk.mean()),
          realised_vol_flagged=float(real_vol[risk].mean()), realised_vol_other=float(real_vol[~risk].mean()),
          dd10_rate_flagged=float(real_dd[risk].mean()), dd10_rate_other=float(real_dd[~risk].mean()))
if FZ["rule"]["direction_usable"]:
    callm = (~risk) & (np.abs(_on("O5") - 0.5) >= FZ["rule"]["conf_threshold"])
    NT["direction_calls_share"] = float(callm.mean())
    NT["direction_calls_accuracy"] = float(((_on("O5")[callm] > 0.5) == Y["O5"][common][callm]).mean()) if callm.any() else None
print(f"\nNO TRADE risk flag on the test years: flags {NT['share_flagged']:.1%} of cases; realised volatility "
      f"{NT['realised_vol_flagged']:.1%} vs {NT['realised_vol_other']:.1%} a year; 10% drawdowns {NT['dd10_rate_flagged']:.1%} vs {NT['dd10_rate_other']:.1%}")

# ---- survivorship sensitivity: refit without the 2001-2009 cache (the cache most exposed to survivorship)
SURV = {}
no_early = FIT_TV & (df["src"].values != "cache_2001_2009")
for t in TARGETS:
    s = SELF[t]
    _, p2 = refit(t, s["feature_set"], s["head"], no_early, MTE[t])
    p2 = cal(t, p2) if TARGETS[t] == "clf" else p2
    y = Y[t][MTE[t]]
    SURV[t] = dict(with_2001_2009=T.r2(y, TEST_PRED[t]) if t == "V" else T.auc(y, TEST_PRED[t]),
                   without_2001_2009=T.r2(y, p2) if t == "V" else T.auc(y, p2), metric="R2" if t == "V" else "AUC")
print("Survivorship sensitivity (test metric with / without the 2001-2009 rows in training):",
      {t: f"{v['metric']} {v['with_2001_2009']:.4f} / {v['without_2001_2009']:.4f}" for t, v in SURV.items()})

# ---- every candidate model on the test years. DESCRIPTIVE ONLY: the frozen choices above are not changed by this table.
t1 = time.time()
TB = []
for t in TARGETS:
    y = Y[t][MTE[t]]; pm, fn = PRIMARY_LB[t]

    def add_tb(label, kind, fs, head, cond):
        _, p = refit(t, fs, head, FIT_TV, MTE[t])                     # same frozen hyper-parameters, refitted on train + validation
        met = T.board_metrics(TARGETS[t], y, p, DATES[MTE[t]], base=BASE.get(t))
        ci, _ = BOOT_TE[t].ci(fn, y, p)
        TB.append(dict(target=t, model=label, kind=kind, condition=cond, feature_set=fs, head=head,
                       chosen=(SELF[t]["feature_set"], SELF[t]["head"]) == (fs, head), primary=pm, primary_value=met[pm],
                       ci_lo=ci[0], ci_hi=ci[1], **met))
    add_tb("Numbers only (33 features)", "reference", "NUM", best_of(t, "NUM"), "numbers only")
    for b in [b for b in NAMES if b in T.PRETRAINED] + FT_NAMES:
        kind = "fine-tuned" if b in FT_NAMES else "frozen"
        add_tb(T.display_name(b), kind, f"NUM+IMG:{b}", best_of(t, f"NUM+IMG:{b}"), "numbers + images")
        add_tb(T.display_name(b), kind, f"IMG:{b}", "linear", "images only")
BOARD_TEST = pd.DataFrame(TB)
save_df(BOARD_TEST, "stage7_leaderboard_test.csv")
for t in TARGETS:
    sub = BOARD_TEST[(BOARD_TEST.target == t) & (BOARD_TEST.condition != "images only")].sort_values("primary_value", ascending=False)
    sub = sub.assign(model=sub["model"] + np.where(sub["chosen"], "  <== chosen", ""))
    print(f"\n{TARGET_NAME[t]} - TEST years, numbers + images (descriptive; probabilities uncalibrated here):")
    print(sub[["model", "kind", "head"] + MET_COLS[t] + ["ci_lo", "ci_hi"]].round(4).to_string(index=False))
fig, axes = plt.subplots(1, 3, figsize=(20, 7))
for ax, t in zip(axes, ("V", "D10", "O5")):
    sub = BOARD_TEST[(BOARD_TEST.target == t) & (BOARD_TEST.condition == "numbers + images")].sort_values("primary_value")
    ref = BOARD_TEST[(BOARD_TEST.target == t) & (BOARD_TEST.feature_set == "NUM")].iloc[0]
    yy = np.arange(len(sub))
    ax.barh(yy, sub["primary_value"], color=[COLORS[k] for k in sub["kind"]],
            xerr=np.vstack([sub["primary_value"] - sub["ci_lo"], sub["ci_hi"] - sub["primary_value"]]), capsize=2)
    ax.axvline(ref["primary_value"], color=COLORS["reference"], ls="--", label=f"numbers only ({ref['primary_value']:.3f})")
    ax.set_yticks(yy); ax.set_yticklabels([m + ("  ★" if c else "") for m, c in zip(sub["model"], sub["chosen"])], fontsize=8)
    ax.set_xlim(min(sub["ci_lo"].min(), ref["primary_value"]) - 0.01, max(sub["ci_hi"].max(), ref["primary_value"]) + 0.01)
    ax.set_title(f"{TARGET_NAME[t]}\nTEST {PRIMARY_LB[t][0]} (numbers + images, 95% CI; ★ = model chosen on validation)", fontsize=10)
    ax.legend(fontsize=8, loc="lower right")
plt.tight_layout(); fpath = os.path.join(FIG, "stage7_leaderboard_test.png"); plt.savefig(fpath, dpi=130); plt.show()
push(fpath, f"figures_{RUN_TAG}/stage7_leaderboard_test.png")
print(f"Test leaderboard: {len(BOARD_TEST)} model rows in {time.time() - t1:.0f}s")

# ---- 120 vs 60 stocks: this run's frozen models vs the pilot's, on the SAME 60 stocks and the SAME test rows
VS_PILOT = None
if PILOT_SET:
    pc = None
    try:
        src = os.environ.get("TFCV_PILOT_BUNDLE")
        pc = json.load(open(os.path.join(src, "config.json") if src else
                            hf_hub_download(PILOT_MODEL_REPO, "config.json", token=HF_TOKEN, local_dir=os.path.join(WORK, "_pilot"))))
    except Exception as e:
        print(f"Pilot bundle {PILOT_MODEL_REPO} not readable ({type(e).__name__}); skipping the 60 vs 120 comparison.")
    if pc:
        mp = df["ticker"].isin(PILOT_SET).values
        rv = []
        for t, mets in (("V", ("R2", "wmR2", "MAE")), ("D10", ("AUC", "BSS", "ECE")), ("O5", ("AUC",))):
            sm = mp[MTE[t]]
            now = eval_target(t, Y[t][MTE[t]][sm], TEST_PRED[t][sm], DATES[MTE[t]][sm],
                              s2=FZ["volatility_residuals"]["s2"] if t == "V" else None, base=BASE.get(t))
            for m_ in mets:
                rv.append(dict(target=t, metric=m_, pilot_model_60=pc["metrics"]["test"][t].get(m_), this_model_on_pilot_60=now.get(m_),
                               this_model_all_stocks=TEST_SUMMARY[t].get(m_), rows_pilot=pc["metrics"]["test"][t].get("n"), rows_now=int(sm.sum()),
                               pilot_choice=pc["targets"][t]["feature_set"] + " / " + pc["targets"][t]["head"],
                               this_choice=SELF[t]["feature_set"] + " / " + SELF[t]["head"]))
        VS_PILOT = pd.DataFrame(rv)
        save_df(VS_PILOT, "stage7_vs_pilot.csv")
        same = (VS_PILOT["rows_pilot"] == VS_PILOT["rows_now"]).all()
        print(f"\nMore data: 60-stock pilot model vs this {df['ticker'].nunique()}-stock model on the pilot's 60 stocks, test years "
              f"({'identical' if same else 'DIFFERENT - check'} test rows: {VS_PILOT['rows_now'].iloc[0]:,}). Lower is better for MAE and ECE.")
        print(VS_PILOT[["target", "metric", "pilot_model_60", "this_model_on_pilot_60", "this_model_all_stocks", "pilot_choice", "this_choice"]].round(4).to_string(index=False))
        fig, ax = plt.subplots(figsize=(10, 4))
        lab = VS_PILOT["target"] + " " + VS_PILOT["metric"]; xx = np.arange(len(VS_PILOT))
        ax.bar(xx - 0.2, VS_PILOT["pilot_model_60"], 0.4, label="pilot model (trained on 60 stocks)", color="#9e9e9e")
        ax.bar(xx + 0.2, VS_PILOT["this_model_on_pilot_60"], 0.4, label=f"this model (trained on {df['ticker'].nunique()} stocks)", color="#3b6ea5")
        ax.set_xticks(xx); ax.set_xticklabels(lab, fontsize=8); ax.legend(fontsize=8)
        ax.set_title("Does more training data help? Same 60 stocks, same test rows (2020-2022)", fontsize=10)
        plt.tight_layout(); fpath = os.path.join(FIG, "stage7_vs_pilot.png"); plt.savefig(fpath, dpi=130); plt.show()
        push(fpath, f"figures_{RUN_TAG}/stage7_vs_pilot.png")
TEST_DD10_RATE = float(np.mean(Y["D10"][MTE["D10"]]))

# ---- deployment rule = frozen rule + the pre-registered test-year check on direction (part b of PREREG['no_trade'])
RULE_DEPLOY = dict(FZ["rule"])
RULE_DEPLOY["direction_usable"] = bool(FZ["rule"]["direction_usable"] and TEST_SUMMARY["O5"]["AUC_CI"][0] > 0.5)
print(f"Direction shown as a call on the dashboard: {RULE_DEPLOY['direction_usable']} "
      f"(validation check {FZ['rule']['direction_usable']}, test AUC CI {TEST_SUMMARY['O5']['AUC_CI'][0]:.3f}-{TEST_SUMMARY['O5']['AUC_CI'][1]:.3f})")
TEST_RESULTS = dict(summary=TEST_SUMMARY, H1=H1, H1_supported=H1_supported, H3=H3, no_trade=NT, survivorship=SURV,
                    rule_deploy=RULE_DEPLOY, n_test_rows={t: int(MTE[t].sum()) for t in TARGETS}, dd10_rate_test=TEST_DD10_RATE,
                    leaderboard=BOARD_TEST.to_dict("records"), vs_pilot=None if VS_PILOT is None else VS_PILOT.to_dict("records"))
tpath = os.path.join(RES, "stage7_test_results.json")
json.dump(TEST_RESULTS, open(tpath, "w"), indent=1, default=float)
push(tpath, f"results_{RUN_TAG}/stage7_test_results.json")
pred_df = df.loc[MTE["V"], ["sample_id", "ticker", "date"]].assign(pred_logrv20=TEST_PRED["V"])
ppath = os.path.join(RES, "stage7_test_predictions.parquet"); pred_df.to_parquet(ppath, index=False)
push(ppath, f"results_{RUN_TAG}/stage7_test_predictions.parquet")
json.dump(dict(utc=dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"), reuse_reason=os.environ.get("TFCV_TEST_REUSE_REASON", "")),
          open(marker, "w"))
push(marker, f"frozen/{os.path.basename(marker)}")
print(f"\nStage 7 done in {time.time() - t0:.0f}s")

## Stage 7M - Monte Carlo price simulation (new in v3)
The dashboard simulates the next 20 trading days from the frozen volatility forecast and its measured error. Here its tail shape and intraday-low range are chosen on validation, and its bands are checked once against the real test-year outcomes.

In [ ]:
# ==================================== STAGE 7M: Monte Carlo price simulation - calibrate on validation, check on the test years ====================================
# The dashboard simulates the next 20 sessions from the frozen volatility forecast. Here the simulation is tuned on validation
# data (tail shape of daily moves by return coverage, then the intraday-low range by drawdown Brier) and checked once on test.
t0 = time.time()
vspec = dict(FZ["volatility_residuals"])
sampler = C.residual_sampler(vspec)
y20 = np.log1p(df["ret20"].to_numpy(float))                 # realised 20-day log return (entry at the next open, as the drawdown label)


def mc_rows(mask_rows, pv, pd_):
    """Rows valid for volatility, drawdown and the 20-day return; returns (forecast, log return, drawdown hit, drawdown prob)."""
    ok = np.isfinite(y20[mask_rows]) & np.isfinite(Y["D10"][mask_rows])
    return pv[ok], y20[mask_rows][ok], Y["D10"][mask_rows][ok], pd_[ok]


# validation arrays: frozen volatility model's validation predictions and the calibrated drawdown probabilities on the same rows
mv = VA & OK["V"] & OK["D10"]
pv_va = pd.Series(VAL_PRED[("V", SELF["V"]["feature_set"], SELF["V"]["head"])], index=np.flatnonzero(VA & OK["V"])).loc[np.flatnonzero(mv)].values
pd_va = pd.Series(cal("D10", VAL_PRED[("D10", SELF["D10"]["feature_set"], SELF["D10"]["head"])]), index=np.flatnonzero(VA & OK["D10"])).loc[np.flatnonzero(mv)].values
xv, lv, hv, qv = mc_rows(mv, pv_va, pd_va)
mt = TE & OK["V"] & OK["D10"]
pv_te = pd.Series(TEST_PRED["V"], index=np.flatnonzero(MTE["V"])).loc[np.flatnonzero(mt)].values
pd_te = pd.Series(TEST_PRED["D10"], index=np.flatnonzero(MTE["D10"])).loc[np.flatnonzero(mt)].values
xt, lt, ht, qt = mc_rows(mt, pv_te, pd_te)
print(f"Rows: validation {len(xv):,}, test {len(xt):,} (each with a volatility forecast, a realised 20-day return and a drawdown label).")

# 1) tail shape: coverage of the realised 20-day returns on validation
grid = []
for df_ in MC_TDF_GRID:
    U, D = T.mc_universal(MC_DRAWS, df_, sampler, seed=SEED, low_k=1.0)
    s_ = T.mc_scores(U, D, xv, lv, hv)
    grid.append(dict(t_df=df_, coverage_error=s_["coverage_error"], **{f"cov_{k}": v for k, v in s_["coverage"].items()}))
GRID_T = pd.DataFrame(grid)
TDF = float(GRID_T.sort_values("coverage_error").iloc[0]["t_df"])
# 2) intraday-low range: Brier of the simulated drawdown chance on validation
grid = []
for k_ in MC_LOWK_GRID:
    U, D = T.mc_universal(MC_DRAWS, TDF, sampler, seed=SEED, low_k=k_)
    s_ = T.mc_scores(U, D, xv, lv, hv)
    grid.append(dict(low_k=k_, dd_brier=s_["dd_brier"], dd_mean_pred=s_["dd_mean_pred"], dd_rate=s_["dd_rate"]))
GRID_K = pd.DataFrame(grid)
LOWK = float(GRID_K.sort_values("dd_brier").iloc[0]["low_k"])
print("Tail shape (validation coverage error, lower is better):"); print(GRID_T.round(4).to_string(index=False))
print("Intraday-low range (validation drawdown Brier, lower is better):"); print(GRID_K.round(4).to_string(index=False))

# 3) the chosen simulation, with and without matching the drawdown model, on validation (choice) and test (check)
U, D = T.mc_universal(MC_DRAWS, TDF, sampler, seed=SEED, low_k=LOWK)
SC = {"val_raw": T.mc_scores(U, D, xv, lv, hv), "val_matched": T.mc_scores(U, D, xv, lv, hv, p_model=qv),
      "test_raw": T.mc_scores(U, D, xt, lt, ht), "test_matched": T.mc_scores(U, D, xt, lt, ht, p_model=qt)}
TILT_DEFAULT = bool(SC["val_matched"]["coverage_error"] <= SC["val_raw"]["coverage_error"] + 0.01)
use = "matched" if TILT_DEFAULT else "raw"
MC = dict(t_df=TDF, low_k=LOWK, res_quantiles=vspec["res_quantiles"], tilt_default=TILT_DEFAULT, horizon=20, drift=0.0,
          coverage={"val": SC[f"val_{use}"]["coverage"], "test": SC[f"test_{use}"]["coverage"]},
          dd_brier={"val": SC[f"val_{use}"]["dd_brier"], "test": SC[f"test_{use}"]["dd_brier"]},
          scores={k: {kk: vv for kk, vv in v.items()} for k, v in SC.items()},
          grid_tdf=GRID_T.to_dict("records"), grid_lowk=GRID_K.to_dict("records"))
tab_mc = pd.DataFrame([{"set": k, **{f"{lvl} band holds": v["coverage"][lvl] for lvl in v["coverage"]},
                        "coverage error": v["coverage_error"], "mean PIT": v["mean_pit"], "drawdown Brier": v["dd_brier"],
                        "drawdown rate": v["dd_rate"], "simulated drawdown chance": v["dd_mean_pred"]} for k, v in SC.items()])
print(f"\nChosen: Student-t with {TDF:g} degrees of freedom, intraday-low range {LOWK:g}; match the drawdown model by default: {TILT_DEFAULT}.")
print("How often the real 20-day outcome fell inside the simulation's central bands (a 90% band should hold about 90%):")
print(tab_mc.round(4).to_string(index=False))
print("(mean PIT above 0.5 = real returns tended to come in above the simulation's middle, i.e. the stocks drifted up; the "
      "simulation deliberately has no drift because the direction model has no edge)")
save_df(tab_mc, "stage7M_monte_carlo_check.csv")
mpath_mc = os.path.join(RES, "stage7M_monte_carlo.json"); json.dump(MC, open(mpath_mc, "w"), indent=1, default=float)
push(mpath_mc, f"results_{RUN_TAG}/stage7M_monte_carlo.json")

# 4) figures: coverage (nominal vs real), PIT histograms, and an example simulation
fig, ax = plt.subplots(1, 3, figsize=(19, 4.6))
lv_ = [0.5, 0.8, 0.9, 0.95]; xx = np.arange(len(lv_))
for i, (k, c) in enumerate((("val_" + use, "#3b6ea5"), ("test_" + use, "#e67e22"))):
    ax[0].bar(xx + (i - 0.5) * 0.36, [SC[k]["coverage"][f"{int(l * 100)}%"] for l in lv_], 0.36, color=c, label=k.replace("_", " "))
ax[0].plot(xx, lv_, "k_", markersize=30, mew=2, label="should be")
ax[0].set_xticks(xx); ax[0].set_xticklabels([f"{int(l * 100)}% band" for l in lv_]); ax[0].set_ylim(0.4, 1.0)
ax[0].set_title("Share of real 20-day outcomes inside the simulated bands"); ax[0].legend(fontsize=8)
for i, (k, c) in enumerate((("val_" + use, "#3b6ea5"), ("test_" + use, "#e67e22"))):
    ax[1].bar(np.arange(10) + (i - 0.5) * 0.4, SC[k]["pit_hist"], 0.4, color=c, label=k.replace("_", " "))
ax[1].axhline(0.1, color="k", ls="--", lw=1); ax[1].set_xticks(range(10)); ax[1].set_xticklabels([f"{i / 10:.1f}" for i in range(10)], fontsize=8)
ax[1].set_title("Where real outcomes fell in the simulated distribution\n(flat at 10% = well calibrated)", fontsize=10); ax[1].legend(fontsize=8)
ex_i = int(np.argsort(xt)[len(xt) // 2])
sim_ex = C.simulate_paths(100.0, xt[ex_i], vspec, mc=MC, n_paths=4000, p_dd_target=qt[ex_i] if TILT_DEFAULT else None, seed=SEED)
for pth in sim_ex["sample_paths"][:40]:
    ax[2].plot(sim_ex["days"], pth, color="#3b6ea5", alpha=0.15, lw=0.8)
ax[2].fill_between(sim_ex["days"], sim_ex["fan"][0], sim_ex["fan"][4], color="#3b6ea5", alpha=0.15, label="90% of paths")
ax[2].fill_between(sim_ex["days"], sim_ex["fan"][1], sim_ex["fan"][3], color="#3b6ea5", alpha=0.3, label="50% of paths")
ax[2].plot(sim_ex["days"], sim_ex["fan"][2], color="#1f4e79", lw=2, label="median")
ax[2].axhline(90, color="#c0392b", ls="--", lw=1, label="10% drawdown level")
ax[2].set_title(f"Example: a typical test-year stock (volatility {C.ann_vol(xt[ex_i]):.0%} a year), start = 100", fontsize=10)
ax[2].set_xlabel("trading days ahead"); ax[2].legend(fontsize=8)
plt.tight_layout(); fpath = os.path.join(FIG, "stage7M_monte_carlo.png"); plt.savefig(fpath, dpi=130); plt.show()
push(fpath, f"figures_{RUN_TAG}/stage7M_monte_carlo.png")
print(f"Stage 7M done in {time.time() - t0:.0f}s")

## Stage 8 - model bundle and parity checks
The bundle must reproduce the notebook's test predictions, the dashboard's input code must reproduce the training inputs exactly, and the dashboard's embedding code must reproduce the notebook's embeddings.

In [ ]:
# ==================================== STAGE 8: the deployable model bundle + two parity checks + model card ====================================
t0 = time.time()
B_DIR = os.path.join(WORK, "bundle")
shutil.rmtree(B_DIR, ignore_errors=True); os.makedirs(B_DIR)
arrays = {"num_fill": num_fill, "num_mean": num_mean, "num_std": num_std}
tcfg, need = {}, {}
for t in TARGETS:
    s, mdl = SELF[t], FINAL[t]
    spec = dict(task=TARGETS[t], feature_set=s["feature_set"], params=s["fit"])
    if mdl["head"] == "linear":
        spec["head"] = "linear"; arrays[f"coef|{t}"] = mdl["coef"]; arrays[f"intercept|{t}"] = np.array(mdl["intercept"])
    else:
        fname = f"{t}.json" if T.tree_backend() == "xgb" else f"{t}.txt"
        spec["head"] = T.tree_save(mdl["model"], os.path.join(B_DIR, fname)); spec["file"] = fname
    if t == "V":
        spec.update(FZ["volatility_residuals"])
    else:
        spec["calibrator"] = FZ["calibrators"][t]
    tcfg[t] = spec
    use_num, b, views = C.Bundle.parse_fs(s["feature_set"])
    if b is not None:
        need.setdefault(b, set()).update(views)
        for v in views:
            mz = np.load(os.path.join(PCA_DIR, f"{b}__{v}_meta.npz"))
            arrays[f"pca_mean|{b}|{v}"], arrays[f"pca_comp|{b}|{v}"] = mz["mean"], mz["comp"]
        _, mu_, sd_ = img_matrix(b, views)
        arrays[f"img_mean|{b}|{''.join(views)}"], arrays[f"img_std|{b}|{''.join(views)}"] = mu_, sd_
recs = json.load(open(os.path.join(RES, "backbones.json")))
for b in need:                                              # a fine-tuned backbone ships its weights inside the (private) bundle
    if b.startswith(T.FT_PREFIX):
        shutil.copy(T.FT_WEIGHTS[b], os.path.join(B_DIR, recs[b]["weights_file"]))
LB_COMPACT = [dict(target=r["target"], model=r["model"], kind=r["kind"], condition=r["condition"], metric=r["primary"],
                   value=r["primary_value"], ci=[r["ci_lo"], r["ci_hi"]], chosen=bool(r["chosen"])) for r in TEST_RESULTS["leaderboard"]]
versions = {"numpy": np.__version__, "pandas": pd.__version__, "torch": torch.__version__}
for mod in ("xgboost", "lightgbm", "timm", "sklearn"):
    try:
        versions[mod] = __import__(mod).__version__
    except Exception:
        pass
BUNDLE_CFG = dict(version="tfcv-1", run_tag=RUN_TAG, created_utc=dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
                  features=FEATS, img_clip=10.0, views=list(T.VIEWS), targets=tcfg,
                  backbones={b: recs[b] for b in need}, rule=RULE_DEPLOY, base_rates=BASE, splits=SPLITS,
                  metrics=dict(val=VAL_SUMMARY, test=TEST_SUMMARY), hypotheses=dict(H1_supported=H1_supported, H2=FZ["H2"]["supported"], H3=H3["supported"]),
                  prereg=FZ["prereg"], frozen_config_sha256=_sha(FROZEN_PATH), versions=versions,
                  data=dict(universe=len(all_tickers), tickers_used=int(df["ticker"].nunique()), rows=len(df)),
                  mc={k: MC[k] for k in ("t_df", "low_k", "res_quantiles", "tilt_default", "horizon", "drift", "coverage", "dd_brier")},
                  risk_quantiles=FZ["risk_quantiles"], reference=dict(FZ["reference"], dd10_rate_test=TEST_DD10_RATE),
                  leaderboard_test=LB_COMPACT, finetune=dict(FZ["finetune"], verdict=FT_VERDICT, backbones=FT_NAMES))
json.dump(BUNDLE_CFG, open(os.path.join(B_DIR, "config.json"), "w"), indent=1, default=float)
np.savez(os.path.join(B_DIR, "arrays.npz"), **{k: np.asarray(v) for k, v in arrays.items()})

# ---- parity 1: the saved bundle reproduces the notebook's own test predictions (same rows, raw inputs)
BD = C.Bundle(B_DIR)
common = MTE["V"] & MTE["O5"] & MTE["D10"]
sel_rows = np.flatnonzero(common)[:3000]
emb = {b: {v: T.assemble(EMB_DIR, f"{b}|{v}", df["sample_id"].values[sel_rows]) for v in T.VIEWS} for b in need}
pb = BD.predict_matrix(Xraw[sel_rows], emb)
pos = {t: np.searchsorted(np.flatnonzero(MTE[t]), sel_rows) for t in TARGETS}
dif = {"V": np.abs(pb["logrv20"] - TEST_PRED["V"][pos["V"]]), "O5": np.abs(pb["p_out5"] - TEST_PRED["O5"][pos["O5"]]),
       "D10": np.abs(pb["p_dd10"] - TEST_PRED["D10"][pos["D10"]])}
print("Bundle vs notebook test predictions:", {k: f"median {np.median(v):.1e}, max {v.max():.1e}, rows > 1e-3: {(v > 1e-3).mean():.2%}" for k, v in dif.items()})
# float32 rounding can flip a tree split on a handful of rows; anything systematic fails here
assert all(np.median(v) < 1e-4 and (v > 1e-3).mean() < 0.005 for v in dif.values()), "the saved bundle does not reproduce the notebook predictions"

# ---- parity 2: dashboard input pipeline (raw bars -> features + charts) equals the training inputs
if os.path.isdir(f"{DST}/final/1day"):
    import pandas_market_calendars as mcal
    sch = mcal.get_calendar("XNYS").schedule("2000-12-01", "2026-12-31")
    early = set(pd.DatetimeIndex(sch.index[sch.market_close.dt.tz_convert("America/New_York").dt.hour < 16]).normalize())
    pick = df[TE].sample(min(40, int(TE.sum())), random_state=SEED)
    fbad = tbad = 0
    for _, r in pick.iterrows():
        d0 = pd.read_parquet(f"{DST}/final/1day/{r.ticker}.parquet")[["date", "open", "high", "low", "close", "volume"]]
        h0 = C.rth_only(pd.read_parquet(f"{DST}/final/1hour/{r.ticker}.parquet"), early)
        k = int(np.searchsorted(d0["date"].values.astype("datetime64[ns]"), np.datetime64(r.date, "ns")))
        f_ = C.features_at(d0, k); ref = r[FEATS].to_numpy(float)
        fbad += int(not np.all(np.isclose(f_, ref, rtol=1e-7, atol=1e-9) | (np.isnan(f_) & np.isnan(ref))))
        tr_ = C.render_at(d0, h0, k); ref_img = st.triplet(r.sample_id)
        tbad += int(not all(np.array_equal(tr_[v], ref_img[v]) for v in T.VIEWS))
    print(f"Dashboard input pipeline vs training inputs on {len(pick)} test rows: feature mismatches {fbad}, chart mismatches {tbad}")
    assert fbad == 0 and tbad == 0, "the dashboard would not see what the model was trained on"

# ---- parity 3: the dashboard's own embedding code (CPU, fp32, weights loaded the way the dashboard loads them) reproduces the
#      notebook's embeddings (GPU, fp16) for the backbones the bundle needs
if need:
    pick3 = df[TE].sample(min(4, int(TE.sum())), random_state=SEED)
    worst = 1.0
    for _, r in pick3.iterrows():
        trip3 = st.triplet(r.sample_id)
        for b in need:
            e_dash = C.embed_views({v: trip3[v] for v in T.VIEWS}, BUNDLE_CFG["backbones"][b], B_DIR) if globals().get("FACTORY") is None else None
            if e_dash is None:
                continue
            for v in T.VIEWS:
                e_nb = T.assemble(emb_dir(b), f"{b}|{v}", np.array([r.sample_id]))[0]
                cos = float(np.dot(e_dash[v], e_nb) / (np.linalg.norm(e_dash[v]) * np.linalg.norm(e_nb) + 1e-12)); worst = min(worst, cos)
    print(f"Dashboard embedding code vs notebook embeddings ({len(pick3)} test rows x {len(need)} backbones x 3 views): lowest cosine similarity {worst:.5f}")
    assert worst > 0.995, "the dashboard would compute different image features from the notebook"

# ---- model card (only computed values)
def _fmt(d):
    return ", ".join(f"{k} {v:.3f}" for k, v in d.items() if isinstance(v, float))
card = [f"# TriFrame CV model bundle ({RUN_TAG} run)", "",
        "Frozen models behind the *Computer Vision with Deep Learning Project Dashboard*. Private: data and weights are not licensed for publication.", "",
        "| Target | Inputs | Head | Test metrics (2020-2022) |", "|---|---|---|---|"]
for t in TARGETS:
    card.append(f"| {TARGET_NAME[t]} | {tcfg[t]['feature_set']} | {tcfg[t]['head']} | {_fmt(TEST_SUMMARY[t])} |")
card += ["", f"H1 (images add to numbers, test): {'supported' if H1_supported else 'not supported'}; "
         f"H2 (backbones differ, validation): {'supported' if FZ['H2']['supported'] else 'not supported'}; "
         f"H3 (abstention helps, test): {'supported' if H3['supported'] else 'not supported'}.", "",
         f"Splits: {SPLITS}. Holdout from {HOLDOUT_START} (see Stage 10 results if run). Configurations tried: {FZ['n_configurations_tried']}.",
         "Limits: 979 survivor US stocks, one data vendor; not investment advice."]
open(os.path.join(B_DIR, "README.md"), "w", encoding="utf-8").write("\n".join(card))
push(B_DIR, "", repo_id=MODEL_REPO, repo_type="model", msg=f"bundle {RUN_TAG}")
print(f"Bundle written to {B_DIR}" + (f" and pushed to {MODEL_REPO} (private)" if UPLOAD else " (upload off)")
      + f". Backbones needed at inference: {sorted(need) or 'none'}")
print(f"Stage 8 done in {time.time() - t0:.0f}s")

## Stage 9 - dashboard files and website page
The dashboard is deployed from the GitHub repo on Streamlit Cloud (docs/DEPLOY.md). This stage writes the HTML block for krishnaharsha.com and names the model repo the dashboard must load.

In [ ]:
# ==================================== STAGE 9: the dashboard files and the krishnaharsha.com page ====================================
# The dashboard (streamlit_app.py in the GitHub repo) loads the model bundle from MODEL_REPO at start-up. Deploy it on Streamlit Cloud (docs/DEPLOY.md);
# an optional Hugging Face Space is built only if DEPLOY_HF_SPACE = True (it needs a paid Hugging Face plan).
SPACE_DIR = os.path.join(WORK, "space")
shutil.rmtree(SPACE_DIR, ignore_errors=True); os.makedirs(SPACE_DIR)
for f in ("tf_render.py", "feat_visible.py"):
    shutil.copy(os.path.join(DST, "scripts", f), SPACE_DIR)
for f in ("tfcv_core.py", "screen_reader.py", "dashboard_logic.py", "app.py"):
    shutil.copy(os.path.join(WORK, f), SPACE_DIR)
req = [f"gradio=={GRADIO_VERSION}", "numpy", "pandas", "pillow", "huggingface_hub", "yfinance", "rapidocr_onnxruntime==1.4.4", "onnxruntime"]
heads = {s["head"] for s in BUNDLE_CFG["targets"].values()}
if "xgb" in heads:
    req.append("xgboost-cpu")                                 # reads model JSON written by the same or older xgboost
if "lgbm" in heads:
    req.append("lightgbm")
if BUNDLE_CFG["backbones"]:                                   # only if a chosen model uses chart images
    req += ["--extra-index-url https://download.pytorch.org/whl/cpu", "torch", f"timm=={BUNDLE_CFG['versions'].get('timm', '')}".rstrip("=")]
open(os.path.join(SPACE_DIR, "requirements.txt"), "w", encoding="utf-8").write("\n".join(req) + "\n")
open(os.path.join(SPACE_DIR, "packages.txt"), "w", encoding="utf-8").write("libgl1\nlibglib2.0-0\n")
open(os.path.join(SPACE_DIR, "README.md"), "w", encoding="utf-8").write(
    "---\ntitle: Computer Vision with Deep Learning Project Dashboard\nemoji: 📈\ncolorFrom: gray\ncolorTo: blue\nsdk: gradio\n"
    f"sdk_version: {GRADIO_VERSION}\npython_version: \"3.11\"\napp_file: app.py\npinned: false\n"
    "short_description: Three chart screenshots to volatility and risk.\n---\n\n"
    "TriFrame dashboard. The model bundle is loaded from a private model repository at start-up.\n")
SPACE_URL = f"https://{SPACE_REPO.replace('/', '-').replace('_', '-').replace('.', '-').lower()}.hf.space"
if UPLOAD and DEPLOY_HF_SPACE:
    try:
        api.create_repo(SPACE_REPO, repo_type="space", space_sdk="gradio", private=SPACE_PRIVATE, exist_ok=True)
        api.upload_folder(folder_path=SPACE_DIR, repo_id=SPACE_REPO, repo_type="space", commit_message=f"dashboard ({RUN_TAG} bundle)")
        api.add_space_variable(SPACE_REPO, "MODEL_REPO", MODEL_REPO)
        if HF_READ_TOKEN:
            api.add_space_secret(SPACE_REPO, "HF_TOKEN", HF_READ_TOKEN)
            print("Space secret HF_TOKEN set from the Colab secret HF_READ_TOKEN.")
        else:
            print("ACTION NEEDED: in the Space settings add a secret HF_TOKEN = a READ-only token (so it can load the private model).")
        print(f"Space pushed: https://huggingface.co/spaces/{SPACE_REPO}  (app URL {SPACE_URL}; first build takes 5-10 min)")
    except Exception as e:
        FAILED_UPLOADS.append((SPACE_DIR, "", SPACE_REPO, "space")); print("Space upload failed:", e)

# ---- the page for krishnaharsha.com (paste into a Custom HTML / Embed block, or upload as its own page)
tm = BUNDLE_CFG["metrics"]["test"]
LIVE_URL = DASHBOARD_URL or (SPACE_URL if (UPLOAD and DEPLOY_HF_SPACE) else "")
IFRAME = (f'<iframe src="{LIVE_URL}{"&" if "?" in LIVE_URL else "?"}embed=true" title="TriFrame dashboard" style="width:100%;height:1700px;border:0;border-radius:8px" loading="lazy"></iframe>'
          if LIVE_URL else "<!-- set DASHBOARD_URL in the config cell once the dashboard is deployed -->")
page = f"""<!-- Computer Vision with Deep Learning Project Dashboard: paste into a Custom HTML / Embed code block -->
<section style="max-width:1100px;margin:0 auto;padding:24px 16px;font-family:system-ui,Segoe UI,Arial,sans-serif;color:#1d2533">
  <h1 style="font-size:32px;margin:0 0 12px">Computer Vision with Deep Learning Project Dashboard</h1>
  <p style="font-size:16px;line-height:1.6">TriFrame tests whether weekly, daily and hourly chart images add information about a US stock's
  next-20-day volatility, drawdown risk and 5-day relative direction beyond plain numbers from the same price history. Seven ImageNet-pretrained
  vision models (frozen and fine-tuned) were compared with a numerical baseline on {df['ticker'].nunique()} of 979 US stocks, with chronological training,
  validation and test periods and a sealed final holdout.</p>
  <p style="font-size:16px;line-height:1.6"><strong>How to use:</strong> choose 1 to 4 companies and upload three TradingView
  screenshots of each (1W, 1D and 1H). The app reads the ticker and the last bar of each picture, checks them against market data,
  rebuilds the exact inputs the models were trained on and returns expected volatility, the chance of a 10% drawdown, the chance of
  beating the median stock over five days, a risk tier, a Monte Carlo simulation of the next 20 trading days and a plain-language
  explanation, then ranks the companies from riskiest to safest. NO TRADE is the default decision.</p>
  <p style="font-size:14px;color:#555">Test-period results (2020-2022): volatility R&sup2; {tm['V'].get('R2', float('nan')):.2f};
  drawdown AUC {tm['D10'].get('AUC', float('nan')):.2f}; direction AUC {tm['O5'].get('AUC', float('nan')):.3f}.
  Research output, not investment advice.</p>
  {IFRAME}
</section>
"""
WEB = os.path.join(WORK, "web", "krishnaharsha_cv_dashboard.html")
open(WEB, "w", encoding="utf-8").write(page)
shutil.copy(WEB, os.path.join(DRIVE_DIR, os.path.basename(WEB)))
push(WEB, "web/krishnaharsha_cv_dashboard.html")
print(f"Website snippet: {WEB} (also on Drive: {DRIVE_DIR}).")

print(f"Model bundle the dashboard must load: {MODEL_REPO}")
print("Next: deploy streamlit_app.py from the GitHub repo on Streamlit Cloud (docs/DEPLOY.md), with secrets HF_TOKEN (read-only) and "
      f"MODEL_REPO = \"{MODEL_REPO}\".")

### Which models the dashboard uses
The exact frozen models behind each dashboard output, why each was chosen, and an example of the dashboard's plain-language explanation on a real test-year row.

In [ ]:
# ==================================== WHICH MODELS THE DASHBOARD USES (v3) ====================================
# The dashboard loads MODEL_REPO and uses exactly the frozen models below (the same code produces this table in the app).
import dashboard_logic as DL
import tfcv_report as RP
importlib.reload(DL); importlib.reload(RP)
MIU = DL.models_in_use(BUNDLE_CFG)
print(f"The dashboard loads {MODEL_REPO} (private) and uses these models:\n")
print(MIU.to_string(index=False))
for t in TARGETS:
    s = BUNDLE_CFG["targets"][t]
    print(f"\n{TARGET_NAME[t]}: {s['feature_set']} / {s['head']}"
          + (f" - chosen over numbers only because the validation gain {SELF[t]['gate']['delta']:+.4f} had a CI that excludes 0 "
             f"[{SELF[t]['gate']['ci'][0]:+.4f}, {SELF[t]['gate']['ci'][1]:+.4f}]" if s["feature_set"] != "NUM" and SELF[t].get("gate") else
             " - numbers only: no image model cleared the pre-registered gate" if s["feature_set"] == "NUM" else ""))
print(f"\nVision networks the dashboard has to run: {BD.needed_backbones() or 'none'}"
      + (" (fine-tuned weights are shipped inside the private bundle)" if any(b.startswith(T.FT_PREFIX) for b in BD.needed_backbones()) else ""))
fpath = RP.fig_pipeline(os.path.join(FIG, "architecture.png"), ft=bool(FT_NAMES))
push(fpath, f"figures_{RUN_TAG}/architecture.png")

# ---- the dashboard's wording on one real test-year example (as of that date; the numbers are the frozen bundle's)
ex_row = df[TE & np.isfinite(Y["V"])].sample(1, random_state=SEED + 5).iloc[0]
ex_emb = {b: {v: T.assemble(emb_dir(b), f"{b}|{v}", np.array([ex_row.sample_id]))[0] for v in T.VIEWS} for b in BD.needed_backbones()}
ex_out = BD.predict_row(ex_row[FEATS].to_numpy(float), ex_emb)
ex_res = dict(ok=True, ticker=ex_row.ticker, asof=pd.Timestamp(ex_row.date).date(), how="test-year example", out=ex_out, p0=100.0)
ex_sim = DL.run_mc(ex_res, BUNDLE_CFG, n_paths=5000)
print("\n" + "=" * 100 + "\nExample of the dashboard's explanation (a test-year row; price shown as an index of 100):\n")
print(DL.report_markdown(ex_res, BUNDLE_CFG, ex_sim))
print(f"\nWhat actually happened next: realised volatility {C.ann_vol(ex_row.logrv20):.0%} a year; "
      f"10% drawdown {'yes' if ex_row.dd20_le_10 == 1 else 'no'}; 20-day return {ex_row.ret20:+.1%}.")

## Stage 10 - holdout (owner only, once)
Leave `RUN_HOLDOUT = False` until you are satisfied with the frozen config. Opening it is logged and single-use.

In [ ]:
# ==================================== STAGE 10 (owner only, once): the sealed holdout 2023-01-03 onward ====================================
# Run this cell only when the frozen config is final. It opens the holdout through holdout_guard (single use, hash-chained log).
# 1) Look at the frozen config:  print(open(FROZEN_PATH).read())
# 2) Compute its SHA-256 yourself:  !sha256sum {FROZEN_PATH}
# 3) Set RUN_HOLDOUT = True below and run; paste the SHA-256 when asked (it is not stored in the notebook).
RUN_HOLDOUT = False
if RUN_HOLDOUT:
    from getpass import getpass
    os.environ["TRIFRAME_HOLDOUT_CONFIG"] = FROZEN_PATH
    os.environ["TRIFRAME_HOLDOUT_UNLOCK"] = getpass("SHA-256 of the frozen config (owner): ").strip()
    t0 = time.time()
    st_h = ImageStore(who=f"tfcv_holdout_{RUN_TAG}", root=DST + "/", allow_holdout=True)      # logged, single use per config
    push(os.path.join(GOV, "holdout_access_log.csv"), "governance/holdout_access_log.csv")
    idx_h = st_h.index()
    hd = st_h.join_features(idx_h).merge(st_h.derived_labels()[["sample_id", "xret5_med", "dd20_le_10"]], on="sample_id", how="left")
    hd = hd[(pd.to_datetime(hd["date"]) >= HOLDOUT_START) & hd["ticker"].isin(df["ticker"].unique())]
    hd = hd.sort_values(["ticker", "date"]).reset_index(drop=True)
    need_b = BD.needed_backbones()
    emb_h = {}
    if need_b:
        HOLD_DIR = os.path.join(WORK, f"emb_holdout_{RUN_TAG}")
        rows_h = st_h._idx.set_index("sample_id").loc[hd["sample_id"].values].reset_index()
        T.extract_embeddings(st_h, rows_h, need_b, HOLD_DIR, DEVICE, chunk_tickers=CHUNK_TICKERS, batch=BATCH, factory=globals().get("FACTORY"))
        emb_h = {b: {v: T.assemble(HOLD_DIR, f"{b}|{v}", hd["sample_id"].values) for v in T.VIEWS} for b in need_b}
    ph = BD.predict_matrix(hd[FEATS].to_numpy(float), emb_h)                 # the deployed bundle, nothing refitted
    HOLD = {}
    for t, col, key in (("V", "logrv20", "logrv20"), ("O5", "y_out5", "p_out5"), ("D10", "dd20_le_10", "p_dd10")):
        y = hd[col].to_numpy(float); ok = np.isfinite(y)
        for part, m in (("all", ok), ("to_2026-02-28", ok & (pd.to_datetime(hd["date"]) <= "2026-02-28").values),
                        ("after_2026-02-28", ok & (pd.to_datetime(hd["date"]) > "2026-02-28").values)):
            if m.sum() < 50:
                continue
            res_ = eval_target(t, y[m], ph[key][m], hd["date"].values[m], s2=FZ["volatility_residuals"]["s2"] if t == "V" else None,
                               xret=hd["xret5_med"].to_numpy(float)[m] if t == "O5" else None, base=BASE.get(t))
            bt = T.MonthBoot(hd["date"].values[m], N_BOOT, seed=SEED)
            res_["CI"] = bt.ci(T.r2 if t == "V" else T.auc, y[m], ph[key][m])[0]
            HOLD[f"{t}|{part}"] = res_
            print(f"{t} {part}: " + ", ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}" for k, v in res_.items()))
    hpath = os.path.join(RES, "stage10_holdout_results.json")
    json.dump(dict(results=HOLD, rows=len(hd), utc=dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")), open(hpath, "w"), indent=1, default=float)
    push(hpath, f"results_{RUN_TAG}/stage10_holdout_results.json")
    BUNDLE_CFG["metrics"]["holdout"] = HOLD
    json.dump(BUNDLE_CFG, open(os.path.join(B_DIR, "config.json"), "w"), indent=1, default=float)
    push(os.path.join(B_DIR, "config.json"), "config.json", repo_id=MODEL_REPO, repo_type="model", msg="add holdout results")
    print(f"Holdout evaluated once in {time.time() - t0:.0f}s and logged in {GOV}.")
else:
    print("Holdout not opened (RUN_HOLDOUT = False). This is the default until the owner decides the frozen config is final.")

## Stage 11 - report, PDF and final upload
REPORT.md and a 5-page plain-language PDF (what was done, step by step; the pretrained models; the architecture; results; fine-tuning; the Monte Carlo check; the dashboard's models), all from the values computed above. The PDF is saved to Google Drive (My Drive > triframe_cv), to Hugging Face, and downloaded to your computer.

In [ ]:
# ==================================== STAGE 11: report (markdown + 5-page PDF, computed values only) and final upload ====================================
def _row(t, d):
    keys = ("R2", "wmR2", "QLIKE") if t == "V" else ("AUC", "BSS", "ECE") + (("rankIC",) if t == "O5" else ())
    return "; ".join(f"{k} {d[k]:.4f}" for k in keys if k in d and isinstance(d[k], float) and np.isfinite(d[k]))


lines = [f"# TriFrame CV - end-to-end run report ({RUN_TAG})", "",
         f"Generated {dt.datetime.now(dt.timezone.utc).isoformat(timespec='seconds')} from the values computed in this notebook.", "",
         "## Data", f"- Universe: {len(all_tickers)} US stocks; this run: {df['ticker'].nunique()} stocks"
         + (f" (including the pilot's {len(PILOT_SET)})" if PILOT_SET else "") + f", {len(df):,} chart sets "
         f"(weekly 104 / daily 126 / hourly 140 bars each) with 33 numeric features.",
         f"- Splits: train {SPLITS['train']}, validation {SPLITS['val']}, test {SPLITS['test']} (20-day purge); holdout from {HOLDOUT_START}.",
         "", "## Models compared",
         f"- {len(NAMES)} frozen backbones ({', '.join(NAMES)})" + (f" and {len(FT_NAMES)} fine-tuned ones ({', '.join(FT_NAMES)})" if FT_NAMES else "")
         + f", PCA {PCA_DIM} per view, linear and boosted-tree heads; {FZ['n_configurations_tried']} configurations tried on validation (all logged in ledger.csv).",
         f"- Fine-tuning: {FT_VERDICT}",
         "", "## Chosen model per target (pre-registered rule) and test results (2020-2022)",
         "| Target | Chosen inputs / head | Validation | Test |", "|---|---|---|---|"]
for t in TARGETS:
    lines.append(f"| {TARGET_NAME[t]} | {SELF[t]['feature_set']} / {SELF[t]['head']} | {_row(t, VAL_SUMMARY[t])} | {_row(t, TEST_SUMMARY[t])} |")
lines += ["", "## Test leaderboard (descriptive; numbers + images, best head per model)"]
for t in TARGETS:
    sub = BOARD_TEST[(BOARD_TEST.target == t) & (BOARD_TEST.condition != "images only")].sort_values("primary_value", ascending=False)
    lines.append(f"- {TARGET_NAME[t]} ({sub['primary'].iloc[0]}): " + "; ".join(
        f"{r.model}{' (chosen)' if r.chosen else ''} {r.primary_value:.3f}" for r in sub.head(6).itertuples()))
lines += ["", "## Hypotheses (rules fixed before the test years were opened)",
          f"- H1, chart images add to the numbers (test): **{'supported' if H1_supported else 'not supported'}**. "
          + "; ".join(f"{t}: delta {v['metric']} {v['delta']:+.4f} [{v['CI'][0]:+.4f}, {v['CI'][1]:+.4f}]" for t, v in H1.items()),
          f"- H2, backbones differ (validation): **{'supported' if FZ['H2']['supported'] else 'not supported'}** "
          f"({len(FZ['H2']['significant_pairs'])} of {FZ['H2']['n_pairs']} pairs after Holm).",
          f"- H3, abstention improves accepted direction calls (test): **{'supported' if H3['supported'] else 'not supported'}** "
          f"(gain at 10% coverage {H3['gain_10pct_vs_all']:+.4f} [{H3['CI'][0]:+.4f}, {H3['CI'][1]:+.4f}]).",
          "", "## Controls (validation)",
          "- " + "; ".join(f"{k}: " + ", ".join(f"{a} {b:.4f}" for a, b in v.items() if isinstance(b, float)) if isinstance(v, dict)
                          else f"{k}: {v:.4f}" for k, v in FZ["controls"].items()),
          "- Random-weight backbones (3 seeds), image-only volatility R2: "
          + ", ".join(f"{b} {FZ['H2']['image_only_R2'][b]['R2']:.4f}" for b in NAMES if b in T.RANDOM),
          "", "## NO TRADE rule on the test years",
          f"- Flags {NT['share_flagged']:.1%} of cases; realised volatility {NT['realised_vol_flagged']:.1%} flagged vs {NT['realised_vol_other']:.1%} other; "
          f"10% drawdowns {NT['dd10_rate_flagged']:.1%} vs {NT['dd10_rate_other']:.1%}. Direction shown as a call on the dashboard: "
          f"{RULE_DEPLOY['direction_usable']} (validation check {FZ['rule']['direction_usable']}, test AUC CI lower bound "
          f"{TEST_SUMMARY['O5']['AUC_CI'][0]:.3f}).",
          "", "## Monte Carlo simulation (Stage 7M)",
          f"- Student-t {MC['t_df']:g} degrees of freedom, intraday-low range {MC['low_k']:g}, drawdown matching by default: {MC['tilt_default']}. "
          f"90% band coverage: validation {MC['coverage']['val']['90%']:.1%}, test {MC['coverage']['test']['90%']:.1%}; "
          f"50% band: validation {MC['coverage']['val']['50%']:.1%}, test {MC['coverage']['test']['50%']:.1%}.",
          "", "## Survivorship sensitivity (test metric with / without 2001-2009 rows in training)"]
lines += [f"- {t}: {v['metric']} {v['with_2001_2009']:.4f} / {v['without_2001_2009']:.4f}" for t, v in SURV.items()]
if VS_PILOT is not None:
    lines += ["", "## More data: this run vs the 60-stock pilot (same 60 stocks, same test rows)"]
    lines += [f"- {r.target} {r.metric}: pilot {r.pilot_model_60:.4f} -> this run {r.this_model_on_pilot_60:.4f}" for r in VS_PILOT.itertuples()]
if "holdout" in BUNDLE_CFG["metrics"]:
    lines += ["", "## Holdout (opened once)"] + [f"- {k}: " + _row(k.split('|')[0], v) for k, v in BUNDLE_CFG["metrics"]["holdout"].items()]
lines += ["", "## Where everything is",
          f"- Results, embeddings, figures, ledger, logs, PDF report: https://huggingface.co/datasets/{RUNS_REPO} (private)",
          f"- Model bundle: https://huggingface.co/{MODEL_REPO} (private)",
          f"- Dashboard: {DASHBOARD_URL or 'streamlit_app.py in the GitHub repo (deploy on Streamlit Cloud, docs/DEPLOY.md)'}",
          "", "Limits: survivor US stocks from one vendor; live inputs from Yahoo Finance; research output, not investment advice."]
REPORT_PATH = os.path.join(RES, f"REPORT_{RUN_TAG}.md")
open(REPORT_PATH, "w", encoding="utf-8").write("\n".join(lines))
shutil.copy(REPORT_PATH, os.path.join(DRIVE_DIR, os.path.basename(REPORT_PATH)))
save_df(pd.DataFrame(LEDGER), "ledger.csv")
push(REPORT_PATH, f"results_{RUN_TAG}/{os.path.basename(REPORT_PATH)}")
print("\n".join(lines))

# ---- the plain-language PDF report (about 5-6 pages), from the values above
if REPORT_PDF:
    yrs = lambda k: (int(SPLITS[k][0][:4]), int(SPLITS[k][1][:4]) - (1 if SPLITS[k][1][5:7] == "01" else 0) + 1)
    figs = dict(pipeline=os.path.join(FIG, "architecture.png"), leaderboard=os.path.join(FIG, "stage7_leaderboard_test.png"),
                finetune=os.path.join(FIG, "stage5F_finetuning.png") if FT_NAMES else None, mc=os.path.join(FIG, "stage7M_monte_carlo.png"),
                timeline=RP.fig_timeline(os.path.join(FIG, "timeline.png"), [("train", *yrs("train"), "#3b6ea5"), ("validation", *yrs("val"), "#e67e22"),
                                                                           ("test", *yrs("test"), "#2e9d5b"), ("holdout (sealed)", yrs("test")[1], 2027, "#7f8c8d")]),
                example=RP.fig_example(os.path.join(FIG, "example_charts.png")))
    hold = BUNDLE_CFG["metrics"].get("holdout", {})
    ts = TEST_SUMMARY
    o5_ci = ts["O5"]["AUC_CI"]
    chosen = []
    for t, lab, test, test_ci, hold_txt, plain in (
            ("V", "Volatility, next 20 days", f"R² {ts['V']['R2']:.2f}", f"R² {ts['V']['R2']:.3f} ({ts['V']['R2_CI'][0]:.2f} to {ts['V']['R2_CI'][1]:.2f})",
             f"R² {hold['V|all']['R2']:.2f}" if "V|all" in hold else "not opened",
             f"explains about {ts['V']['R2']:.0%} of the variation in volatility"),
            ("D10", "10%+ drawdown within 20 days", f"AUC {ts['D10']['AUC']:.2f}", f"AUC {ts['D10']['AUC']:.3f} ({ts['D10']['AUC_CI'][0]:.2f} to {ts['D10']['AUC_CI'][1]:.2f})",
             f"AUC {hold['D10|all']['AUC']:.2f}" if "D10|all" in hold else "not opened",
             f"ranks a stock that drops 10% above one that does not {ts['D10']['AUC']:.0%} of the time"),
            ("O5", "Beats the median stock, 5 days", f"AUC {ts['O5']['AUC']:.3f}", f"AUC {ts['O5']['AUC']:.3f} ({o5_ci[0]:.3f} to {o5_ci[1]:.3f})",
             f"AUC {hold['O5|all']['AUC']:.3f}" if "O5|all" in hold else "not opened",
             "no better than a coin flip" if o5_ci[0] <= 0.5 else "a small but real edge")):
        miu = MIU.iloc[["V", "D10", "O5"].index(t)]
        chosen.append(dict(target=lab, model=f"{miu['Vision network']} + {miu['Head']}" if SELF[t]["feature_set"] != "NUM" else f"numbers only + {miu['Head']}",
                           inputs=miu["Inputs"], network=miu["Vision network"], head=miu["Head"],
                           val=(f"R² {VAL_SUMMARY[t]['R2']:.3f}" if t == "V" else f"AUC {VAL_SUMMARY[t]['AUC']:.3f}"),
                           test=test, test_ci=test_ci, holdout=hold_txt, plain=plain))
    cards_pdf = []
    for r in CARDS.to_dict("records"):
        if r["key"].startswith(T.FT_PREFIX):
            continue
        ftr = records.get(T.FT_PREFIX + r["key"], {}).get("ft") if FT_NAMES else None
        cards_pdf.append(dict(model=r["model"], family=r["family"], pretrained_on=r["pretrained on"], params_m=r["parameters (M)"],
                              finetuning=(f"{ftr['epochs_run']} epochs (best {ftr['best_epoch']}), {ftr['minutes']:.0f} min" if ftr else "-")))
    img_best = max((v["R2"] for b, v in FZ["H2"]["image_only_R2"].items() if b in T.PRETRAINED), default=float("nan"))
    rnd = [v["R2"] for b, v in FZ["H2"]["image_only_R2"].items() if b in T.RANDOM]
    R = dict(
        title_run=f"{df['ticker'].nunique()}-stock run ({RUN_TAG}, notebook v3)", version="v3", n_stocks=int(df["ticker"].nunique()),
        universe=len(all_tickers), n_rows=len(df), pilot_included=bool(PILOT_SET),
        splits={"train": (f"{yrs('train')[0]}-{yrs('train')[1] - 1}", int(TR.sum()), int(df.loc[TR, 'ticker'].nunique()), "fit the models"),
                "validation": (f"{yrs('val')[0]}-{yrs('val')[1] - 1}", int(VA.sum()), int(df.loc[VA, 'ticker'].nunique()), "tune, select, calibrate, set the NO TRADE rule and the simulation"),
                "test": (f"{yrs('test')[0]}-{yrs('test')[1] - 1}", int(TE.sum()), int(df.loc[TE, 'ticker'].nunique()), "one honest check of the frozen choices"),
                "holdout": (f"{HOLDOUT_START[:4]}-2026", 0, 0, "sealed; opened once by the owner (Stage 10)" + (" - opened" if hold else " - not opened in this run"))},
        cards=cards_pdf, n_configs=int(len(LEDGER)), chosen=chosen,
        metrics=dict(test_V_R2=ts["V"]["R2"], direction_usable=bool(RULE_DEPLOY["direction_usable"])),
        hypotheses=dict(
            H1=(bool(H1_supported), "On the test years, adding all three chart views to the numbers changed " + " and ".join(
                f"{'volatility R²' if t == 'V' else 'direction AUC'} by {v['delta']:+.3f} (95% CI {v['CI'][0]:+.3f} to {v['CI'][1]:+.3f})" for t, v in H1.items())
                + (". Both intervals include zero." if not H1_supported else ".")),
            H2=(bool(FZ["H2"]["supported"]), f"{len(FZ['H2']['significant_pairs'])} of {FZ['H2']['n_pairs']} pairs of frozen pretrained networks "
                f"differ in how well their pictures alone predict volatility (validation, Holm-corrected); the best reaches R² {img_best:.3f}."),
            H3=(bool(H3["supported"]), f"Keeping only the 10% most confident direction calls changed their accuracy by {H3['gain_10pct_vs_all']:+.3f} "
                f"(95% CI {H3['CI'][0]:+.3f} to {H3['CI'][1]:+.3f}) on the test years.")),
        controls=(f"With shuffled labels the pipeline finds nothing (volatility R² {FZ['controls']['label_shuffle_V']['R2']:.3f}, direction AUC "
                  f"{FZ['controls']['label_shuffle_O5']['AUC']:.3f}), and randomly initialised ResNet-18s reach volatility R² {min(rnd):.3f} to "
                  f"{max(rnd):.3f} from pictures alone, against {img_best:.3f} for the best pretrained network." if rnd else None),
        no_trade=dict(NT, vol_q80=FZ["rule"]["vol_ann_q80"], pdd_q80=FZ["rule"]["p_dd10_q80"]),
        finetune=(dict(intro=f"<b>In this run all {len(FT_NAMES)} pretrained networks were fine-tuned</b> (up to {FT_EPOCHS} epochs on the training "
                             "years, early stopping on validation) and competed with their frozen versions.", verdict=FT_VERDICT,
                       rows=[[r.model, f"{r.V_num_img_frozen:.3f}", f"{r.V_num_img_finetuned:.3f}", f"{r.V_gain:+.3f} ({r.V_gain_lo:+.3f} to {r.V_gain_hi:+.3f})",
                              f"{r.minutes:.0f} min"] for r in FT_CMP.itertuples()]) if FT_NAMES else None),
        mc=dict(t_df=MC["t_df"], tilt_default=MC["tilt_default"], test_cov90=MC["coverage"]["test"]["90%"], val_cov90=MC["coverage"]["val"]["90%"],
                test_cov50=MC["coverage"]["test"]["50%"]),
        vs_pilot=(dict(text=f"The pilot trained on 60 stocks; this run trained on {df['ticker'].nunique()}, including those 60. Scoring both "
                            "frozen models on the pilot's 60 stocks and the same test rows isolates the effect of more training data.",
                       rows=[[TARGET_NAME[r.target].split(' (')[0], r.metric, f"{r.pilot_model_60:.3f}", f"{r.this_model_on_pilot_60:.3f}",
                              f"{r.this_model_all_stocks:.3f}" if r.this_model_all_stocks == r.this_model_all_stocks else "-"] for r in VS_PILOT.itertuples()])
                  if VS_PILOT is not None else None),
        dashboard=dict(url=DASHBOARD_URL, models=MIU.to_dict("records")),
        limits=[f"{df['ticker'].nunique()} of {len(all_tickers)} stocks; the full-universe run uses the same notebook with QUICK = False.",
                "Survivor-biased universe (only stocks that still exist) from one data vendor, so risk may be understated for stocks that later fail.",
                "The chart renderer scales each picture to fill its panel, so pictures cannot show the volatility level; numbers carry that.",
                "Live dashboard inputs come from Yahoo Finance, which can differ slightly from the training vendor.",
                "Research output, not investment advice. NO TRADE is the default decision."],
        figures=figs, leaderboard_caption="Every candidate on the test years (numbers + images, 95% CIs, ★ = the model chosen on "
                                          "validation). Descriptive only: the choice was frozen before these numbers were computed.")
    PDF_PATH = os.path.join(RES, f"TriFrame_CV_report_{RUN_TAG}.pdf")
    RP.build_pdf(R, PDF_PATH)
    shutil.copy(PDF_PATH, os.path.join(DRIVE_DIR, os.path.basename(PDF_PATH)))
    push(PDF_PATH, f"results_{RUN_TAG}/{os.path.basename(PDF_PATH)}")
    print(f"\nPDF report: {PDF_PATH}\n  also on Google Drive: {DRIVE_DIR}/{os.path.basename(PDF_PATH)} (My Drive > triframe_cv)")
    if IN_COLAB:
        from google.colab import files
        files.download(PDF_PATH)                                    # lands in your computer's Downloads folder

push(FIG, f"figures_{RUN_TAG}")
retry = list(FAILED_UPLOADS); FAILED_UPLOADS.clear()
for local, pir, rid, rtype in retry:
    push(local, pir, repo_id=rid, repo_type=rtype)
print(f"\nUploads still failing: {len(FAILED_UPLOADS)}" + (" - re-run this cell" if FAILED_UPLOADS else " (none)"))